# LSM-ECG replication (Perez-Lopez et al., IEEE Access 2026)
Replicates the paper with the authors' own code, fixes the two bugs found in it, and runs the extra experiments (bug ablations, input-gain sweep, liquid ablations, RR timing, baselines, re-tuning, Brian2 cross-check).

**On Kaggle** (`MACHINE = "kaggle"`): turn Internet ON, pick one GPU, then *Save Version → Save & Run All*. The run stops itself before the 12-hour limit. To continue, attach this version's output as an input and list its folder in `RESUME_DIRS`.

**On your own Linux machine** (`MACHINE = "local"`): set the two paths under `"local"`, then run the notebook headless inside tmux (`jupyter nbconvert --to notebook --execute ...`). Running it again with the same output folder continues where it stopped.

**Modes** (first cell, same on both machines): every session starts with a self-test of the machine; `SELF_TEST = True` stops after it; `SMOKE_TEST = True` is a few-minute end-to-end test (use its own output folder); `BENCHMARK = True` runs representative full-size jobs and projects the total time; all three `False` = the real run.

**Two machines:** set `SHARE = "A"` on one and `SHARE = "B"` on the other (each run belongs to exactly one share). When both are done, run once with `MERGE_DIRS = [<output A>, <output B>]` and a new, empty `WORK_DIR`: it combines them and rebuilds the tables and summary, running nothing.

Everything printed also goes to `logs/session_<time>.log` in the output folder (`logs/latest.log` points to the current one), so `tail -f` shows progress from another terminal. `cache.txt` lists what is DONE, FAILED and PENDING.

## 0. Constants (edit here)

In [1]:
# ╔══════════════════════════════════════════════════════════════════════╗
# ║  CONSTANTS — the only cell you normally need to edit                 ║
# ╚══════════════════════════════════════════════════════════════════════╝

# ---- Machine -----------------------------------------------------------
MACHINE = "local"                  # "kaggle" or "local" (your own Linux machine; set its two paths below)

MACHINES = {
    "kaggle": {
        "INPUT_DIR":   "/kaggle/input/datasets/tienthinh22/mitdb-1-0-0/mit-bih-arrhythmia-database-1.0.0",
        "WORK_DIR":    "/kaggle/working",              # saved as the notebook output (Kaggle cap: 20 GB)
        "SCRATCH_DIR": "/kaggle/working/scratch",      # working files (authors' repo clone); inside the output folder
        # Previous session's output (read-only). Upload the whole /kaggle/working output of the last session
        # here, not only cache.txt. Missing folder or no cache.txt = fresh start.
        "RESUME_DIRS": ["/kaggle/input/datasets/tienthinh22/mitdb-common"],
        "SESSION_BUDGET_HOURS": 11.3,                  # Kaggle stops at 12 h
        "DISK_LIMIT_GB": 19.0,
    },
    "local": {
        # Your own Linux machine. Edit these two paths. The input is only read; the output folder is reused in place:
        # every session reads its cache.txt and skips what is finished (nothing is copied, nothing is fed back in).
        "INPUT_DIR":   "~/lsm_ecg/mitdb-1.0.0/mit-bih-arrhythmia-database-1.0.0",
        "WORK_DIR":    "~/lsm_ecg/output-benchmark",
        "SCRATCH_DIR": None,                           # None = WORK_DIR/scratch (authors' repo clone)
        "RESUME_DIRS": [],
        "SESSION_BUDGET_HOURS": 240.0,                 # no session limit; this is a safety stop
        "DISK_LIMIT_GB": 250.0,
    },
}
_M = MACHINES[MACHINE]
INPUT_DIR            = _M["INPUT_DIR"]
WORK_DIR             = _M["WORK_DIR"]
SCRATCH_DIR          = _M["SCRATCH_DIR"] or WORK_DIR.rstrip("/") + "/scratch"
RESUME_DIRS          = _M["RESUME_DIRS"]   # or "auto" (search AUTO_RESUME_ROOT for cache.txt), or [] to ignore
AUTO_RESUME_ROOT     = "/kaggle/input"
SESSION_BUDGET_HOURS = _M["SESSION_BUDGET_HOURS"]   # no new work is started after this
DISK_LIMIT_GB        = _M["DISK_LIMIT_GB"]          # stop starting new work if WORK_DIR exceeds this

# ---- Session control ---------------------------------------------------
SELF_TEST            = False        # True = stop after the checks (packages, GPU, data checksums, authors' code, a
                                    #        64-beat liquid + readout). The checks run at the start of every session anyway.
SMOKE_TEST           = False       # True = tiny end-to-end test (few beats, 1 fold, 1 seed), a few minutes.
                                    #        Give it its own WORK_DIR: smoke results never go into the real output
BENCHMARK            = True        # True = speed test: representative full-size jobs (kept as real results),
                                    #        then a projection of the total time for everything
ANALYSIS_RESERVE_MIN = 25           # minutes kept free at the end for tables/figures/summary
RETRY_FAILED         = True         # re-attempt runs that failed in an earlier session

# ---- Software ----------------------------------------------------------
PIN_TORCH_240  = True               # use the authors' PyTorch 2.4.0 where possible (Kaggle: only on Python <= 3.12;
                                    # local: pip install torch==2.4.0 where Python <= 3.12). Results are not bit-identical across GPUs anyway.
AUTHORS_REPO   = "https://github.com/Richard97Code/LSM-ECG-Classification.git"
AUTHORS_COMMIT = "02e45c4f3232886aed16a40d77de1f851ef4fcdd"

# ---- Experiment scope --------------------------------------------------
SEEDS = [2026, 69, 420]             # liquid seeds; 2026 is the authors' seed

# Groups run in this order (earlier = more important if the quota runs out).
GROUP_ORDER = [
    "checks",          # data, folds, Gamma bug, measured A_in, liquid structure
    "brian2",          # independent Brian2 implementation of the fixed liquid
    "main_final",      # 24 configs x {published, fixed, fix_gamma, fix_ain} x seeds, DS1 -> DS2
    "ablation_final",  # recurrence off, denser liquids, no liquid, random-feature (ELM) control
    "rr_final",        # RR timing: in the readout, in the liquid, RR-only, RR+shape, LDA
    "baseline_final",  # raw beat, untuned 1D CNN and GRU (with and without RR)
    "gain_final",      # input-gain sweep (fixed liquid)
    "main_cv",         # same as main_final, 3x4-fold grouped CV on DS1 (paper protocol)
    "gain_cv",
    "ablation_cv",
    "rr_cv",
    "baseline_cv",
    "nn_tune",         # tune the CNN and GRU on DS1 CV only (stage 1), then evaluate the winners (stage 2)
    "retune",          # re-tune the fixed liquid on DS1 CV only, then evaluate
    "kstale_final",    # diagnostic: Rate/Latency configs with the Multi-Threshold A_in (0.05892)
    "kstale_cv",
    # extensions (handover section 6); appended last, so no existing run changes group
    "huang_final",     # Huang et al. 2014: random projections + RBF-SVM ensemble for VEB, RR-ratio rule for SVEB
    "huang_cv",
    "meanrr_final",    # RR-only and shape + RR with the extra feature pre-RR / recording mean RR
    "meanrr_cv",
    "elmrr_final",     # ELM fed the 30 RR channels through the same input weights as the RR-in-liquid liquid
    "elmrr_cv",
    "rrliqmean_final", # LSM with RR in the liquid, plus 10 channels for pre-RR / recording mean RR (40 RR channels)
    "rrliqmean_cv",
    "meanrr_nn",       # tuned CNN / GRU + RR with the extra feature (appears once nn_tune is done)
    "hybrid",          # Huang's SVEB rule applied to saved predictions of existing models (seconds per run)
    # a liquid in the regime where LSMs are known to work (one knob, chosen without labels)
    "working_diag",    # one job: perturbation diagnostic over WORKING_GAINS at lambda = 2, picks g_rec
    "working_final",   # then: recurrence on vs off x readout on all windows vs the last window only
    "working_cv",
]
GROUPS_ENABLED = {g: True for g in GROUP_ORDER}

# ---- Two machines: split, then merge ----------------------------------
SHARE = "BOTH"           # "BOTH" = this machine runs everything (one machine). "A" or "B" = this machine runs only
                         # its share; every run belongs to exactly one share, so A + B together = the whole plan, nothing twice.
SHARE_GROUPS = {         # A = the replication, B = the extensions (each about half the work); "checks" runs on both
    "A": ["checks", "brian2", "main_final", "main_cv", "kstale_final", "kstale_cv"],
    "B": ["checks", "ablation_final", "rr_final", "baseline_final", "gain_final", "gain_cv", "ablation_cv", "rr_cv",
          "baseline_cv", "nn_tune", "retune", "huang_final", "huang_cv", "meanrr_final", "meanrr_cv", "elmrr_final",
          "elmrr_cv", "rrliqmean_final", "rrliqmean_cv", "meanrr_nn", "working_diag", "working_final", "working_cv"],
}   # "hybrid" is in neither share: it reads predictions from both, so it runs only with SHARE = "BOTH" (e.g. after a merge)
MERGE_DIRS = []          # merge mode: the two output folders to combine, e.g. ["/data/output_A", "/data/output_B"].
                         # They are only read; the combined result goes to WORK_DIR (use a new, empty folder).
                         # Nothing is run in merge mode: it copies the runs together and rebuilds tables and summary.

GAINS         = [1, 2, 3, 5, 8, 13, 21, 34]   # multiplies the fixed liquid's input weights
DENSE_LAMBDAS = [1.5, 2.5]                    # paper uses 0.8; larger = more recurrent synapses
RETUNE_TRIALS = 30                            # random combinations from RETUNE_GRID (stage 1)
RETUNE_GRID   = {"gain": [1, 2, 3, 5, 8, 13], "lambda_val": [0.8, 1.5, 2.5], "num_levels": [50, 100, 200, 400]}
NN_EPOCHS     = 15                            # untuned CNN / GRU: training epochs (fixed, no early stopping)
BRIAN2_BEATS  = 10                            # beats used in the Brian2 cross-check

# Tuned CNN / GRU baselines: random search on DS1 CV (rep 1, 4 folds) only; DS2 is used once, at the end.
NN_TUNE_TRIALS     = 30                       # per model and input variant (same budget as the liquid re-tuning)
NN_PARAM_CAP       = 35000                    # trainable parameters, about the liquid readout's 33k
NN_TUNE_MAX_EPOCHS = 30
NN_TUNE_CHECKPOINTS = [5, 10, 15, 20, 25, 30] # epoch counts the search may choose from (scored on the held-out fold)

# Huang et al. 2014 baseline (BioMed Eng Online 13:90), on our beats, classes and splits
HUANG_MATRICES  = 15                          # Gaussian random matrices, one RBF SVM each, majority vote
HUANG_DIM       = 50                          # rows per matrix; each beat is 200 raw samples (R-100 .. R+99), z-scored
HUANG_C         = 1.0
HUANG_GAMMAS    = [1.3, 1 / (2 * 1.3 ** 2)]   # the paper's delta = 1.3 read as libsvm gamma, or as Gaussian width sigma
HUANG_THRESHOLD = 0.8                         # SVEB if pre-RR / recording mean RR < this (beats not voted VEB)
HUANG_SVM_CACHE_MB = 500                      # libsvm kernel cache per fit (speed only)
HYBRID_RULES    = ["replace", "add"]          # replace the model's N/SVEB call with the rule, or add the rule's SVEB calls

# Working liquid: Maass et al. (2002) connection range, recurrent gain picked at the edge of chaos
WORKING_LAMBDA     = 2.0                       # connection range (the paper: 0.8)
WORKING_GAINS      = [1, 3, 10, 30, 100, 300, 1000]   # g_rec values the diagnostic tries (multiplies recurrent weights)
WORKING_DIAG_BEATS = 300                       # DS1 beats for the diagnostic (no labels used)
WORKING_T_PERTURB  = 150                       # step at which one input spike is flipped
WORKING_MAX_FIRING = 0.2                       # settings firing more than this per neuron and step are saturated

# Liquid state vectors to save (DS2 test set only, compressed).
SAVE_STATES = [{"config_id": 11, "mode": m, "seed": s} for m in ("published", "fixed") for s in SEEDS]

# ---- Performance -------------------------------------------------------
SIM_BATCH_CV    = 4096   # authors' batch size in the CV script (kept for the exact-replication attempt)
SIM_BATCH_FINAL = 2000   # authors' batch size in the final script
PIPELINE        = True   # True: the GPU works through simulations / NN training one job at a time while finished
                         # jobs' classifier fits run on all CPU cores in parallel (1 fit per core; results identical).
                         # False: the older one-job-at-a-time runner.
FIT_WORKERS     = "auto" # parallel classifier fits in the pipeline: "auto" = every CPU core (the GPU thread
                         # borrows one while it is busy), capped by RAM; or a number (e.g. 3 = keep one core free)
CORE_SEARCH     = True   # MACHINE = "local" with BENCHMARK = True and FIT_WORKERS = "auto" only: first try several numbers
                         # of parallel fits (with the GPU simulating at the same time) and use the fastest; the choice is
                         # saved in checks/core_search.json and used by later sessions. (Kaggle always uses its 4 cores.)
RAM_RESERVE_GB  = 4.0    # RAM never handed out to jobs/fits (Python, CUDA context, the data)
READOUT_JOBS    = 4      # PIPELINE=False only: cores shared by one job's fits


## 1. Environment

In [2]:
NOTEBOOK_BUILD = "3c8e984bf131"   # fingerprint of this build's code cells (do not edit)
# Environment: installs, imports, versions, device. Smoke-test overrides.
import os, sys, subprocess, time, json, platform, shutil, glob, hashlib, traceback, math, re, gc, random
import importlib, importlib.util, datetime, itertools, warnings, contextlib, io, heapq, collections
warnings.filterwarnings("ignore")
SESSION_START = time.time()
ON_KAGGLE = os.path.isdir("/kaggle/working") and "KAGGLE_KERNEL_RUN_TYPE" in os.environ
NOTEBOOK_BUILD = globals().get("NOTEBOOK_BUILD", "dev")      # fingerprint of the code cells, set by the build
if SHARE is None:
    SHARE = "BOTH"
if SHARE != "BOTH" and SHARE not in SHARE_GROUPS:
    raise ValueError(f"SHARE must be 'BOTH', 'A' or 'B', not {SHARE!r}")
MERGE_MODE = bool(MERGE_DIRS)
if MERGE_MODE and SHARE != "BOTH":
    print(f"merge mode: SHARE = {SHARE!r} ignored (the merged result is checked against the whole plan)")
    SHARE = "BOTH"
if MACHINE == "kaggle" and not ON_KAGGLE:
    print("NOTE: MACHINE = 'kaggle' but this does not look like a Kaggle session (paths may not exist).")
INPUT_DIR, WORK_DIR, SCRATCH_DIR = (os.path.abspath(os.path.expanduser(p)) for p in (INPUT_DIR, WORK_DIR, SCRATCH_DIR))
if isinstance(RESUME_DIRS, (list, tuple)):
    RESUME_DIRS = [os.path.abspath(os.path.expanduser(p)) for p in RESUME_DIRS]
os.makedirs(os.path.join(WORK_DIR, "logs"), exist_ok=True)
import threading
SESSION_TAG = datetime.datetime.utcnow().strftime("%Y%m%d_%H%M%S")
LOG_PATH = os.path.join(WORK_DIR, "logs", f"session_{SESSION_TAG}.log")

class _Tee:
    """Everything the notebook prints (all cells, all threads, warnings and tracebacks) also goes to LOG_PATH,
    flushed line by line, so `tail -f WORK_DIR/logs/latest.log` shows it live from another terminal."""
    def __init__(self, stream, path):
        self._s, self._f, self._lock = stream, open(path, "a", buffering=1), threading.Lock()
    def write(self, text):
        with self._lock:
            try:
                self._f.write(text); self._f.flush()
            except Exception:
                pass
        return self._s.write(text)
    def flush(self):
        self._s.flush()
    def __getattr__(self, name):
        return getattr(self._s, name)

if getattr(sys.stdout, "_lsm_tee", False):           # notebook re-run in the same kernel: switch to the new log
    for _t in (sys.stdout, sys.stderr):
        _t._f = open(LOG_PATH, "a", buffering=1)
else:
    sys.stdout, sys.stderr = _Tee(sys.stdout, LOG_PATH), _Tee(sys.stderr, LOG_PATH)
    sys.stdout._lsm_tee = sys.stderr._lsm_tee = True
try:                                                    # a crash's traceback goes into the log file too
    def _exc_to_log(shell, etype, value, tb, tb_offset=None):
        with open(LOG_PATH, "a") as f:
            f.write("\n" + "".join(traceback.format_exception(etype, value, tb)))
        shell.showtraceback((etype, value, tb), tb_offset=tb_offset)
    get_ipython().set_custom_exc((Exception,), _exc_to_log)
except NameError:
    pass
_latest = os.path.join(WORK_DIR, "logs", "latest.log")
try:
    if os.path.islink(_latest) or os.path.exists(_latest):
        os.remove(_latest)
    os.symlink(os.path.basename(LOG_PATH), _latest)
except OSError:
    pass
print(f"session {SESSION_TAG}: everything printed is also written to {LOG_PATH} (and {_latest})")

N_CPU = len(os.sched_getaffinity(0)) if hasattr(os, "sched_getaffinity") else os.cpu_count()
if PIPELINE:
    # one fit per core: the maths libraries must not start their own thread pools inside each fit
    for _v in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS", "NUMEXPR_NUM_THREADS"):
        os.environ[_v] = "1"

import importlib.metadata as _md

def _ver(pkg):
    try:
        return _md.version(pkg)
    except _md.PackageNotFoundError:
        return None

def _pip(*args):
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", *args], capture_output=True, text=True)
    return r.returncode, (r.stdout + r.stderr)[-3000:]

# Kaggle's core packages must not change version mid-session (a NumPy swap after it is loaded breaks imports),
# so every install is constrained to the versions already present; pip then picks compatible package versions.
CORE = ("numpy", "scipy", "pandas", "matplotlib", "torch")
CORE_BEFORE = {p: _ver(p) for p in CORE}
INSTALL_REPORT = []
if not ON_KAGGLE:
    # Your own machine: you install the packages (in a virtual environment); here we only check.
    _need = {"numpy": None, "scipy": None, "pandas": None, "matplotlib": None, "torch": None, "snntorch": "0.9.4",
             "wfdb": None, "tqdm": None, "scikit-learn": "1.7.2", "joblib": None, "threadpoolctl": None}
    _miss = [p for p in _need if _ver(p) is None]
    if _miss:
        raise RuntimeError(f"missing packages {_miss}: pip install them in the virtual environment, activate it, then run the notebook")
    for p, want in _need.items():
        if want and _ver(p) != want:
            print(f"WARNING: {p} {_ver(p)} installed, {want} expected")
    INSTALL_REPORT.append({"package": "(local machine: installed by hand)", "status": "checked", "tail": ""})
    if _ver("brian2") is None:
        print("WARNING: Brian2 is not installed; the brian2 check will be marked FAILED, everything else runs")
if ON_KAGGLE:
    os.makedirs(SCRATCH_DIR, exist_ok=True)
    def _constraints(skip=()):
        path = os.path.join(SCRATCH_DIR, "pip_constraints.txt")
        with open(path, "w") as f:
            f.write("\n".join(f"{p}=={v}" for p, v in CORE_BEFORE.items() if v and p not in skip) + "\n")
        return path
    if PIN_TORCH_240 and sys.version_info[:2] <= (3, 12):
        code, out = _pip("torch==2.4.0", "-c", _constraints(skip=("torch",)))
        INSTALL_REPORT.append({"package": "torch==2.4.0", "status": "ok" if code == 0 else "FAILED", "tail": out})
    elif PIN_TORCH_240:
        INSTALL_REPORT.append({"package": "torch==2.4.0", "status": "skipped",
                               "tail": f"Python {platform.python_version()} has no torch 2.4.0; using Kaggle's torch {CORE_BEFORE['torch']}"})
    for pkg, need in [("snntorch==0.9.4", "required"), ("wfdb", "required"), ("tqdm", "required"),
                      ("scikit-learn==1.7.2", "preferred"), ("brian2", "optional")]:
        code, out = _pip(pkg, "-c", _constraints())
        INSTALL_REPORT.append({"package": pkg, "need": need, "status": "ok" if code == 0 else "FAILED", "tail": out})
    for r in INSTALL_REPORT:
        print(f"install {r['package']:<22} -> {r['status']}" + (f"  ({r['tail']})" if r["status"] == "skipped" else ""))
    bad = [r for r in INSTALL_REPORT if r["status"] == "FAILED" and r.get("need") == "required"]
    if bad:
        raise RuntimeError("Required install failed (is Internet ON in the notebook settings?):\n" + bad[0]["tail"])
    for r in INSTALL_REPORT:
        if r["status"] == "FAILED" and r.get("need") == "preferred":
            print(f"WARNING: {r['package']} could not be installed; using Kaggle's scikit-learn {_ver('scikit-learn')}")
        if r["status"] == "FAILED" and r.get("need") == "optional":
            print("WARNING: Brian2 could not be installed; the brian2 check will be marked FAILED, everything else runs")
    changed = {p: (CORE_BEFORE[p], _ver(p)) for p in CORE if _ver(p) != CORE_BEFORE[p]
               and not (p == "torch" and any(r["package"] == "torch==2.4.0" and r["status"] == "ok" for r in INSTALL_REPORT))}
    if changed:
        raise RuntimeError(f"pip changed core packages {changed}. Restart the session and run again.")

import numpy as np, pandas as pd, torch, snntorch, sklearn, scipy, wfdb, joblib
import torch.nn as nn
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import SGDClassifier
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import confusion_matrix
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.svm import SVC
from sklearn.model_selection import StratifiedGroupKFold
from scipy import stats as sstats
import concurrent.futures
from concurrent.futures import ThreadPoolExecutor

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
TORCH_PINNED = torch.__version__.split("+")[0] == "2.4.0"

def gpu_supported(cap, arch_list):
    """Can this PyTorch build run on a GPU of compute capability cap=(major, minor)? A GPU runs code built for its own
    major version and a lower or equal minor (sm_61 runs sm_60), or PTX (compute_XY) at or below its capability."""
    for a in arch_list:
        kind, _, num = a.partition("_")
        if not num.isdigit():
            continue
        maj, mnr = int(num[:-1]), int(num[-1])
        if kind == "sm" and maj == cap[0] and mnr <= cap[1]:
            return True
        if kind == "compute" and (maj, mnr) <= tuple(cap):
            return True
    return False

if DEVICE.type == "cuda":
    _cap = torch.cuda.get_device_capability(0)
    _gpu = torch.cuda.get_device_name(0)
    _archs = torch.cuda.get_arch_list()
    _hint = (f"Install a PyTorch build that includes it: on Python <= 3.12 'pip install torch==2.4.0' (the authors' version); "
             f"otherwise a CUDA 12.6 build, 'pip install \"torch<2.15\" --index-url https://download.pytorch.org/whl/cu126'. "
             f"Older GPUs (e.g. Pascal sm_6x) are not in the CUDA 12.8+ / 13.x builds.")
    if not gpu_supported(_cap, _archs):
        raise RuntimeError(f"PyTorch {torch.__version__} (CUDA {torch.version.cuda}) has no code for this GPU: {_gpu}, "
                           f"compute capability {_cap[0]}.{_cap[1]}; this build supports {_archs}. {_hint}")
    try:
        torch.ones(8, device=DEVICE).add_(1).sum().item()          # a real kernel launch, not just the device query
    except Exception as ex:
        raise RuntimeError(f"PyTorch cannot run on {_gpu} (capability {_cap[0]}.{_cap[1]}): {ex}. {_hint}")
    print(f"GPU: {_gpu}, compute capability {_cap[0]}.{_cap[1]}: supported by PyTorch {torch.__version__} (CUDA {torch.version.cuda})")
# Plain float32 everywhere: Ampere+ GPUs (e.g. RTX 3090) would otherwise run convolutions in TF32 (a T4 cannot)
torch.backends.cuda.matmul.allow_tf32 = False
torch.backends.cudnn.allow_tf32 = False
if PIPELINE:
    from threadpoolctl import threadpool_limits
    threadpool_limits(1)                 # same as the environment variables above, in case NumPy was already loaded
    torch.set_num_threads(max(1, N_CPU // 4) if DEVICE.type == "cpu" else 1)
import threading
READOUT_EPOCHS = 15            # authors' value (part of the replication, not a tuning knob)

def _ram_gb():
    try:
        with open("/proc/meminfo") as f:
            return round(int(f.readline().split()[1]) / 1e6, 1)
    except Exception:
        return None

ENV = {
    "python": platform.python_version(), "platform": platform.platform(),
    "torch": torch.__version__, "torch_pinned_2_4_0": TORCH_PINNED, "snntorch": snntorch.__version__,
    "sklearn": sklearn.__version__, "numpy": np.__version__, "scipy": scipy.__version__,
    "pandas": pd.__version__, "wfdb": wfdb.__version__, "device": str(DEVICE),
    "cpu_count": N_CPU, "ram_gb": _ram_gb(), "on_kaggle": ON_KAGGLE, "machine": MACHINE, "smoke_test": SMOKE_TEST,
    "pipeline": PIPELINE, "cpu_model": next((l.split(":", 1)[1].strip() for l in open("/proc/cpuinfo") if l.startswith("model name")), "?")
                 if os.path.exists("/proc/cpuinfo") else platform.processor(),
}
if DEVICE.type == "cuda":
    p = torch.cuda.get_device_properties(0)
    ENV.update({"gpu": p.name, "gpu_sm_count": p.multi_processor_count, "gpu_capability": f"{p.major}.{p.minor}",
                "gpu_mem_gb": round(p.total_memory / 1e9, 1), "cuda": torch.version.cuda})
try:
    import brian2
    ENV["brian2"] = brian2.__version__
except Exception as e:
    ENV["brian2"] = f"unavailable ({e.__class__.__name__})"

if SMOKE_TEST:
    SEEDS = SEEDS[:1]
    GAINS = [1, 21]
    DENSE_LAMBDAS = DENSE_LAMBDAS[:1]
    RETUNE_TRIALS = 2
    NN_EPOCHS = 1
    BRIAN2_BEATS = 2
    NN_TUNE_TRIALS = 2
    NN_TUNE_MAX_EPOCHS = 2
    NN_TUNE_CHECKPOINTS = [1, 2]
    HUANG_MATRICES = 2
    WORKING_DIAG_BEATS = 40
    SAVE_STATES = [s for s in SAVE_STATES if s["seed"] in SEEDS]
SMOKE_CONFIGS   = [11, 12, 13, 22]       # used only when SMOKE_TEST
SMOKE_CV_SPLITS = [(1, 1)]               # used only when SMOKE_TEST
SMOKE_PER_CLASS = {0: 10, 1: 5, 2: 5}    # beats kept per record and class when SMOKE_TEST

for k, v in ENV.items():
    print(f"{k:>20}: {v}")
if PIN_TORCH_240 and not TORCH_PINNED:
    print("NOTE: torch 2.4.0 is not active; the exact-replication comparison will still run but cannot match.")


session 20261121_064329: everything printed is also written to /root/lsm_ecg/output-benchmark/logs/session_20261121_064329.log (and /root/lsm_ecg/output-benchmark/logs/latest.log)


GPU: Tesla P40, compute capability 6.1: supported by PyTorch 2.14.1+cu126 (CUDA 12.6)


              python: 3.13.13
            platform: Linux-6.8.0-124-generic-x86_64-with-glibc2.39
               torch: 2.14.1+cu126
  torch_pinned_2_4_0: False
            snntorch: 0.9.4
             sklearn: 1.7.2
               numpy: 2.1.3
               scipy: 1.14.1
              pandas: 2.2.3
                wfdb: 4.3.1
              device: cuda
           cpu_count: 38
              ram_gb: 113.3
           on_kaggle: False
             machine: local
          smoke_test: False
            pipeline: True
           cpu_model: Intel(R) Xeon(R) CPU E5-2696 v4 @ 2.20GHz
                 gpu: Tesla P40
        gpu_sm_count: 30
      gpu_capability: 6.1
          gpu_mem_gb: 24.0
                cuda: 12.6
              brian2: 2.10.1
NOTE: torch 2.4.0 is not active; the exact-replication comparison will still run but cannot match.


## 2. The authors' code (pinned commit)

In [3]:
# The authors' code, cloned at a pinned commit. Their functions are used directly;
# bugs are patched only inside the "fixed" liquid modes (see the liquid cell).
REPO_DIR = os.path.join(SCRATCH_DIR, "authors_repo")
os.makedirs(SCRATCH_DIR, exist_ok=True)
if not os.path.isdir(os.path.join(REPO_DIR, ".git")):
    subprocess.run(["git", "clone", "-q", AUTHORS_REPO, REPO_DIR], check=True)
subprocess.run(["git", "-C", REPO_DIR, "checkout", "-q", AUTHORS_COMMIT], check=True)
ENV["authors_commit"] = subprocess.run(["git", "-C", REPO_DIR, "rev-parse", "HEAD"],
                                       capture_output=True, text=True).stdout.strip()
assert ENV["authors_commit"] == AUTHORS_COMMIT, ENV["authors_commit"]
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

from src.preprocessing import signals as A_sig
from src.preprocessing import stratification as A_strat
from src.encoding import spike_coding as A_enc
from src.utils.reproducibility import seed_everything

def _load_script(name, rel):
    spec = importlib.util.spec_from_file_location(name, os.path.join(REPO_DIR, rel))
    mod = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(mod)          # only imports + definitions; their main blocks are guarded
    return mod

A03 = _load_script("authors_cv_script", "scripts/03_run_24_configurations_cv.py")
LSM3D = A03.LiquidStateMachine3D           # identical class in scripts 02 and 03 (checked)

AUTHORS_CONFIGS = pd.read_csv(os.path.join(REPO_DIR, "configs/all_24_configurations.csv"))
ENC_SHORT = {"multithreshold": "multi", "temporal_rate": "rate", "bounded_delay_latency": "latency"}
CONFIGS = {int(r.config_id): {"config_id": int(r.config_id), "filter": r.filtering, "norm": r.normalization,
                              "encoder": ENC_SHORT[r.encoding], "readout": "svm" if r.readout_loss == "hinge" else "lr",
                              "authors_csv": r.source_result_file}
           for r in AUTHORS_CONFIGS.itertuples()}
print(f"authors' code at {ENV['authors_commit'][:10]}; {len(CONFIGS)} configurations loaded")


authors' code at 02e45c4f32; 24 configurations loaded


## 3. Utilities: cache, resume, guards, metrics, the authors' readout

In [4]:
# Utilities: output folders, logging, cache.txt, resume, time/disk guards, metrics, the authors' readout.
CODE_VERSION = "v1"   # part of every run's hash; bump only if experiment code changes meaning
for sub in ["runs", "reservoirs", "states", "tables", "figures", "logs", "checks"]:
    os.makedirs(os.path.join(WORK_DIR, sub), exist_ok=True)

def elapsed_h():
    return (time.time() - SESSION_START) / 3600

LOCK = threading.RLock()     # cache.txt, the log, speeds and disk counters are shared with the fit workers

def log(*parts):
    line = f"[{elapsed_h():6.2f} h] " + " ".join(str(p) for p in parts)
    with LOCK:
        print(line, flush=True)               # the tee writes it to LOG_PATH

def _json_default(o):
    if isinstance(o, (np.integer,)): return int(o)
    if isinstance(o, (np.floating,)): return float(o)
    if isinstance(o, np.ndarray): return o.tolist()
    if isinstance(o, (np.bool_,)): return bool(o)
    if isinstance(o, torch.Tensor): return o.detach().cpu().tolist()
    return str(o)

def atomic_write_text(path, text):
    tmp = path + ".tmp"
    with open(tmp, "w") as f:
        f.write(text)
    os.replace(tmp, path)

def jdump(obj, path):
    atomic_write_text(path, json.dumps(obj, indent=2, default=_json_default))

def stable_hash(obj, n=8):
    return hashlib.sha1(json.dumps(obj, sort_keys=True, default=_json_default).encode()).hexdigest()[:n]

# ---------------- cache.txt ----------------
CACHE_PATH = os.path.join(WORK_DIR, "cache.txt")
MACHINE_ID = f"{MACHINE}|{ENV.get('gpu', DEVICE.type)}|{N_CPU} cores"      # identifies the machine in merged outputs

def out_tag(share, machine_id):
    """Folder-safe name of an output: its share and machine."""
    return re.sub(r"[^A-Za-z0-9.]+", "_", f"{share if share not in (None, 'None') else 'all'}__{machine_id or 'unknown'}").strip("_")

class RunCache:
    """cache.txt = the record of what has run. Rewritten (atomically) after every run.
    Sections: [DONE] run_id, finished_utc, seconds, macro_f1, accuracy, group
              [FAILED] run_id, when_utc, error
              [PENDING] run_id, group"""
    def __init__(self):
        self.done, self.failed, self.pending = {}, {}, []

    def merge_file(self, path):
        if not os.path.exists(path):
            return
        section = None
        for line in open(path):
            line = line.rstrip("\n")
            if line.startswith("[") and line.endswith("]"):
                section = line[1:-1]
                continue
            if not line or line.startswith("#"):
                continue
            parts = line.split("\t")
            if section == "DONE":
                self.done[parts[0]] = parts[1:]
                self.failed.pop(parts[0], None)
            elif section == "FAILED" and parts[0] not in self.done:
                self.failed[parts[0]] = parts[1:]

    def is_done(self, rid):
        return rid in self.done

    def mark_done(self, rid, seconds, metrics, group):
        with LOCK:
            self._mark_done(rid, seconds, metrics, group)

    def _mark_done(self, rid, seconds, metrics, group):
        mf1 = f"{metrics['macro_f1']:.4f}" if metrics and "macro_f1" in metrics else "-"
        acc = f"{metrics['accuracy']:.4f}" if metrics and "accuracy" in metrics else "-"
        self.done[rid] = [datetime.datetime.utcnow().isoformat(timespec="seconds"), f"{seconds:.1f}", mf1, acc, group]
        self.failed.pop(rid, None)
        self.write()

    def mark_failed(self, rid, err):
        with LOCK:
            self._mark_failed(rid, err)

    def _mark_failed(self, rid, err):
        self.failed[rid] = [datetime.datetime.utcnow().isoformat(timespec="seconds"), err.replace("\t", " ").replace("\n", " | ")[:500]]
        self.write()

    def write(self):
        with LOCK:
            self._write()

    def _write(self):
        lines = ["# LSM-ECG replication: run cache. Rewritten after every run; read by the next session to skip finished work.",
                 f"# updated_utc: {datetime.datetime.utcnow().isoformat(timespec='seconds')}   session_elapsed_h: {elapsed_h():.2f}",
                 f"# totals: DONE {len(self.done)} | FAILED {len(self.failed)} | PENDING {len(self.pending)}",
                 f"# smoke_test: {SMOKE_TEST}",
                 f"# share: {SHARE}",
                 f"# machine_id: {MACHINE_ID}",
                 f"# code_version: {CODE_VERSION}",
                 f"# notebook_build: {NOTEBOOK_BUILD}",
                 "# DONE columns: run_id, finished_utc, seconds, macro_f1, accuracy, group", "[DONE]"]
        lines += ["\t".join([k] + v) for k, v in self.done.items()]
        lines += ["", "# FAILED columns: run_id, when_utc, error", "[FAILED]"]
        lines += ["\t".join([k] + v) for k, v in self.failed.items()]
        lines += ["", "# PENDING columns: run_id, group", "[PENDING]"]
        lines += [f"{rid}\t{g}" for rid, g in self.pending]
        atomic_write_text(CACHE_PATH, "\n".join(lines) + "\n")

CACHE = RunCache()

# ---------------- resume (previous session) and merge mode (two machines' outputs) ----------------
RESUME_SUBDIRS = ["runs", "reservoirs", "states", "logs", "checks"]   # outputs worth carrying; tables/figures are rebuilt

def _cache_is_smoke(p):
    return any(l.strip() == "# smoke_test: True" for l in itertools.islice(open(p), 12))

def cache_header(p):
    h = {}
    for l in itertools.islice(open(p), 12):
        m = re.match(r"^# (smoke_test|share|machine_id|code_version|notebook_build): (.*)$", l.rstrip("\n"))
        if m:
            h[m.group(1)] = m.group(2).strip()
    return h

def copy_forward(src_root):
    n = 0
    for sub in RESUME_SUBDIRS:
        for root, _, files in os.walk(os.path.join(src_root, sub)):
            rel = os.path.relpath(root, src_root)
            for fn in files:
                dst = os.path.normpath(os.path.join(WORK_DIR, rel, fn))
                if not os.path.exists(dst):
                    os.makedirs(os.path.dirname(dst), exist_ok=True)
                    shutil.copy2(os.path.join(root, fn), dst)
                    n += 1
    return n

if MERGE_MODE:
    for md in MERGE_DIRS:
        if os.path.abspath(md) == os.path.abspath(WORK_DIR):
            raise ValueError("merge mode: WORK_DIR must be a new folder, not one of MERGE_DIRS")
        if not os.path.exists(os.path.join(md, "cache.txt")):
            raise FileNotFoundError(f"merge mode: no cache.txt in {md} (give the output folders themselves)")
    _hdrs = [cache_header(os.path.join(md, "cache.txt")) for md in MERGE_DIRS]
    if len({h.get("notebook_build") for h in _hdrs}) > 1:
        print("WARNING: the folders were made by different notebook builds: "
              + ", ".join(f"{md}: {h.get('notebook_build', '?')}" for md, h in zip(MERGE_DIRS, _hdrs)))

if RESUME_DIRS == "auto":
    RESUME_DIRS = sorted({os.path.dirname(p) for p in glob.glob(os.path.join(AUTO_RESUME_ROOT, "**", "cache.txt"), recursive=True)})
for rd in (MERGE_DIRS if MERGE_MODE else RESUME_DIRS):
    cp = os.path.join(rd, "cache.txt")
    if os.path.abspath(rd) == os.path.abspath(WORK_DIR):
        continue
    if not os.path.exists(cp):
        print(f"resume: no cache.txt in {rd} -> fresh start")
        continue
    if _cache_is_smoke(cp) != SMOKE_TEST:
        msg = f"{rd}: its smoke_test flag differs from this session's SMOKE_TEST={SMOKE_TEST}"
        if MERGE_MODE:
            raise ValueError("merge mode: " + msg)
        print(f"resume: skipping {msg}")
        continue
    before = len(CACHE.done)
    _o = RunCache(); _o.merge_file(cp); n_there = len(_o.done)
    CACHE.merge_file(cp)
    n = copy_forward(rd)
    n_dirs = len(os.listdir(os.path.join(rd, "runs"))) if os.path.isdir(os.path.join(rd, "runs")) else 0
    h = cache_header(cp)
    print(f"{'merged' if MERGE_MODE else 'resumed'} from {rd} (share {h.get('share', '?')}, {h.get('machine_id', '?')}): "
          f"{n_there} finished runs there, {len(CACHE.done) - before} of them new here, {n} files copied")
    if n_dirs < len(CACHE.done) - before - 2:
        print(f"WARNING: its cache.txt lists {len(CACHE.done) - before} finished runs but only {n_dirs} run folders are there. "
              "They will be skipped, but their results will be missing from tables and figures. "
              "Bring the whole output folder, not only cache.txt.")
if os.path.exists(CACHE_PATH) and _cache_is_smoke(CACHE_PATH) != SMOKE_TEST:
    raise RuntimeError(f"{WORK_DIR} holds {'smoke-test' if _cache_is_smoke(CACHE_PATH) else 'real'} results but SMOKE_TEST = "
                       f"{SMOKE_TEST}. Use a separate WORK_DIR for the smoke test so the two never mix.")
CACHE.merge_file(CACHE_PATH)       # this machine's earlier sessions (same folder, reused in place)
print(f"cache: {len(CACHE.done)} done, {len(CACHE.failed)} failed")

# ---------------- time and disk guards ----------------
def seconds_left():
    return SESSION_BUDGET_HOURS * 3600 - (time.time() - SESSION_START) - ANALYSIS_RESERVE_MIN * 60

_DISK = {"bytes": None, "since": 0}
def work_dir_gb(force=False):
    if force or _DISK["bytes"] is None or _DISK["since"] >= 50:
        total = 0
        for root, _, files in os.walk(WORK_DIR):
            for fn in files:
                try:
                    total += os.path.getsize(os.path.join(root, fn))
                except OSError:
                    pass
        _DISK.update(bytes=total, since=0)
    return _DISK["bytes"] / 1e9

def disk_add(nbytes):
    with LOCK:
        if _DISK["bytes"] is not None:
            _DISK["bytes"] += nbytes
            _DISK["since"] += 1

# Throughput model, measured as the session runs (and carried over between sessions in checks/speeds.json).
# Units: beats/s for simulation and encoding; seconds per (training beats x features / 1e9) for a classifier fit;
# seconds per epoch per 1,000 training beats for the CNN/GRU.
_GPU = DEVICE.type == "cuda"
# GPU defaults: roughly what a Kaggle T4 measured (used only until this machine has measured its own)
SPEED = {"sim_beats_per_s": {e: (2000.0 if _GPU else 300.0) for e in ("multi", "rate", "latency")},
         "encode_beats_per_s": {e: (20000.0 if _GPU else 1500.0) for e in ("multi", "rate", "latency")},
         "fit_s_per_Gcell": {"1": 400.0, "2": 220.0, "3": 160.0, "pipe": 400.0}, "fit_threads": float(READOUT_JOBS),
         "nn_s_per_epoch_kbeat": {"cnn": 0.04 if _GPU else 3.0, "gru": 0.025 if _GPU else 8.0},
         "nn_unit_s_per_epoch_kbeat": {"cnn": 0.03 if _GPU else 3.0, "gru": 0.04 if _GPU else 8.0},
         "lsm_job_overhead_s": 3.0, "huang_s_per_run": 1200.0}
SPEED_SAMPLES = {}                # recent measurements of the robust (median) entries
# fit_s_per_Gcell: "1"/"2"/"3" = one fit using that many cores (older runner); "pipe" = one fit per core while all
# the pipeline's workers are busy (includes their competition for memory bandwidth).
# nn_unit_*: seconds per epoch per 1,000 beats for a tuned architecture, divided by its relative cost (see nn_cost).
SPEED_MEASURED = set()            # which entries were measured (vs default guesses)
SPEEDS_PATH = os.path.join(WORK_DIR, "checks", "speeds.json")
if os.path.exists(SPEEDS_PATH):
    try:
        _sp = json.load(open(SPEEDS_PATH))
        if _sp.get("device") == DEVICE.type and _sp.get("gpu") == ENV.get("gpu") and _sp.get("cpu_count") == N_CPU:
            for k, v in _sp["speed"].items():
                if isinstance(SPEED.get(k), dict) and isinstance(v, dict):
                    SPEED[k].update(v)
                elif k in SPEED and not isinstance(SPEED[k], dict) and not isinstance(v, dict):
                    SPEED[k] = v                                               # entries whose format changed are skipped
            SPEED_MEASURED.update(m for m in _sp["measured"] if not (m == "fit_s_per_Gcell"))
            SPEED_SAMPLES.update(_sp.get("samples", {}))
            print("speeds carried over from an earlier session:", sorted(SPEED_MEASURED))
    except Exception as ex:
        print("could not read speeds.json:", ex)

def update_speed(key, value, sub=None, alpha=0.3, robust=False):
    with LOCK:
        if robust:            # median of the last 40 measurements: one odd fit does not move the projection
            tag = f"{key}:{sub}" if sub else key
            xs = SPEED_SAMPLES.setdefault(tag, []); xs.append(float(value)); del xs[:-40]
            value, alpha = float(np.median(xs)), 1.0
        _update_speed(key, value, sub, alpha)

def _update_speed(key, value, sub, alpha):
    tag = f"{key}:{sub}" if sub else key
    if sub is None:
        SPEED[key] = value if tag not in SPEED_MEASURED else alpha * value + (1 - alpha) * SPEED[key]
    else:
        SPEED[key][sub] = value if tag not in SPEED_MEASURED else alpha * value + (1 - alpha) * SPEED[key][sub]
    SPEED_MEASURED.add(tag)

def save_speeds():
    with LOCK:
        jdump({"device": DEVICE.type, "gpu": ENV.get("gpu"), "cpu_count": N_CPU, "speed": SPEED, "samples": SPEED_SAMPLES, "measured": sorted(SPEED_MEASURED)}, SPEEDS_PATH)

# ---------------- metrics ----------------
CLASS_NAMES = ["N", "SVEB", "VEB"]

def compute_metrics(y_true, y_pred):
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1, 2])
    tp = np.diag(cm).astype(float)
    support, predicted, total = cm.sum(1).astype(float), cm.sum(0).astype(float), cm.sum()
    rec = np.divide(tp, support, out=np.zeros(3), where=support > 0)
    prec = np.divide(tp, predicted, out=np.zeros(3), where=predicted > 0)
    f1 = np.divide(2 * prec * rec, prec + rec, out=np.zeros(3), where=(prec + rec) > 0)
    fp = predicted - tp
    tn = total - support - fp
    spec = np.divide(tn, tn + fp, out=np.zeros(3), where=(tn + fp) > 0)
    m = {"accuracy": float(tp.sum() / total), "macro_f1": float(f1.mean()), "macro_recall": float(rec.mean()),
         "macro_precision": float(prec.mean()), "n": int(total), "confusion_matrix": cm.tolist()}
    for i, c in enumerate(CLASS_NAMES):
        m.update({f"recall_{c}": float(rec[i]), f"precision_{c}": float(prec[i]), f"f1_{c}": float(f1[i]),
                  f"specificity_{c}": float(spec[i]), f"fpr_{c}": float(1 - spec[i]), f"support_{c}": int(support[i])})
    return m

# ---------------- the authors' readout (exact settings) ----------------
def soft_class_weight(y, p=0.6):
    classes = np.unique(y)
    base = compute_class_weight(class_weight="balanced", classes=classes, y=y).astype(np.float64)
    return dict(zip(classes.astype(int), (base ** p).astype(float)))

CURVE_EVAL_EPOCHS = {1, 3, 5, 10, 15}      # epochs at which the validation curve is measured (CV runs only)

class Cancelled(Exception):
    """Raised inside a fit when the session's time is up; the run stays pending for the next session."""

def prepare_train(Xtr_fn, ntr, prescale):
    """StandardScaler fitted with batched partial_fit (authors' way); optionally the whole scaled training matrix."""
    bs = 4096
    scaler = StandardScaler(with_mean=True, with_std=True)
    for i in range(0, ntr, bs):
        scaler.partial_fit(Xtr_fn(np.arange(i, min(i + bs, ntr))))
    XS = None
    if prescale:
        XS = np.concatenate([scaler.transform(Xtr_fn(np.arange(i, min(i + bs, ntr)))).astype(np.float32) for i in range(0, ntr, bs)])
    return scaler, XS

def fit_readout(Xtr_fn, ntr, ytr, Xev_fn, nev, loss, rng_seed, yev_for_curve=None, curve_sub=4000, prescale=False, n_jobs=None,
                prepared=None, cancel=None, epochs=None):
    """Authors' readout: StandardScaler (batched partial_fit) + SGDClassifier(hinge|log_loss, l2, alpha=1e-4,
    optimal LR, average=True, random_state=2026, soft class weights ^0.6), 15 epochs, batch 4096,
    shuffling with np.random.default_rng(rng_seed). Batches are scaled on the fly as in the authors' code, or, when
    prescale=True and RAM allows, the whole matrix is scaled once (row-wise identical values, faster).
    prepared=(scaler, XS) reuses a scaler / scaled matrix shared by several fits on the same training rows.
    Xtr_fn(positions) / Xev_fn(positions) return float32 feature rows."""
    bs = 4096
    scaler, XS = prepared if prepared is not None else prepare_train(Xtr_fn, ntr, prescale)
    tf = lambda X: scaler.transform(X).astype(np.float32)
    get_tr = (lambda p: XS[p]) if XS is not None else (lambda p: tf(Xtr_fn(p)))
    clf = SGDClassifier(loss=loss, penalty="l2", alpha=1e-4, fit_intercept=True, learning_rate="optimal",
                        max_iter=1, tol=None, class_weight=soft_class_weight(ytr), average=True, random_state=2026,
                        n_jobs=n_jobs)   # n_jobs only parallelises the 3 one-vs-rest problems: results are bit-identical
    classes = np.unique(ytr)
    rng = np.random.default_rng(rng_seed)
    sub = np.sort(np.random.default_rng(0).choice(ntr, min(curve_sub, ntr), replace=False))
    Xsub = get_tr(sub)
    def predict_eval():
        preds, scores = [], []
        for i in range(0, nev, bs):
            Xe = tf(Xev_fn(np.arange(i, min(i + bs, nev))))
            preds.append(clf.predict(Xe)); scores.append(clf.decision_function(Xe).astype(np.float32))
        return np.concatenate(preds), np.concatenate(scores)
    curve = []
    for ep in range(epochs or READOUT_EPOCHS):          # epochs: only the core search uses fewer
        idx = rng.permutation(ntr)
        for i in range(0, ntr, bs):
            if cancel is not None and cancel.is_set():
                raise Cancelled()
            sel = idx[i:i + bs]
            Xb = get_tr(sel)
            if ep == 0 and i == 0:
                clf.partial_fit(Xb, ytr[sel], classes=classes)
            else:
                clf.partial_fit(Xb, ytr[sel])
        row = {"epoch": ep + 1, "train_sub_macro_f1": compute_metrics(ytr[sub], clf.predict(Xsub))["macro_f1"]}
        if loss == "hinge":
            dec = clf.decision_function(Xsub)
            onehot = -np.ones_like(dec); onehot[np.arange(len(sub)), ytr[sub]] = 1
            row["train_sub_hinge_loss"] = float(np.maximum(0, 1 - onehot * dec).mean())
        if yev_for_curve is not None and (ep + 1) in CURVE_EVAL_EPOCHS:
            row["eval_macro_f1"] = compute_metrics(yev_for_curve, predict_eval()[0])["macro_f1"]
        curve.append(row)
    y_pred, scores = predict_eval()
    return scaler, clf, y_pred, scores, curve

# ---------------- saving a run ----------------
def save_run(rid, spec, group, metrics, eval_idx, eval_pid, y_true, y_pred, scores, model_obj,
             curve, liquid_stats, timing, extra=None):
    d = os.path.join(WORK_DIR, "runs", rid)
    os.makedirs(d, exist_ok=True)
    jdump({"run_id": rid, "group": group, "spec": spec, "env": ENV, "code_version": CODE_VERSION,
           "session": SESSION_TAG, **(extra or {})}, os.path.join(d, "config.json"))
    jdump(metrics, os.path.join(d, "metrics.json"))
    np.savez_compressed(os.path.join(d, "predictions.npz"), eval_index=np.asarray(eval_idx, np.int32),
                        patient_id=np.asarray(eval_pid).astype("U4"), y_true=np.asarray(y_true, np.int8),
                        y_pred=np.asarray(y_pred, np.int8),
                        scores=np.asarray(scores, np.float32) if scores is not None else np.zeros((0, 3), np.float32))
    if model_obj is not None:
        if isinstance(model_obj, dict) and model_obj.get("_torch_state"):
            torch.save(model_obj, os.path.join(d, "model.pt"))
        else:
            joblib.dump(model_obj, os.path.join(d, "readout.joblib"), compress=3)
    if curve:
        pd.DataFrame(curve).to_csv(os.path.join(d, "train_curve.csv"), index=False)
    if liquid_stats:
        jdump(liquid_stats, os.path.join(d, "liquid_stats.json"))
    jdump(timing, os.path.join(d, "timing.json"))
    with open(os.path.join(d, "log.txt"), "w") as f:
        f.write(f"{rid}\nmacro_f1={metrics.get('macro_f1')}\naccuracy={metrics.get('accuracy')}\n"
                f"confusion_matrix={metrics.get('confusion_matrix')}\ntiming={timing}\n")
    disk_add(sum(os.path.getsize(os.path.join(d, f)) for f in os.listdir(d)))


cache: 7801 done, 0 failed
speeds carried over from an earlier session: ['encode_beats_per_s:latency', 'encode_beats_per_s:multi', 'encode_beats_per_s:rate', 'fit_s_per_Gcell:pipe', 'huang_s_per_run', 'lsm_job_overhead_s', 'nn_unit_s_per_epoch_kbeat:cnn', 'nn_unit_s_per_epoch_kbeat:gru', 'sim_beats_per_s:latency', 'sim_beats_per_s:multi', 'sim_beats_per_s:rate']


## 4. Data: MIT-BIH → DS1 / DS2, checks against the authors' files

In [5]:
# Data: locate + checksum MIT-BIH, build DS1/DS2 exactly as the authors do (both filters),
# track each beat's position for RR-interval features, verify against the authors' released files.
def locate_input():
    cands = [INPUT_DIR] + sorted({os.path.dirname(p) for p in glob.glob("/kaggle/input/**/100.hea", recursive=True)})
    for c in cands:
        if os.path.exists(os.path.join(c, "100.hea")) and os.path.exists(os.path.join(c, "234.atr")):
            return c
    raise FileNotFoundError(f"MIT-BIH not found at {INPUT_DIR} or anywhere under /kaggle/input")

DATA_DIR = locate_input()
print("MIT-BIH at:", DATA_DIR)

CHECKS = {}
_recs = pd.read_csv(os.path.join(REPO_DIR, "splits/ds1_ds2_records.csv")).set_index("set")["records"]
DS_RECORDS = {"DS1": _recs["DS1"].split(","), "DS2": _recs["DS2"].split(",")}
DS_RECORDS_ALL = DS_RECORDS["DS1"] + DS_RECORDS["DS2"]
sums_path = os.path.join(DATA_DIR, "SHA256SUMS.txt")
if os.path.exists(sums_path):
    bad, missing, n = [], [], 0
    for line in open(sums_path):
        h, fn = line.strip().split(maxsplit=1)
        fp = os.path.join(DATA_DIR, fn)
        if os.path.isfile(fp):
            n += 1
            if hashlib.sha256(open(fp, "rb").read()).hexdigest() != h:
                bad.append(fn)
        else:
            missing.append(fn)
    used = {f"{r}.{x}" for r in DS_RECORDS_ALL for x in ("dat", "hea", "atr")}
    CHECKS["checksums"] = {"files_listed": n + len(missing), "files_checked": n, "mismatches": bad, "missing": missing,
                           "missing_files_used_by_pipeline": sorted(set(missing) & used)}
    print(f"checksums: {n} files checked, {len(bad)} mismatches, {len(missing)} missing"
          + (f" ({', '.join(missing[:12])}{' ...' if len(missing) > 12 else ''})" if missing else ""))
    if set(missing) & used:
        raise FileNotFoundError(f"record files needed by the pipeline are missing: {sorted(set(missing) & used)}")
else:
    CHECKS["checksums"] = {"files_checked": 0, "note": "SHA256SUMS.txt not present"}


NORMAL_S, SVEB_S, VEB_S = ['N', 'L', 'R', 'e', 'j'], ['A', 'a', 'J', 'S'], ['V', 'E']
BEAT_SYMBOLS = set("NLRBAaJSVrFejnE/fQ?")      # all beat annotations (for RR intervals)
RR_NAMES = ["pre", "post", "local10", "rec_median", "pre_over_local", "post_over_local", "pre_over_rec", "pre_over_post"]

def rr_features(all_beats, pos, fs):
    """RR features (seconds / ratios) for beats at sample positions `pos`, from every beat annotation."""
    all_beats = np.sort(np.asarray(all_beats))
    rr = np.diff(all_beats) / fs
    rec_med = float(np.median(rr))
    i = np.searchsorted(all_beats, pos)
    pre = np.where(i > 0, rr[np.clip(i - 1, 0, len(rr) - 1)], rec_med)
    post = np.where(i < len(all_beats) - 1, rr[np.clip(i, 0, len(rr) - 1)], rec_med)
    csum = np.concatenate([[0], np.cumsum(rr)])
    lo = np.clip(i - 10, 0, None)
    cnt = np.maximum(i - lo, 1)
    local = np.where(i > 0, (csum[np.clip(i, 0, len(rr))] - csum[lo]) / cnt, rec_med)
    f = np.stack([pre, post, local, np.full_like(pre, rec_med), pre / local, post / local, pre / rec_med, pre / post], 1)
    return f.astype(np.float32)

def rr_mean_ratio(all_beats, pos, fs):
    """pre-RR / mean RR of the whole recording (Huang et al. 2014); the same pre-RR as the "pre" column."""
    rr = np.diff(np.sort(np.asarray(all_beats))) / fs
    return (rr_features(all_beats, pos, fs)[:, 0] / rr.mean()).astype(np.float32)

def huang_windows(sig, pos, half=100):
    """Huang et al.: 200 raw (unfiltered) samples, R-100 .. R+99 at 360 Hz, minus their mean, divided by their SD."""
    W = np.stack([sig[p - half:p + half] for p in pos]).astype(np.float64)
    return ((W - W.mean(1, keepdims=True)) / W.std(1, keepdims=True)).astype(np.float32)

def build_set(records, filt):
    Xs, syms, pids, poss, rrs = [], [], [], [], []
    rrms, hws = [], []          # extensions: pre-RR / record mean RR, Huang windows
    for rec in records:
        sig, atr_sym, atr_sample, fs = A03.load_ecg(os.path.join(DATA_DIR, rec), target_fs=360)
        if filt == "median":
            sig_f, _ = A_sig.median_filters(sig, fs=fs, kernel_size_ms1=200, kernel_size_ms2=600, mode='reflect')
        else:   # the authors' (commented-out) morphological line
            _, sig_f, _, _ = A_sig.morph_filter_baseline(sig, fs=360, k1_ratio=0.20, k2_ratio=0.35, k3_ratio=0.005, mode='reflect')
        df_all = pd.DataFrame({'atr_sym': atr_sym, 'atr_sample': atr_sample})
        df = df_all[df_all.atr_sym.isin(NORMAL_S + SVEB_S + VEB_S)].reset_index(drop=True)
        Xp, _, sp, pp = A03.build_XY(sig_f, df, 0.8, 0.8, fs, NORMAL_S, SVEB_S + VEB_S, patient_id=rec)
        L = int(round(0.8 * fs))
        keep = ((df.atr_sample - L) >= 0) & ((df.atr_sample + L + 1) <= len(sig_f))
        pos = df.atr_sample[keep].to_numpy()
        assert len(pos) == len(Xp) and list(df.atr_sym[keep]) == list(sp)
        beats_all = [s for s, y in zip(atr_sample, atr_sym) if y in BEAT_SYMBOLS]
        Xs.append(Xp); syms += list(sp); pids += list(pp); poss.append(pos); rrs.append(rr_features(beats_all, pos, fs))
        rrms.append(rr_mean_ratio(beats_all, pos, fs)); hws.append(huang_windows(sig, pos))
    X = np.vstack(Xs); pos = np.concatenate(poss); rr = np.vstack(rrs); pid = np.array(pids)
    num = A03.Numerical_Labels(syms)
    X_sh, y_sh, pid_sh = A03.Shuffle_data(X, num, pid, seed=42)
    np.random.seed(42); perm = np.random.permutation(len(X))           # the same permutation Shuffle_data used
    assert np.array_equal(X[perm], X_sh)
    with contextlib.redirect_stdout(io.StringIO()):     # silence the authors' "taking all of them" prints
        Xd, yd, pidd = A_strat.Make_Data_Stratified(X_sh, y_sh, pid_sh, num_data=50000, Type_Class='SuperC')
        idx_d, _, _ = A_strat.Make_Data_Stratified(np.arange(len(X))[:, None], y_sh, pid_sh, num_data=50000, Type_Class='SuperC')
    src = perm[np.asarray(idx_d)[:, 0].astype(int)]
    assert np.array_equal(X[src], Xd), "beat-order tracking failed"
    return {"X": np.asarray(Xd, np.float64), "y": np.asarray(yd, int), "pid": np.asarray(pidd).astype(str),
            "pos": pos[src], "rr": rr[src], "sym": np.array(syms)[src],
            "rr_mean_ratio": np.concatenate(rrms)[src], "huang_beats": np.vstack(hws)[src]}

t0 = time.time()
DATA = {f: {ds: build_set(DS_RECORDS[ds], f) for ds in ("DS1", "DS2")} for f in ("morph", "median")}
print(f"datasets built in {time.time()-t0:.0f}s")
for ds in ("DS1", "DS2"):
    assert np.array_equal(DATA["morph"][ds]["y"], DATA["median"][ds]["y"])
    assert np.array_equal(DATA["morph"][ds]["pid"], DATA["median"][ds]["pid"])

# verify against the authors' released DS1 order and counts
_auth = pd.read_csv(os.path.join(REPO_DIR, "splits/patient_ids_train.csv"))
CHECKS["ds1_matches_authors_order"] = bool(np.array_equal(_auth.patient_id.astype(str).values, DATA["morph"]["DS1"]["pid"])
                                           and np.array_equal(_auth.label.values, DATA["morph"]["DS1"]["y"]))
CHECKS["class_counts"] = {ds: np.bincount(DATA["morph"][ds]["y"], minlength=3).tolist() for ds in ("DS1", "DS2")}
CHECKS["expected_counts"] = {"DS1": [45827, 943, 3787], "DS2": [44216, 1837, 3219]}
print("DS1 order identical to authors' file:", CHECKS["ds1_matches_authors_order"], "| counts:", CHECKS["class_counts"])

# folds: the authors' exact indices, and a check that they regenerate
_fd = json.load(open(os.path.join(REPO_DIR, "splits/fold_definitions.json")))
FOLDS = [{"rep": f["rep"], "fold": f["fold"], "train_idx": np.array(f["train_idx"]), "val_idx": np.array(f["val_idx"])}
         for f in _fd["folds"]]
_ok, _k = True, 0
_y1, _p1 = DATA["morph"]["DS1"]["y"], DATA["morph"]["DS1"]["pid"]
for rep, sd in enumerate([11, 23, 37], 1):
    for fold, (tr, va) in enumerate(StratifiedGroupKFold(4, shuffle=True, random_state=sd).split(np.zeros(len(_y1)), _y1, _p1), 1):
        f = FOLDS[_k]; _k += 1
        _ok &= (f["rep"], f["fold"]) == (rep, fold) and np.array_equal(np.sort(f["train_idx"]), tr) and np.array_equal(np.sort(f["val_idx"]), va)
CHECKS["folds_regenerate_exactly"] = bool(_ok)
print("authors' fold indices regenerate exactly:", _ok)

# smoke test: keep a few beats per record and class, remap folds
if SMOKE_TEST:
    def _keep(D):
        keep = []
        for p in np.unique(D["pid"]):
            for c, k in SMOKE_PER_CLASS.items():
                keep += list(np.where((D["pid"] == p) & (D["y"] == c))[0][:k])
        return np.sort(np.array(keep))
    _k1, _k2 = _keep(DATA["morph"]["DS1"]), _keep(DATA["morph"]["DS2"])
    for f in DATA:
        for ds, kk in (("DS1", _k1), ("DS2", _k2)):
            DATA[f][ds] = {k: v[kk] for k, v in DATA[f][ds].items()}
    _remap_arr = -np.ones(int(_k1.max()) + 1, int); _remap_arr[_k1] = np.arange(len(_k1))
    def _remap(idx):
        r = _remap_arr[idx[idx <= _k1.max()]]
        return r[r >= 0]
    FOLDS = [{"rep": f["rep"], "fold": f["fold"], "train_idx": _remap(f["train_idx"]), "val_idx": _remap(f["val_idx"])}
             for f in FOLDS]
    print("SMOKE subset:", {ds: np.bincount(DATA['morph'][ds]['y'], minlength=3).tolist() for ds in ("DS1", "DS2")})

CV_SPLITS = [(f["rep"], f["fold"]) for f in FOLDS]
if SMOKE_TEST:
    CV_SPLITS = SMOKE_CV_SPLITS
FOLD_BY = {(f["rep"], f["fold"]): f for f in FOLDS}

# ---------------- normalization (two paths, as the authors have them) ----------------
def normalize_cv(X, tr_idx, norm):
    """CV path (script 03 get_fold_data + float32 cast before encoding)."""
    if norm == "standard":
        return np.asarray(A_strat.minmax_per_beat(X), np.float32), {}
    mn, mx = A_strat.robust_minmax_fit(X[tr_idx], low=0.1, high=99.9)
    return np.asarray(A_strat.robust_minmax_apply(X, mn, mx), np.float32), {"p_low": mn, "p_high": mx}

def normalize_final(X1, X2, norm):
    """Final DS1 -> DS2 path (script 01): float64, global DS1 percentiles for robust."""
    if norm == "standard":
        return A_strat.Rowwise_Normalization(X1), A_strat.Rowwise_Normalization(X2), {}
    lo, hi = np.percentile(X1, [0.1, 99.9])
    return A_strat.minmax_apply(X1, lo, hi), A_strat.minmax_apply(X2, lo, hi), {"p_low": float(lo), "p_high": float(hi)}


MIT-BIH at: /root/lsm_ecg/mitdb-1.0.0/mit-bih-arrhythmia-database-1.0.0


checksums: 704 files checked, 0 mismatches, 0 missing


datasets built in 102s
DS1 order identical to authors' file: True | counts: {'DS1': [45827, 943, 3787], 'DS2': [44216, 1837, 3219]}


authors' fold indices regenerate exactly: True


## 5. Spike encoders and measured A_in

In [6]:
# Encoders (the authors' functions, device passed explicitly), RR spike channels, A_in measurement.
PAPER_AIN = {"multi": 0.05892, "rate": 20.8245, "latency": 0.9473}   # values hard-coded by the authors

def multithreshold_t(Xb, num_levels=200):
    """GPU version of the authors' MultiThreshold (detect_edges='both', include_endpoints=True,
    separate_polarity=False). Same thresholds (np.linspace in the data's dtype) and the same comparisons:
    rise = x_t >= thr & x_{t-1} < thr, fall = x_t < thr & x_{t-1} >= thr, spike = rise | fall."""
    X = np.ascontiguousarray(np.asarray(Xb))
    thr = torch.from_numpy(np.linspace(0.0, 1.0, num_levels, dtype=X.dtype)).to(DEVICE)
    x = torch.from_numpy(X).to(DEVICE)
    out = torch.zeros((X.shape[0], X.shape[1], num_levels), dtype=torch.uint8, device=DEVICE)
    above = x[:, :, None] >= thr                     # (B, T, K)
    out[:, 1:] = (above[:, 1:] != above[:, :-1]).to(torch.uint8)
    return out

def roundrobin_t(counts, bin_pop=10):
    """GPU version of the authors' counts_to_binary_population_roundrobin. Their loop keeps a pointer p per
    (beat, channel), sets sub-channels (p + j) % bin_pop for j < min(k, bin_pop), then p = (p + k) % bin_pop.
    So p_t = (counts before t) % bin_pop, and sub-channel b fires iff (b - p_t) % bin_pop < min(k_t, bin_pop)."""
    c = torch.as_tensor(np.asarray(counts).astype(np.int64), device=DEVICE)          # (B, L, C)
    p = torch.remainder(torch.cumsum(c, dim=1) - c, bin_pop)
    keff = torch.clamp(c, max=bin_pop)
    b = torch.arange(bin_pop, device=DEVICE)
    rel = torch.remainder(b[None, None, None, :] - p[..., None], bin_pop)
    out = (rel < keff[..., None]).to(torch.uint8)
    return out.reshape(c.shape[0], c.shape[1], c.shape[2] * bin_pop)

def encode_t(Xb, enc, num_levels=200):
    """(B, 577) normalized beats -> (B, T_in, C) uint8 spikes as a tensor on DEVICE."""
    if enc == "multi":
        return multithreshold_t(Xb, num_levels)
    if enc == "rate":
        s = A_enc.TemporalRateCoding(Xb, None, gain=1.0, offset=0.0, pop_size=64, batch_size=1024, device=DEVICE,
                                     out_dir=None, spike_filename=None, labels_filename=None, save_to_disk=False)
        return torch.from_numpy(np.asarray(s, dtype=np.uint8)).to(DEVICE)
    if enc == "latency":
        cnt = A_enc.BoundedDelayLatencyCoding(Xb, Dmax=64, pop_size=1, batch_size=1024, device=DEVICE)
        return roundrobin_t(cnt, bin_pop=10)
    raise ValueError(enc)

def encode(Xb, enc, num_levels=200):
    """Same as encode_t, returned as a NumPy array."""
    return encode_t(Xb, enc, num_levels).cpu().numpy()

N_IN = {"multi": 200, "rate": 64, "latency": 10}

# RR timing as extra input channels: thermometer code of 3 ratio features, 10 levels each.
RR_LIQ_FEATS = [RR_NAMES.index(n) for n in ("pre_over_local", "post_over_local", "pre_over_rec")]
RR_LEVELS, RR_PERIOD = 10, 5            # an active channel spikes every 5th step

def rr_liq_feats(rr_rows):
    """Columns turned into RR channels: the 3 ratios; with the 9-column array of rr_inputs(..., True) also the
    9th column, pre-RR / recording mean RR (40 channels instead of 30)."""
    return RR_LIQ_FEATS + ([len(RR_NAMES)] if rr_rows.shape[1] > len(RR_NAMES) else [])

def rr_thresholds(rr_train):
    q = np.linspace(0.05, 0.95, RR_LEVELS)
    return np.stack([np.quantile(rr_train[:, j], q) for j in rr_liq_feats(rr_train)])     # (3 or 4, 10)

def rr_channels(rr_rows, thr, T):
    active = (rr_rows[:, rr_liq_feats(rr_rows)][:, :, None] >= thr[None]).reshape(len(rr_rows), -1)   # (B, 30 or 40)
    pulse = (np.arange(T) % RR_PERIOD == 0).astype(np.uint8)
    return (pulse[None, :, None] * active[:, None, :]).astype(np.uint8)                   # (B, T, 30)

def rr_window_counts(rr_rows, thr, T):
    """rr_channels(rr_rows, thr, T) summed over the 32 windows of 20 steps (steps >= T are silent), (B, 32, 30) uint8."""
    active = (rr_rows[:, rr_liq_feats(rr_rows)][:, :, None] >= thr[None]).reshape(len(rr_rows), -1)
    steps = np.arange(NWIN * WIN)
    per_window = ((steps % RR_PERIOD == 0) & (steps < T)).reshape(NWIN, WIN).sum(1)
    return (per_window[None, :, None] * active[:, None, :]).astype(np.uint8)

def measure_ain(Xn, rows, enc, num_levels=200, rr=None, rr_thr=None, n=5000):
    """Average number of active input spikes per time step (summed over channels), over a random
    subset of training beats. Same quantity as the authors' estimate_K_spikes_per_timestep."""
    rows = np.asarray(rows)
    torch.manual_seed(12345)           # the Rate encoder is stochastic: make A_in independent of job order
    pick = np.sort(np.random.default_rng(0).choice(rows, min(n, len(rows)), replace=False))
    tot, cnt = 0.0, 0
    for i in range(0, len(pick), 1000):
        r = pick[i:i + 1000]
        S = encode(Xn[r], enc, num_levels)
        if rr is not None:
            S = np.concatenate([S, rr_channels(rr[r], rr_thr, S.shape[1])], axis=2)
        tot += float(S.sum(dtype=np.int64)); cnt += S.shape[0] * S.shape[1]
    return tot / cnt


## 6. The liquid (published / fixed modes) and the simulator

In [7]:
# The liquid: built with the authors' class; modes patch the two bugs. GPU simulator with diagnostics.
LIQ = dict(P_in=0.1, betaE=0.95, betaI=0.8, Uthr=1.0, rho=0.9, reservoir_dim=(7, 7, 7), excit_ratio=0.8,
           dt=1.0, num_sim_steps=650, C_vals={"exc_exc": 0.3, "exc_inh": 0.2, "inh_exc": 0.4, "inh_inh": 0.1},
           lambda_val=0.8, scl_v=1, mu_rsv=[0.020, 0.025, 0.050, 0.040], g_rec=1, tau_syn=0.01)
I_TARGET = LIQ["rho"] * (1 - LIQ["betaE"]) * LIQ["Uthr"]
MODES = {   # gamma_fix: Gamma(shape, scale) as intended; ain: "paper" = hard-coded value, "measured" = Eq. as described
    "published": {"gamma_fix": False, "ain": "paper"},
    "fixed":     {"gamma_fix": True,  "ain": "measured"},
    "fix_gamma": {"gamma_fix": True,  "ain": "paper"},
    "fix_ain":   {"gamma_fix": False, "ain": "measured"},
}
_TorchGamma = torch.distributions.Gamma

def build_liquid(input_size, mu_in, seed, gamma_fix, lambda_val=0.8, recurrence=True, g_rec=None):
    """seed_everything(seed) then the authors' constructor, exactly as in their scripts."""
    seed_everything(seed, deterministic=False)
    if gamma_fix:
        torch.distributions.Gamma = lambda conc, scale: _TorchGamma(conc, 1.0 / scale)   # (shape, scale) as intended
    try:
        model = LSM3D(input_size, LIQ["P_in"], mu_in, LIQ["betaE"], LIQ["betaI"], LIQ["reservoir_dim"],
                      LIQ["excit_ratio"], LIQ["dt"], LIQ["num_sim_steps"], LIQ["C_vals"], lambda_val, LIQ["scl_v"],
                      np.array(LIQ["mu_rsv"]), LIQ["g_rec"] if g_rec is None else g_rec, LIQ["tau_syn"], DEVICE).to(DEVICE)
    finally:
        torch.distributions.Gamma = _TorchGamma
    if not recurrence:
        model.rsv_weights = torch.zeros_like(model.rsv_weights)
    return model

def structure_stats(model):
    W = model.rsv_weights.detach().cpu().numpy(); Win = model.input_rsv_weights.detach().cpu().numpy()
    E = model.is_excit.cpu().numpy()
    out = {"n_recurrent_synapses": int((W != 0).sum()), "mean_recurrent_in_degree": float((W != 0).sum(0).mean()),
           "neurons_without_recurrent_input": int(((W != 0).sum(0) == 0).sum()),
           "n_input_synapses": int((Win != 0).sum()), "mean_input_in_degree": float((Win != 0).sum(0).mean()),
           "mean_nonzero_input_weight": float(Win[Win != 0].mean()) if (Win != 0).any() else 0.0,
           "threshold": LIQ["Uthr"], "n_excitatory": int(E.sum())}
    for name, pre, post in (("EE", E, E), ("EI", E, ~E), ("IE", ~E, E), ("II", ~E, ~E)):
        blk = W[np.ix_(pre, post)]
        nz = blk[blk != 0]
        out[f"{name}_synapses"] = int(nz.size)
        out[f"{name}_mean_abs_weight"] = float(np.abs(nz).mean()) if nz.size else 0.0
    return out

def save_reservoir(model, meta):
    rid = stable_hash(meta, 12)
    path = os.path.join(WORK_DIR, "reservoirs", f"{rid}.pt")
    if not os.path.exists(path):
        Win = model.input_rsv_weights.detach().cpu(); W = model.rsv_weights.detach().cpu()
        iw, ir = Win.nonzero().T, W.nonzero().T
        torch.save({"meta": meta, "input_size": Win.shape[0], "n_neurons": W.shape[0],
                    "win_index": iw.to(torch.int16), "win_value": Win[iw[0], iw[1]].float(),
                    "wrec_index": ir.to(torch.int16), "wrec_value": W[ir[0], ir[1]].float(),
                    "is_excit": model.is_excit.cpu(), "beta": torch.where(model.is_excit.cpu(), LIQ["betaE"], LIQ["betaI"]).float(),
                    "structure": structure_stats(model), "torch": torch.__version__, "device": str(DEVICE)}, path)
        disk_add(os.path.getsize(path))
    return rid

WIN, NWIN, N_NEUR = 20, 32, 343          # fr_windows(win=20, stride=20) over 650 steps -> 32 windows

def simulate(model, Xn, enc, batch, num_levels=200, rr=None, rr_thr=None, y=None, enc_seed=0):
    """Encode + run the liquid exactly as the authors' loop does. Returns spike counts per (window, neuron)
    as uint8 (N, 32*343) in the authors' feature order (window-major), plus liquid diagnostics."""
    torch.manual_seed(enc_seed)
    N = len(Xn)
    out = np.empty((N, NWIN * N_NEUR), np.uint8)
    Win, Wr = model.input_rsv_weights, model.rsv_weights
    neur = torch.zeros(N_NEUR, dtype=torch.float64, device=DEVICE)
    per_beat = np.empty(N, np.float32)
    acc = {"in_spikes": 0.0, "in_steps": 0, "Iin_sum": 0.0, "Iin_steps": 0, "Irec_abs_sum": 0.0}
    t_enc = t_sim = 0.0
    with torch.no_grad():
        for s in range(0, N, batch):
            t0 = time.time()
            x = encode_t(Xn[s:s + batch], enc, num_levels)
            if rr is not None:
                x = torch.cat([x, torch.from_numpy(rr_channels(rr[s:s + batch], rr_thr, x.shape[1])).to(DEVICE)], dim=2)
            if DEVICE.type == "cuda":
                torch.cuda.synchronize()
            t_enc += time.time() - t0; t0 = time.time()
            B, Tin, C = x.shape
            spk = torch.zeros(B, N_NEUR, device=DEVICE); mem = torch.zeros(B, N_NEUR, device=DEVICE)
            zero_in = torch.zeros(B, C, device=DEVICE)
            cnt = torch.zeros(B, NWIN, N_NEUR, dtype=torch.int16, device=DEVICE)
            Iin_acc = torch.zeros((), dtype=torch.float64, device=DEVICE); Irec_acc = torch.zeros((), dtype=torch.float64, device=DEVICE)
            for t in range(LIQ["num_sim_steps"]):
                x_t = x[:, t, :].float() if t < Tin else zero_in
                I_in = x_t @ Win
                I_rec = spk @ Wr
                I = (I_in + I_rec) * model.dt
                spk, mem = model.lif(I, mem)
                if t < NWIN * WIN:
                    cnt[:, t // WIN] += spk.to(torch.int16)
                if t < Tin:
                    Iin_acc += I_in.mean(dtype=torch.float64)
                Irec_acc += I_rec.abs().mean(dtype=torch.float64)
            c = cnt.reshape(B, -1)
            out[s:s + B] = c.to(torch.uint8).cpu().numpy()
            neur += cnt.sum((0, 1), dtype=torch.float64)
            per_beat[s:s + B] = c.sum(1).float().cpu().numpy()
            acc["in_spikes"] += float(x.sum(dtype=torch.int64)); acc["in_steps"] += B * Tin
            acc["Iin_sum"] += float(Iin_acc) * B; acc["Iin_steps"] += Tin * B
            acc["Irec_abs_sum"] += float(Irec_acc) * B
            if DEVICE.type == "cuda":
                torch.cuda.synchronize()
            t_sim += time.time() - t0
            del x, cnt, spk, mem
    neur = neur.cpu().numpy()
    steps = NWIN * WIN
    E = model.is_excit.cpu().numpy()
    stats = {"liquid_firing_prob_per_step": float(neur.sum() / (N * steps * N_NEUR)),
             "excitatory_firing_prob": float(neur[E].sum() / (N * steps * E.sum())),
             "inhibitory_firing_prob": float(neur[~E].sum() / (N * steps * (~E).sum())),
             "silent_neurons": int((neur == 0).sum()), "silent_fraction": float((neur == 0).mean()),
             "liquid_spikes_per_beat_mean": float(per_beat.mean()),
             "input_spikes_per_step": acc["in_spikes"] / max(acc["in_steps"], 1),
             "mean_input_current_per_neuron_step": acc["Iin_sum"] / max(acc["Iin_steps"], 1),
             "input_current_over_target": (acc["Iin_sum"] / max(acc["Iin_steps"], 1)) / I_TARGET,
             "mean_abs_recurrent_current": acc["Irec_abs_sum"] / max(N * LIQ["num_sim_steps"], 1),
             "n_beats": N, "encode_seconds": t_enc, "simulate_seconds": t_sim}
    if y is not None:
        for c, nm in enumerate(CLASS_NAMES):
            if (y == c).any():
                stats[f"liquid_spikes_per_beat_{nm}"] = float(per_beat[y == c].mean())
    return out, stats

def liquid_raster(model, S, batch=256):
    """Full spike raster (B, 650, 343) with the same update as simulate(), on the device. Also returns the mean
    |recurrent current| and mean input current per neuron and step (input steps only)."""
    Win, Wr = model.input_rsv_weights, model.rsv_weights
    T = LIQ["num_sim_steps"]; out = np.zeros((len(S), T, N_NEUR), bool); irec = iin = 0.0; tin = 1
    with torch.no_grad():
        for s in range(0, len(S), batch):
            x = torch.from_numpy(np.ascontiguousarray(S[s:s + batch])).to(DEVICE).float()
            B, tin, Cin = x.shape
            spk = torch.zeros(B, N_NEUR, device=DEVICE); mem = torch.zeros(B, N_NEUR, device=DEVICE)
            zero_in = torch.zeros(B, Cin, device=DEVICE)
            r = torch.zeros(B, T, N_NEUR, dtype=torch.bool, device=DEVICE)
            for t in range(T):
                I_in = (x[:, t, :] if t < tin else zero_in) @ Win
                I_rec = spk @ Wr
                spk, mem = model.lif((I_in + I_rec) * model.dt, mem)
                r[:, t] = spk > 0
                irec += float(I_rec.abs().mean()) * B
                if t < tin:
                    iin += float(I_in.mean()) * B
            out[s:s + B] = r.cpu().numpy()
    return out, irec / (len(S) * T), iin / (len(S) * tin)

def counts_to_rates(counts_rows):
    """Authors' features: window mean of 0/1 spikes in float64, stored float32."""
    return (counts_rows.astype(np.float64) / WIN).astype(np.float32)


## 7. Non-liquid controls and neural-network baselines

In [8]:
# Non-liquid feature sets and neural-network baselines.
def encoder_window_counts(Xn, enc, num_levels=200, batch=2048):
    """'No liquid' control: the encoder's own spikes counted in the same 32 windows of 20 steps."""
    C = num_levels if enc == "multi" else N_IN[enc]
    out = np.zeros((len(Xn), NWIN, C), np.uint8)
    for s in range(0, len(Xn), batch):
        S = encode(Xn[s:s + batch], enc, num_levels)
        T = min(S.shape[1], NWIN * WIN)
        Sp = np.zeros((S.shape[0], NWIN * WIN, C), np.uint8); Sp[:, :T] = S[:, :T]
        out[s:s + batch] = Sp.reshape(S.shape[0], NWIN, WIN, C).sum(2)
    return out

def huang_matrices(seed, n, dim):
    """Huang Eq. 1 uses F = X (A+)^-1, the pseudo-inverse of the random matrix A (dim x 200): returns pinv(A), 200 x dim."""
    A = np.random.default_rng(seed).standard_normal((n, dim, 200))
    return [np.linalg.pinv(a) for a in A]

def hybrid_rule(y_pred, ratio, rule, threshold):
    """Huang's SVEB rule on beats not predicted VEB. replace: SVEB iff ratio < threshold, else N.
    add: keep the model's SVEB calls and also call SVEB every beat with ratio < threshold."""
    y_pred = np.asarray(y_pred); low = np.asarray(ratio) < threshold
    sveb = low if rule == "replace" else (low | (y_pred == 1))
    return np.where(y_pred == 2, 2, np.where(sveb, 1, 0)).astype(np.int64)

def huang_fit_predict(spec, tr, ev, Dtr, Dev, cancel=None):
    """One RBF SVM per random matrix on [projection, pre-RR in s], majority vote (ties go to the lower class index:
    N, then SVEB, then VEB), then the SVEB rule on the beats not voted VEB. Our three classes, no class weights (libsvm)."""
    votes = np.zeros((len(ev), 3), np.int16); models = []
    for P in huang_matrices(spec["seed"], spec["n_matrices"], spec["dim"]):
        if cancel is not None and cancel.is_set():
            raise Cancelled()
        Ftr = np.hstack([Dtr["huang_beats"][tr] @ P, Dtr["rr"][tr, :1]])
        Fev = np.hstack([Dev["huang_beats"][ev] @ P, Dev["rr"][ev, :1]])
        clf = SVC(C=spec["C"], kernel="rbf", gamma=spec["gamma"], cache_size=HUANG_SVM_CACHE_MB).fit(Ftr, Dtr["y"][tr])
        votes[np.arange(len(ev)), clf.predict(Fev).astype(int)] += 1
        models.append(clf)
    vote = votes.argmax(1)
    return hybrid_rule(vote, Dev["rr_mean_ratio"][ev], "replace", spec["threshold"]), vote, votes, models

def shape_features(Xn):
    return np.asarray(Xn, np.float32)[:, ::8]              # 73 samples of the normalized beat

class SmallCNN(nn.Module):
    def __init__(self, n_rr=0):
        super().__init__()
        # GroupNorm, not BatchNorm: BatchNorm's running statistics made eval-mode predictions unstable
        self.f = nn.Sequential(nn.Conv1d(1, 16, 7, padding=3), nn.GroupNorm(4, 16), nn.ReLU(), nn.MaxPool1d(2),
                               nn.Conv1d(16, 32, 5, padding=2), nn.GroupNorm(4, 32), nn.ReLU(), nn.MaxPool1d(2),
                               nn.Conv1d(32, 64, 5, padding=2), nn.GroupNorm(4, 64), nn.ReLU(), nn.AdaptiveAvgPool1d(1))
        self.h = nn.Sequential(nn.Linear(64 + n_rr, 32), nn.ReLU(), nn.Linear(32, 3))
    def forward(self, x, rr=None):
        z = self.f(x[:, None, :]).squeeze(-1)
        return self.h(torch.cat([z, rr], 1) if rr is not None else z)

class SmallGRU(nn.Module):
    def __init__(self, n_rr=0):
        super().__init__()
        self.g = nn.GRU(1, 64, batch_first=True)
        self.h = nn.Sequential(nn.Linear(64 + n_rr, 32), nn.ReLU(), nn.Linear(32, 3))
    def forward(self, x, rr=None):
        _, hN = self.g(x[:, ::4, None])                     # 577 -> 145 steps
        z = hN[-1]
        return self.h(torch.cat([z, rr], 1) if rr is not None else z)

# ---- Tunable versions for the baseline search (the fixed models above stay as the untuned reference) ----
DEFAULT_ARCH = {"cnn": {"layers": 3, "width": 16, "kernels": [7, 5, 5], "pool": "avg", "head": 32, "dropout": 0.0, "lr": 1e-3, "wd": 1e-4},
                "gru": {"hidden": 64, "layers": 1, "bidir": False, "ds": 4, "readout": "last", "head": 32, "dropout": 0.0,
                        "lr": 1e-3, "wd": 1e-4}}

class TunableCNN(nn.Module):
    def __init__(self, a, n_rr=0):
        super().__init__()
        layers, cin = [], 1
        for i in range(a["layers"]):
            c, k = min(64, a["width"] * 2 ** i), a["kernels"][i]
            layers += [nn.Conv1d(cin, c, k, padding=k // 2), nn.GroupNorm(4, c), nn.ReLU()]
            if a["dropout"] > 0:
                layers.append(nn.Dropout(a["dropout"]))
            if i < a["layers"] - 1:
                layers.append(nn.MaxPool1d(2))
            cin = c
        self.f, self.pool = nn.Sequential(*layers), a["pool"]
        nz = cin * {"avg": 1, "max": 1, "avgmax": 2, "flat8": 8}[a["pool"]]
        self.h = nn.Sequential(nn.Linear(nz + n_rr, a["head"]), nn.ReLU(), nn.Dropout(a["dropout"]), nn.Linear(a["head"], 3))
    def forward(self, x, rr=None):
        z = self.f(x[:, None, :])
        z = {"avg": lambda: z.mean(-1), "max": lambda: z.amax(-1), "avgmax": lambda: torch.cat([z.mean(-1), z.amax(-1)], 1),
             "flat8": lambda: nn.functional.adaptive_avg_pool1d(z, 8).flatten(1)}[self.pool]()
        return self.h(torch.cat([z, rr], 1) if rr is not None else z)

class TunableGRU(nn.Module):
    def __init__(self, a, n_rr=0):
        super().__init__()
        self.a = a
        self.g = nn.GRU(1, a["hidden"], num_layers=a["layers"], batch_first=True, bidirectional=a["bidir"],
                        dropout=a["dropout"] if a["layers"] > 1 else 0.0)
        nz = a["hidden"] * (2 if a["bidir"] else 1)
        self.h = nn.Sequential(nn.Linear(nz + n_rr, a["head"]), nn.ReLU(), nn.Dropout(a["dropout"]), nn.Linear(a["head"], 3))
    def forward(self, x, rr=None):
        out, hN = self.g(x[:, ::self.a["ds"], None])
        if self.a["readout"] == "mean":
            z = out.mean(1)
        else:
            z = torch.cat([hN[-2], hN[-1]], 1) if self.a["bidir"] else hN[-1]
        return self.h(torch.cat([z, rr], 1) if rr is not None else z)

def nn_params(kind, arch, n_rr=0):
    m = (TunableCNN if kind == "cnn" else TunableGRU)(arch, n_rr)
    return sum(p.numel() for p in m.parameters())

def nn_cost(kind, arch):
    """Compute of one forward pass relative to the untuned model (used only to estimate training time)."""
    def flops(a):
        if kind == "cnn":
            L, cin, tot = 577, 1, 0
            for i in range(a["layers"]):
                c = min(64, a["width"] * 2 ** i); tot += L * cin * c * a["kernels"][i]; cin = c; L //= 2
            return tot
        steps, h = math.ceil(577 / a["ds"]), a["hidden"]
        return steps * (2 if a["bidir"] else 1) * sum(3 * h * ((1 if l == 0 else h * (2 if a["bidir"] else 1)) + h)
                                                       for l in range(a["layers"]))
    return flops(arch) / flops(DEFAULT_ARCH[kind])

NN_SPACE = {"cnn": {"layers": [2, 3, 4], "width": [8, 12, 16, 24, 32], "kernel": [5, 7, 9, 11], "pool": ["avg", "max", "avgmax", "flat8"],
                    "head": [32, 64], "dropout": [0.0, 0.1, 0.3], "lr": [3e-4, 1e-3, 3e-3], "wd": [0.0, 1e-4, 1e-3]},
            "gru": {"hidden": [32, 48, 64, 80, 96], "layers": [1, 2], "bidir": [False, True], "ds": [2, 4, 8], "readout": ["last", "mean"],
                    "head": [32, 64], "dropout": [0.0, 0.1, 0.3], "lr": [3e-4, 1e-3, 3e-3], "wd": [0.0, 1e-4, 1e-3]}}

def nn_search_space(kind):
    """NN_TUNE_TRIALS architectures for one model: trial 0 is the untuned model, the rest are random draws that stay
    under NN_PARAM_CAP parameters (counted with the RR inputs, so both input variants use the same list)."""
    rng = np.random.default_rng({"cnn": 101, "gru": 202}[kind])
    out, seen = [dict(DEFAULT_ARCH[kind])], {json.dumps(DEFAULT_ARCH[kind], sort_keys=True)}
    S = NN_SPACE[kind]
    for _ in range(5000):
        if len(out) >= NN_TUNE_TRIALS:
            break
        a = {k: v[rng.integers(len(v))] for k, v in S.items()}
        a = {k: (v.item() if hasattr(v, "item") else v) for k, v in a.items()}
        if kind == "cnn":
            a["kernels"] = [a.pop("kernel")] * a["layers"]
        key = json.dumps(a, sort_keys=True)
        if key in seen or nn_params(kind, a, len(RR_NAMES)) > NN_PARAM_CAP:
            continue
        seen.add(key); out.append(a)
    return out[:NN_TUNE_TRIALS]

def train_nn(kind, Xtr, ytr, Xev, rr_tr=None, rr_ev=None, seed=0, yev_for_curve=None, arch=None, epochs=None):
    """arch=None: the fixed untuned model (Adam 1e-3, wd 1e-4, NN_EPOCHS epochs). arch=dict: a tuned architecture."""
    torch.manual_seed(seed); np.random.seed(seed)
    torch.backends.cudnn.deterministic, torch.backends.cudnn.benchmark = True, False   # same result every session
    epochs = epochs or NN_EPOCHS
    use_rr = rr_tr is not None
    if use_rr:
        mu, sd = rr_tr.mean(0), rr_tr.std(0) + 1e-6
        rr_tr_s, rr_ev_s = (rr_tr - mu) / sd, (rr_ev - mu) / sd
    n_rr = rr_tr.shape[1] if use_rr else 0
    if arch is None:
        model = (SmallCNN if kind == "cnn" else SmallGRU)(n_rr=n_rr).to(DEVICE)
        lr, wd = 1e-3, 1e-4
    else:
        model = (TunableCNN if kind == "cnn" else TunableGRU)(arch, n_rr).to(DEVICE)
        lr, wd = arch["lr"], arch["wd"]
    cw = soft_class_weight(ytr)
    lossf = nn.CrossEntropyLoss(weight=torch.tensor([cw.get(c, 1.0) for c in range(3)], dtype=torch.float32, device=DEVICE))
    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=wd)
    # the training set lives on the GPU (same values as copying each batch, much less transfer)
    Xt = torch.tensor(np.asarray(Xtr, np.float32), device=DEVICE); yt = torch.tensor(ytr, dtype=torch.long, device=DEVICE)
    Rt = torch.tensor(rr_tr_s, dtype=torch.float32, device=DEVICE) if use_rr else None
    def predict(X, R):
        model.eval(); outs = []
        with torch.no_grad():
            for i in range(0, len(X), 2048):
                xb = torch.tensor(np.asarray(X[i:i + 2048], np.float32), device=DEVICE)
                rb = torch.tensor(R[i:i + 2048], dtype=torch.float32, device=DEVICE) if R is not None else None
                outs.append(model(xb, rb).float().cpu())
        model.train()
        return torch.cat(outs).numpy()
    curve = []
    g = torch.Generator().manual_seed(seed)
    sub = np.random.default_rng(0).choice(len(Xtr), min(4000, len(Xtr)), replace=False)
    cost = 1.0 if arch is None else nn_cost(kind, arch) ** 0.5
    for ep in range(epochs):
        t0 = time.time(); perm = torch.randperm(len(Xt), generator=g).to(DEVICE); tot = 0.0
        for i in range(0, len(Xt), 256):
            b = perm[i:i + 256]
            xb, yb = Xt[b], yt[b]
            rb = Rt[b] if use_rr else None
            opt.zero_grad(); l = lossf(model(xb, rb), yb); l.backward(); opt.step(); tot += float(l) * len(b)
        row = {"epoch": ep + 1, "train_loss": tot / len(Xt), "epoch_seconds": time.time() - t0,
               "train_sub_macro_f1": compute_metrics(ytr[sub], predict(Xtr[sub], rr_tr_s[sub] if use_rr else None).argmax(1))["macro_f1"]}
        if yev_for_curve is not None:
            row["eval_macro_f1"] = compute_metrics(yev_for_curve, predict(Xev, rr_ev_s if use_rr else None).argmax(1))["macro_f1"]
        curve.append(row)
        if arch is None:
            update_speed("nn_s_per_epoch_kbeat", row["epoch_seconds"] / (len(Xtr) / 1000), sub=kind)
        else:
            update_speed("nn_unit_s_per_epoch_kbeat", row["epoch_seconds"] / (len(Xtr) / 1000) / cost, sub=kind)
    logits = predict(Xev, rr_ev_s if use_rr else None)
    state = {"_torch_state": True, "kind": kind, "arch": arch, "epochs": epochs,
             "n_params": sum(p.numel() for p in model.parameters()),
             "state_dict": {k: v.cpu() for k, v in model.state_dict().items()},
             "rr_mean": mu if use_rr else None, "rr_std": sd if use_rr else None}
    return logits.argmax(1), logits.astype(np.float32), curve, state


## 8. Experiment plan

In [9]:
# The experiment plan: every run as a spec; identical specs across groups are run once.
ENCS = ["multi", "rate", "latency"]
CONFIG_IDS = SMOKE_CONFIGS if SMOKE_TEST else sorted(CONFIGS)
SPLITS_CV = [f"cv-r{r}f{f}" for r, f in CV_SPLITS]
def parse_split(s):
    m = re.match(r"cv-r(\d+)f(\d+)", s)
    return (int(m.group(1)), int(m.group(2))) if m else None

def lsm_spec(mode, seed, filt, norm, enc, readout, split, gain=1, lambda_val=0.8, recurrence=True,
             rr_in_liquid=False, ain_override=None, num_levels=200, rr_set=None, g_rec=None):
    s = {"kind": "lsm", "mode": mode, "seed": seed, "filter": filt, "norm": norm, "encoder": enc,
         "num_levels": num_levels if enc == "multi" else None, "gain": gain, "lambda_val": lambda_val,
         "recurrence": recurrence, "rr_in_liquid": rr_in_liquid, "ain_override": ain_override,
         "readout": readout, "split": split}
    if rr_set is not None:          # only new runs carry this key, so every earlier run id is unchanged
        s["rr_set"] = rr_set
    if g_rec is not None:           # likewise: recurrent gain of the working liquid
        s["g_rec"] = float(g_rec)
    return s

def feat_spec(model, split, enc=None, seed=None):
    return {"kind": "feat", "model": model, "encoder": enc, "seed": seed, "filter": "morph", "norm": "robust",
            "readout": "lda" if model == "lda" else "svm", "split": split}

def nn_spec(model, use_rr, seed, split, arch=None, epochs=None, trial=None):
    s = {"kind": "nn", "model": model, "use_rr": use_rr, "seed": seed, "filter": "morph", "norm": "robust", "split": split}
    if model == "cnn":
        s["norm_layer"] = "groupnorm"    # new run ids: CNN runs from before the BatchNorm fix are not reused
    if arch is not None:      # tuned baselines; the untuned spec keeps its original keys (and run ids)
        s.update({"arch": arch, "epochs": epochs, "trial": trial})
    return s

def huang_spec(seed, gamma, split):
    return {"kind": "huang", "seed": seed, "gamma": float(gamma), "C": HUANG_C, "n_matrices": HUANG_MATRICES, "dim": HUANG_DIM,
            "threshold": HUANG_THRESHOLD, "filter": "none", "norm": "zscore", "split": split}

def config_of(spec):
    if spec.get("kind") != "lsm" or spec["gain"] != 1 or spec["lambda_val"] != 0.8 or not spec["recurrence"] \
            or spec["rr_in_liquid"] or spec["num_levels"] not in (None, 200) or spec["readout"] == "svm_rr" \
            or spec.get("ain_override") is not None or spec.get("g_rec") is not None:   # k-stale / working liquid: not one of the 24
        return None
    for cid, c in CONFIGS.items():
        if (c["filter"], c["norm"], c["encoder"], c["readout"]) == (spec["filter"], spec["norm"], spec["encoder"], spec["readout"]):
            return cid
    return None

def run_id_of(spec):
    h = stable_hash({**spec, "_v": CODE_VERSION, "_smoke": SMOKE_TEST})
    k = spec["kind"]
    if k == "lsm":
        tag = [spec["mode"], f"s{spec['seed']}", f"{spec['filter']}-{spec['norm']}-{spec['encoder']}"
               + (f"K{spec['num_levels']}" if spec["num_levels"] not in (None, 200) else ""), spec["readout"]]
        if spec["gain"] != 1: tag.append(f"g{spec['gain']}")
        if spec["lambda_val"] != 0.8: tag.append(f"lam{spec['lambda_val']}")
        if not spec["recurrence"]: tag.append("norec")
        if spec["rr_in_liquid"]: tag.append("rrliq")
        if spec.get("rr_set"): tag.append(f"{spec['rr_set']}rr")
        if spec.get("g_rec") is not None: tag.append(f"grec{spec['g_rec']:g}")
        if spec["ain_override"] is not None: tag.append(f"ain{spec['ain_override']}")
    elif k == "feat":
        tag = [spec["model"]] + ([spec["encoder"]] if spec["encoder"] else []) + ([f"s{spec['seed']}"] if spec["seed"] is not None else [])
    elif k == "nn":
        t = spec.get("trial")
        tag = [spec["model"] + ("+rr" if spec["use_rr"] else "") + ("" if t is None else "-tuned" if t == "best" else f"-t{t}"),
               f"s{spec['seed']}"] + ([f"e{spec['epochs']}"] if t == "best" else [])
        if spec.get("rr_set"):
            tag[0] += f"-{spec['rr_set']}rr"
    elif k == "huang":
        tag = [f"g{spec['gamma']:.4g}", f"s{spec['seed']}"]
    elif k == "hybrid":
        tag = [spec["rule"], spec["base"].split("__")[1]]
    else:
        tag = [k]
    return f"{'SMOKE__' if SMOKE_TEST else ''}{k}__{'_'.join(tag)}__{spec.get('split', 'na')}__{h}"

def _retune_combos():
    grid = list(itertools.product(RETUNE_GRID["gain"], RETUNE_GRID["lambda_val"], RETUNE_GRID["num_levels"]))
    pick = np.random.default_rng(0).permutation(len(grid))[:RETUNE_TRIALS]
    return [dict(zip(("gain", "lambda_val", "num_levels"), grid[i])) for i in sorted(pick)]

RETUNE_STAGE1_SPLITS = SPLITS_CV if SMOKE_TEST else [f"cv-r1f{f}" for f in range(1, 5)]

def retune_choice():
    """Best stage-1 combination by mean Macro-F1 on DS1 CV (rep 1), or None if stage 1 is unfinished."""
    rows = []
    for c in _retune_combos():
        vals = []
        for sp in RETUNE_STAGE1_SPLITS:
            rid = run_id_of(lsm_spec("fixed", SEEDS[0], "morph", "robust", "multi", "svm", sp, **c))
            if rid not in CACHE.done or CACHE.done[rid][2] == "-":
                return None, None
            vals.append(float(CACHE.done[rid][2]))
        rows.append({**c, "cv_macro_f1": float(np.mean(vals))})
    df = pd.DataFrame(rows).sort_values("cv_macro_f1", ascending=False)
    return df.iloc[0][["gain", "lambda_val", "num_levels"]].to_dict(), df

NN_TUNE_SPLITS = RETUNE_STAGE1_SPLITS
NN_ARCHS = {m: nn_search_space(m) for m in ("cnn", "gru")}

def _run_metrics(rid):
    p = os.path.join(WORK_DIR, "runs", rid, "metrics.json")
    return json.load(open(p)) if os.path.exists(p) else None

def nn_tune_choice(model, use_rr):
    """Best (architecture, epoch count) by mean held-out Macro-F1 over the rep-1 DS1 folds, or None if unfinished."""
    rows = []
    for t, a in enumerate(NN_ARCHS[model]):
        per_fold = []
        for sp in NN_TUNE_SPLITS:
            rid = run_id_of(nn_spec(model, use_rr, SEEDS[0], sp, arch=a, epochs=NN_TUNE_MAX_EPOCHS, trial=t))
            m = _run_metrics(rid) if rid in CACHE.done else None
            if m is None or "eval_macro_f1_by_epoch" not in m:
                return None, None
            per_fold.append(m["eval_macro_f1_by_epoch"])
        for ep in NN_TUNE_CHECKPOINTS:
            rows.append({"model": model, "use_rr": use_rr, "trial": t, "epochs": ep,
                         "cv_macro_f1": float(np.mean([f[str(ep)] for f in per_fold])),
                         "cv_macro_f1_sd": float(np.std([f[str(ep)] for f in per_fold], ddof=1)) if len(per_fold) > 1 else np.nan,
                         "n_params": nn_params(model, a, len(RR_NAMES) if use_rr else 0), "arch": json.dumps(a, sort_keys=True)})
    df = pd.DataFrame(rows).sort_values(["cv_macro_f1", "trial", "epochs"], ascending=[False, True, True]).reset_index(drop=True)
    b = df.iloc[0]
    return {"trial": int(b.trial), "arch": NN_ARCHS[model][int(b.trial)], "epochs": int(b.epochs), "cv_macro_f1": float(b.cv_macro_f1)}, df

def working_diag_spec():
    return {"kind": "wdiag", "seed": SEEDS[0], "encoder": "multi", "filter": "morph", "norm": "robust",
            "lambda_val": WORKING_LAMBDA, "gains": list(WORKING_GAINS), "n_beats": WORKING_DIAG_BEATS,
            "t_perturb": WORKING_T_PERTURB, "max_firing": WORKING_MAX_FIRING, "split": "na"}

def working_choice():
    """g_rec picked by the finished diagnostic (checks/working_liquid_diag.json, tied to its run id), else None."""
    rid = run_id_of(working_diag_spec())
    p = os.path.join(WORK_DIR, "checks", "working_liquid_diag.json")
    if rid not in CACHE.done or not os.path.exists(p):
        return None
    d = json.load(open(p))
    return d["chosen_g_rec"] if d.get("run_id") == rid else None

def hybrid_bases():
    """(label, spec) of the runs the hybrid rule is applied to: published config 11, Multi + RR in the liquid
    (published and fixed), tuned GRU / CNN + RR; every seed, the final test and all CV splits."""
    nn_best = {m: nn_tune_choice(m, True)[0] for m in ("gru", "cnn")}
    out = []
    for seed in SEEDS:
        for sp in ["final"] + SPLITS_CV:
            out.append(("LSM published, config 11", lsm_spec("published", seed, "morph", "robust", "multi", "svm", sp)))
            for mode in ("published", "fixed"):
                out.append((f"LSM {mode}, Multi + RR in liquid",
                            lsm_spec(mode, seed, "morph", "robust", "multi", "svm", sp, rr_in_liquid=True)))
            for m, best in nn_best.items():
                if best is not None:
                    out.append((f"{m.upper()} tuned + RR",
                                nn_spec(m, True, seed, sp, arch=best["arch"], epochs=best["epochs"], trial="best")))
    return out

def build_plan():
    P = []
    def add(group, spec):
        if GROUPS_ENABLED.get(group, False):
            P.append((group, spec))
    for group in GROUP_ORDER:
        part = group.rsplit("_", 1)[-1]
        splits = ["final"] if part == "final" else SPLITS_CV
        if group == "checks":
            add(group, {"kind": "check", "split": "na"})
        elif group == "brian2":
            add(group, {"kind": "brian2", "split": "na", "beats": BRIAN2_BEATS})
        elif group.startswith("main_"):
            for modes in (["published", "fixed"], ["fix_gamma", "fix_ain"]):
                for seed in SEEDS:
                    for mode in modes:
                        for cid in CONFIG_IDS:
                            c = CONFIGS[cid]
                            for sp in splits:
                                add(group, lsm_spec(mode, seed, c["filter"], c["norm"], c["encoder"], c["readout"], sp))
        elif group.startswith("gain_"):
            for seed in SEEDS:
                for enc in ENCS:
                    for g in GAINS:
                        for sp in splits:
                            add(group, lsm_spec("fixed", seed, "morph", "robust", enc, "svm", sp, gain=g))
        elif group.startswith("ablation_"):
            for seed in SEEDS:
                for enc in ENCS:
                    for sp in splits:
                        for mode in ("published", "fixed"):
                            add(group, lsm_spec(mode, seed, "morph", "robust", enc, "svm", sp, recurrence=False))
                        for lam in DENSE_LAMBDAS:
                            add(group, lsm_spec("fixed", seed, "morph", "robust", enc, "svm", sp, lambda_val=lam))
                        add(group, feat_spec("no_liquid", sp, enc=enc, seed=seed if enc == "rate" else None))
                        add(group, feat_spec("elm", sp, enc=enc, seed=seed))
        elif group.startswith("rr_"):
            for sp in splits:
                for m in ("rr_only", "rr_shape", "lda"):
                    add(group, feat_spec(m, sp))
            for seed in SEEDS:
                for enc in ENCS:
                    for mode in ("published", "fixed"):
                        for sp in splits:
                            add(group, lsm_spec(mode, seed, "morph", "robust", enc, "svm_rr", sp))
                            add(group, lsm_spec(mode, seed, "morph", "robust", enc, "svm", sp, rr_in_liquid=True))
        elif group.startswith("baseline_"):
            for sp in splits:
                add(group, feat_spec("raw_beat", sp))
            for seed in SEEDS:
                for m in ("cnn", "gru"):
                    for use_rr in (False, True):
                        for sp in splits:
                            add(group, nn_spec(m, use_rr, seed, sp))
        elif group == "nn_tune":
            for m in ("cnn", "gru"):
                for use_rr in (False, True):
                    for t, a in enumerate(NN_ARCHS[m]):
                        for sp in NN_TUNE_SPLITS:
                            add(group, nn_spec(m, use_rr, SEEDS[0], sp, arch=a, epochs=NN_TUNE_MAX_EPOCHS, trial=t))
                    best, _ = nn_tune_choice(m, use_rr)
                    if best is not None:            # stage 2 appears once all of this variant's trials are done
                        for seed in SEEDS:
                            for sp in ["final"] + SPLITS_CV:
                                add(group, nn_spec(m, use_rr, seed, sp, arch=best["arch"], epochs=best["epochs"], trial="best"))
        elif group == "retune":
            for c in _retune_combos():
                for sp in RETUNE_STAGE1_SPLITS:
                    add(group, lsm_spec("fixed", SEEDS[0], "morph", "robust", "multi", "svm", sp, **c))
            best, _ = retune_choice()
            if best is not None:
                best = {"gain": int(best["gain"]), "lambda_val": float(best["lambda_val"]), "num_levels": int(best["num_levels"])}
                for seed in SEEDS:
                    for sp in ["final"] + SPLITS_CV:
                        add(group, lsm_spec("fixed", seed, "morph", "robust", "multi", "svm", sp, **best))
        elif group.startswith("kstale_"):
            for cid in CONFIG_IDS:
                c = CONFIGS[cid]
                if c["encoder"] == "multi":
                    continue
                for sp in splits:
                    add(group, lsm_spec("published", SEEDS[0], c["filter"], c["norm"], c["encoder"], c["readout"], sp,
                                        ain_override=PAPER_AIN["multi"]))
        elif group.startswith("huang_"):
            for seed in SEEDS:
                for g in HUANG_GAMMAS:
                    for sp in splits:
                        add(group, huang_spec(seed, g, sp))
        elif group.startswith("meanrr_") and group != "meanrr_nn":
            for sp in splits:
                for m in ("rr_only_mrr", "rr_shape_mrr"):
                    add(group, feat_spec(m, sp))
        elif group.startswith("elmrr_"):
            for seed in SEEDS:
                for enc in ENCS:
                    for sp in splits:
                        add(group, feat_spec("elm_rr", sp, enc=enc, seed=seed))
        elif group.startswith("rrliqmean_"):
            for seed in SEEDS:
                for enc in ENCS:
                    for mode in ("published", "fixed"):
                        for sp in splits:
                            add(group, lsm_spec(mode, seed, "morph", "robust", enc, "svm", sp, rr_in_liquid=True, rr_set="mean"))
        elif group == "working_diag":
            add(group, working_diag_spec())
        elif group.startswith("working_"):
            g = working_choice()
            if g is not None:                   # appears once the diagnostic has picked g_rec
                for seed in SEEDS:
                    for sp in splits:
                        for rec in (True, False):
                            for ro in ("svm", "svm_late"):
                                add(group, lsm_spec("fixed", seed, "morph", "robust", "multi", ro, sp,
                                                    lambda_val=WORKING_LAMBDA, recurrence=rec, g_rec=g))
        elif group == "meanrr_nn":
            for m in ("cnn", "gru"):
                best, _ = nn_tune_choice(m, True)
                if best is not None:            # appears once the tuned + RR architecture is known
                    for seed in SEEDS:
                        for sp in ["final"] + SPLITS_CV:
                            s = nn_spec(m, True, seed, sp, arch=best["arch"], epochs=best["epochs"], trial="best")
                            s["rr_set"] = "mean"
                            add(group, s)
        elif group == "hybrid" and GROUPS_ENABLED.get(group, False):
            for label, bspec in hybrid_bases():
                brid = run_id_of(bspec)
                if brid in CACHE.done and os.path.exists(os.path.join(WORK_DIR, "runs", brid, "predictions.npz")):
                    for rule in HYBRID_RULES:
                        add(group, {"kind": "hybrid", "base": brid, "base_label": label, "rule": rule,
                                    "threshold": HUANG_THRESHOLD, "seed": bspec["seed"], "split": bspec["split"]})
    seen, plan = {}, []
    for group, spec in P:
        rid = run_id_of(spec)
        if rid in seen:
            seen[rid]["groups"].append(group)
            continue
        e = {"rid": rid, "group": group, "groups": [group], "spec": spec}
        seen[rid] = e
        plan.append(e)
    if SHARE != "BOTH":              # each run belongs to the share of its (first) group: the shares never overlap
        plan = [e for e in plan if e["group"] in SHARE_GROUPS[SHARE]]
    return plan

def group_on(g):
    return GROUPS_ENABLED.get(g, False) and (SHARE == "BOTH" or g in SHARE_GROUPS[SHARE])

def states_key(spec):
    """Runs with the same states_key share one simulation (or one feature computation)."""
    sp = spec["split"]
    if spec["kind"] == "lsm":
        measured = MODES[spec["mode"]]["ain"] == "measured" and spec["ain_override"] is None
        if sp == "final":
            nk = "final"
        elif spec["norm"] == "standard" and not measured and not spec["rr_in_liquid"]:
            nk = "cv-shared"
        else:
            nk = sp
        core = {k: v for k, v in spec.items() if k not in ("readout", "split")}
        return ("lsm", json.dumps(core, sort_keys=True), nk)
    if spec["kind"] == "feat":
        core = {k: v for k, v in spec.items() if k not in ("readout", "split")}
        return ("feat", json.dumps(core, sort_keys=True), sp)
    return (spec["kind"], run_id_of(spec), sp)

def make_jobs(plan):
    jobs, index = [], {}
    for e in plan:
        k = states_key(e["spec"])
        if k not in index:
            index[k] = len(jobs)
            jobs.append({"key": k, "runs": []})
        jobs[index[k]]["runs"].append(e)
    return jobs

_p = build_plan()
_counts = pd.Series([e["group"] for e in _p]).value_counts().reindex(GROUP_ORDER).fillna(0).astype(int)
print(f"plan: {len(_p)} unique runs in {len(make_jobs(_p))} jobs")
print(_counts.to_string())

if SHARE != "BOTH":
    print(f"SHARE = {SHARE!r}: this machine runs the groups {', '.join(SHARE_GROUPS[SHARE])}")
    if GROUPS_ENABLED.get("hybrid"):
        print("NOTE: the hybrid group runs only with SHARE = 'BOTH' (it reads predictions made on both machines)")


plan: 7802 unique runs in 5152 jobs
checks                1
brian2                1
main_final          288
ablation_final       50
rr_final             39
baseline_final       13
gain_final           63
main_cv            3456
gain_cv             756
ablation_cv         600
rr_cv               468
baseline_cv         156
nn_tune             636
retune              143
kstale_final         16
kstale_cv           192
huang_final           6
huang_cv             72
meanrr_final          2
meanrr_cv            24
elmrr_final           9
elmrr_cv            108
rrliqmean_final      18
rrliqmean_cv        216
meanrr_nn            78
hybrid              390
working_diag          1
working_final         0
working_cv            0


## 9. Executors (incl. the Brian2 cross-check)

In [10]:
# Executors: one function per job kind. Each saves its runs and updates cache.txt as it goes.
# With PIPELINE=True the GPU-side work (simulation, encoding, NN training) runs in the main thread, one job at a time,
# and every classifier fit becomes a task for the FIT_POOL: one fit per CPU core, in parallel with the next simulation.
def _avail_ram_gb():
    try:
        for line in open("/proc/meminfo"):
            if line.startswith("MemAvailable"):
                return int(line.split()[1]) / 1e6
    except Exception:
        pass
    return 8.0

def _proc_ram_gb():
    """(current, peak) resident memory of this process."""
    out = {"VmRSS": 0.0, "VmHWM": 0.0}
    try:
        for line in open("/proc/self/status"):
            k = line.split(":")[0]
            if k in out:
                out[k] = int(line.split()[1]) / 1e6
    except Exception:
        pass
    return out["VmRSS"], out["VmHWM"]

def fit_n_jobs(threads):
    """Cores per classifier fit: spare cores go to the fit's 3 one-vs-rest problems (results unchanged)."""
    return max(1, min(3, READOUT_JOBS // max(threads, 1)))

def fit_ram_gb(nfeat, ntr=4096):
    """Working memory of one fit on the batch-by-batch path (batches, their float64 temporaries, the curve subset)."""
    return min(4096, ntr) * nfeat * 4 * 6 / 1e9 + min(4000, ntr) * nfeat * 4 / 1e9 + 0.3

def _readout_plan(n_runs, ntr, nev, nfeat):
    """Older runner: threads and whether to pre-scale: as many parallel fits as cores allow; pre-scale only if RAM fits."""
    avail = _avail_ram_gb() - 2.5
    per_batch = 4096 * nfeat * 4 * 6 / 1e9 + 0.3
    per_pre = ntr * nfeat * 4 * 1.15 / 1e9 + per_batch
    available = max(1, min(READOUT_JOBS, int(avail // max(per_batch, 0.05))))
    update_speed("fit_threads", float(available))      # parallel fits the machine allows (for the projection)
    threads = min(available, n_runs)
    return threads, threads * per_pre < avail

def _split_rows(split, n1):
    """train rows, eval rows (into DS1 or DS2), readout rng seed (authors' formulas)."""
    if split == "final":
        return np.arange(n1), None, 2026
    rep, fold = parse_split(split)
    f = FOLD_BY[(rep, fold)]
    return f["train_idx"], f["val_idx"], 2026 + rep * 10 + fold

def _finish_run(e, metrics, eval_idx, eval_pid, y_true, y_pred, scores, model_obj, curve, liq, timing, extra=None):
    save_run(e["rid"], e["spec"], e["group"], metrics, eval_idx, eval_pid, y_true, y_pred, scores, model_obj,
             curve, liq, timing, {"groups": e["groups"], "config_id": config_of(e["spec"]), "share": SHARE,
                                  "machine_id": MACHINE_ID, "notebook_build": NOTEBOOK_BUILD, **(extra or {})})
    CACHE.mark_done(e["rid"], timing.get("total_seconds", 0.0), metrics, e["group"])
    log(f"  done {e['rid'][:90]:<90} mF1={metrics['macro_f1']:.4f} acc={metrics['accuracy']:.4f} "
        f"Se S/V={metrics['recall_SVEB']:.3f}/{metrics['recall_VEB']:.3f}")

def _record_failure(rid, tb):
    CACHE.mark_failed(rid, tb.strip().splitlines()[-1])
    os.makedirs(os.path.join(WORK_DIR, "logs", "failures"), exist_ok=True)
    atomic_write_text(os.path.join(WORK_DIR, "logs", "failures", rid + ".txt"), tb)
    log("  FAILED", rid, tb.strip().splitlines()[-1])

# ---------------- pipeline: RAM accounting and the fit-worker pool ----------------
class RamBudget:
    """Bookkeeping of RAM promised to jobs (their spike counts) and fits (their working memory)."""
    def __init__(self, total_gb):
        self.total, self.used, self.cv = total_gb, 0.0, threading.Condition()
    def reserve(self, gb, headroom=0.0, block=True):
        """Waits until gb (+ headroom kept free for fits) fits in the budget; something bigger than the whole budget
        waits until nothing else holds RAM."""
        t0, warned = time.time(), 0
        with self.cv:
            while self.used > 1e-6 and self.used + gb + headroom > self.total:
                if not block:
                    return False
                self.cv.wait(2.0)
                if time.time() - t0 > 300 * (warned + 1):
                    warned += 1
                    log(f"  waiting for RAM for {(time.time() - t0) / 60:.0f} min: need {gb:.2f}+{headroom:.2f} GB, "
                        f"{self.used:.2f} of {self.total:.2f} GB promised, {_avail_ram_gb():.1f} GB actually available")
            self.used += gb
            return True
    def release(self, gb):
        with self.cv:
            self.used -= gb
            if self.used < 1e-6:
                self.used = 0.0              # no floating-point leftovers
            self.cv.notify_all()

class SharedPrep:
    """Scaler (and, if RAM allows, the scaled training matrix) shared by the fits of one job on the same rows."""
    def __init__(self):
        self.lock, self.items, self.extra_gb = threading.Lock(), {}, 0.0
    def get(self, key, Xtr_fn, ntr, nfeat, ram):
        with self.lock:
            if key not in self.items:
                xs_gb = ntr * nfeat * 4 * 1.05 / 1e9
                pre = ram.reserve(xs_gb, block=False)
                if pre:
                    self.extra_gb += xs_gb
                self.items[key] = prepare_train(Xtr_fn, ntr, pre)
            return self.items[key]

class FitPool:
    def __init__(self, n_workers, ram):
        self.n, self.ram = n_workers, ram
        self.ex = ThreadPoolExecutor(n_workers, thread_name_prefix="fit")
        self.lock, self.cancel = threading.Lock(), threading.Event()
        self.queued = self.running = 0
        self.backlog_s = 0.0                 # estimated seconds of fits not yet finished
        self.jobs, self.next_id, self.futs = {}, 0, set()
        self.n_done = self.n_failed = self.n_cancelled = 0
    def submit_job(self, tasks, job_ram_gb, shared):
        with self.lock:
            jid = self.next_id; self.next_id += 1
            self.jobs[jid] = {"left": len(tasks), "ram": job_ram_gb, "shared": shared}
            for t in tasks:
                self.queued += 1; self.backlog_s += t["est_s"]
                fu = self.ex.submit(self._run, jid, t)
                self.futs.add(fu); fu.add_done_callback(self.futs.discard)
    def _job_task_done(self, jid):
        with self.lock:
            j = self.jobs[jid]; j["left"] -= 1
            if j["left"] == 0:
                self.ram.release(j["ram"] + j["shared"].extra_gb)
                del self.jobs[jid]            # drops the job's spike counts and shared matrices
    def _run(self, jid, t):
        with self.lock:
            self.queued -= 1
        reserved = started = False
        try:
            if self.cancel.is_set():
                raise Cancelled()
            self.ram.reserve(t["ram_gb"]); reserved = True
            with self.lock:
                self.running += 1; busy = self.running
            started = True
            t0 = time.time()
            res = t["fit"](self.cancel, self.jobs[jid]["shared"])
            dt = time.time() - t0
            with self.lock:
                busy = min(busy, self.running); self.running -= 1
            started = False
            if t.get("gcells") and busy >= max(1, 0.75 * self.n):   # speed under full load (SGD fits only)
                update_speed("fit_s_per_Gcell", dt / t["gcells"], sub="pipe", robust=True)
            t["record"](res, dt, busy)
            with self.lock:
                self.n_done += 1
        except Cancelled:
            with self.lock:
                self.n_cancelled += 1
            log(f"  cancelled (time is up; stays pending): {t['rid'][:90]}")
        except Exception:
            with self.lock:
                self.n_failed += 1
            _record_failure(t["rid"], traceback.format_exc())
        finally:
            with self.lock:
                if started:
                    self.running -= 1
                self.backlog_s -= t["est_s"]
            if reserved:
                self.ram.release(t["ram_gb"])
            self._job_task_done(jid)
    def outstanding(self):
        with self.lock:
            return self.queued + self.running
    def wait_any(self, timeout=5.0):
        futs = list(self.futs)
        if futs:
            concurrent.futures.wait(futs, timeout=timeout, return_when=concurrent.futures.FIRST_COMPLETED)

FIT_POOL = None          # created by the runner when PIPELINE=True
_JOB_CTX = {}            # the job the main thread is executing: its RAM reservation is handed to the pool

def _run_readouts(runs, feat_tr, feat_ev, ytr_all, D_eval, final, liq, base_timing, ntr_full):
    """feat_tr(rows)->float32 train features; feat_ev(rows)->float32 eval features (DS2 rows if final, else DS1)."""
    D_TR_RR = DATA["morph"]["DS1"]["rr"]
    D_EV_RR = D_eval["rr"]
    nfeat0 = len(feat_tr(np.arange(1)).ravel())
    LATE = slice((NWIN - 1) * N_NEUR, NWIN * N_NEUR)
    late_of = lambda e: e["spec"].get("readout") == "svm_late"
    nfeat_of = lambda e: (N_NEUR if late_of(e) else nfeat0) + (len(RR_NAMES) if e["spec"].get("readout") == "svm_rr" else 0)
    def parts(e):
        tr, ev, rng_seed = _split_rows(e["spec"]["split"], ntr_full)
        if final:
            ev = np.arange(len(D_eval["y"]))
        add_rr = e["spec"].get("readout") == "svm_rr"
        if late_of(e):
            ftr = lambda r: feat_tr(tr[r])[:, LATE]; fev = lambda r: feat_ev(ev[r])[:, LATE]
        else:
            ftr = (lambda r: np.hstack([feat_tr(tr[r]), D_TR_RR[tr[r]]])) if add_rr else (lambda r: feat_tr(tr[r]))
            fev = (lambda r: np.hstack([feat_ev(ev[r]), D_EV_RR[ev[r]]])) if add_rr else (lambda r: feat_ev(ev[r]))
        return tr, ev, rng_seed, add_rr, ftr, fev
    def one(e, nj, prescale=False, cancel=None, shared=None):
        tr, ev, rng_seed, add_rr, ftr, fev = parts(e)
        yev = D_eval["y"][ev]
        loss = "log_loss" if e["spec"].get("readout") == "lr" else "hinge"
        nfeat = nfeat_of(e)
        prep = shared.get((e["spec"]["split"], add_rr, late_of(e)), ftr, len(tr), nfeat, FIT_POOL.ram) if shared is not None else None
        scaler, clf, yp, sc, curve = fit_readout(ftr, len(tr), ytr_all[tr], fev, len(ev), loss, rng_seed,
                                                 yev_for_curve=None if final else yev, prescale=prescale, n_jobs=nj,
                                                 prepared=prep, cancel=cancel)
        pre = prep[1] is not None if prep is not None else prescale
        return e, compute_metrics(yev, yp), ev, D_eval["pid"][ev], yev, yp, sc, {"scaler": scaler, "classifier": clf}, curve, pre
    def record(res, dt, threads, nj):
        e, m, ev, pid, yev, yp, sc, mobj, curve, pre = res
        _finish_run(e, m, ev, pid, yev, yp, sc, mobj, curve, liq,
                    {**base_timing, "readout_seconds": dt, "total_seconds": base_timing.get("shared_seconds", 0) + dt,
                     "readout_threads": threads, "readout_cores_per_fit": nj, "prescaled": pre,
                     "runner": "pipeline" if FIT_POOL is not None else "sequential"})
    if FIT_POOL is not None:
        tasks = []
        for e in runs:
            tr = _split_rows(e["spec"]["split"], ntr_full)[0]
            nfeat = nfeat_of(e)
            g = len(tr) * nfeat / 1e9
            tasks.append({"rid": e["rid"], "gcells": g, "est_s": g * SPEED["fit_s_per_Gcell"]["pipe"], "ram_gb": fit_ram_gb(nfeat, len(tr)),
                          "fit": (lambda cancel, shared, e=e: one(e, 1, cancel=cancel, shared=shared)),
                          "record": (lambda res, dt, busy: record(res, dt, busy, 1))})
        FIT_POOL.submit_job(tasks, _JOB_CTX.get("ram_gb", 0.0), SharedPrep())
        _JOB_CTX["submitted"] = True
        return
    # older runner: this job's fits in parallel threads, sharing the cores
    n_ev = len(D_eval["y"]) if final else int(np.mean([len(_split_rows(e["spec"]["split"], ntr_full)[1]) for e in runs]))
    n_tr = ntr_full if final else int(ntr_full * 0.76)
    nfeat = nfeat0 + (len(RR_NAMES) if any(e["spec"].get("readout") == "svm_rr" for e in runs) else 0)
    threads, prescale = _readout_plan(len(runs), n_tr, n_ev, nfeat)
    nj = fit_n_jobs(threads)
    def timed(e):
        t0 = time.time(); r = one(e, nj, prescale); dt = time.time() - t0
        tr = _split_rows(e["spec"]["split"], ntr_full)[0]
        update_speed("fit_s_per_Gcell", dt / (len(tr) * nfeat / 1e9), sub=str(nj))
        return r, dt
    with ThreadPoolExecutor(threads) as ex:
        futs = [ex.submit(timed, e) for e in runs]
        for fu, e in zip(futs, runs):
            try:
                res, dt = fu.result()
                record(res, dt, threads, nj)
            except Exception:
                _record_failure(e["rid"], traceback.format_exc())

def _maybe_save_states(runs, counts_test, D2):
    for e in runs:
        s = e["spec"]
        if s["split"] != "final":
            continue
        cid = config_of(s)
        for want in SAVE_STATES:
            if cid == want["config_id"] and s["mode"] == want["mode"] and s["seed"] == want["seed"]:
                p = os.path.join(WORK_DIR, "states", f"{e['rid']}__DS2_window_spike_counts.npz")
                if not os.path.exists(p):
                    np.savez_compressed(p, counts=counts_test, y=D2["y"], patient_id=D2["pid"].astype("U4"),
                                        layout="(beat, window*343 + neuron) spike counts per 20-step window; rate = count/20")
                    disk_add(os.path.getsize(p))
                    log(f"  saved state vectors -> {os.path.basename(p)} ({os.path.getsize(p)/1e6:.0f} MB)")

def execute_lsm(job, runs):
    s0 = runs[0]["spec"]; nk = job["key"][2]; final = nk == "final"
    enc, mode, K = s0["encoder"], s0["mode"], s0["num_levels"] or 200
    D1, D2 = DATA[s0["filter"]]["DS1"], DATA[s0["filter"]]["DS2"]
    t0 = time.time()
    if final:
        X1n, X2n, normp = normalize_final(D1["X"], D2["X"], s0["norm"])
        tr_rows = np.arange(len(D1["y"]))
    elif nk == "cv-shared":
        X1n, normp = normalize_cv(D1["X"], None, "standard"); X2n = None
        tr_rows = np.arange(len(D1["y"]))
    else:
        f = FOLD_BY[parse_split(nk)]
        X1n, normp = normalize_cv(D1["X"], f["train_idx"], s0["norm"]); X2n = None
        tr_rows = f["train_idx"]
    rr1 = rr2 = None
    if s0["rr_in_liquid"]:           # 8 RR columns (30 channels), or 9 with the mean-RR ratio (40 channels)
        mean_set = s0.get("rr_set") == "mean"
        rr1 = rr_inputs(D1, np.arange(len(D1["y"])), mean_set); rr2 = rr_inputs(D2, np.arange(len(D2["y"])), mean_set)
    rr_thr = rr_thresholds(rr1[tr_rows]) if rr1 is not None else None
    if s0["ain_override"] is not None:
        ain, src = float(s0["ain_override"]), "override"
    elif MODES[mode]["ain"] == "paper":
        ain, src = PAPER_AIN[enc], "paper (hard-coded)"
    else:
        ain = measure_ain(X1n, tr_rows, enc, K, rr=rr1, rr_thr=rr_thr)
        src = "measured on training rows"
    mu_in = I_TARGET / (ain * LIQ["P_in"]) * s0["gain"]
    n_in = (K if enc == "multi" else N_IN[enc]) + (len(rr_thr) * RR_LEVELS if rr_thr is not None else 0)
    model = build_liquid(n_in, mu_in, s0["seed"], MODES[mode]["gamma_fix"], s0["lambda_val"], s0["recurrence"], s0.get("g_rec"))
    res_meta = {"mode": mode, "seed": s0["seed"], "input_size": n_in, "mu_in": repr(mu_in),
                "lambda_val": s0["lambda_val"], "recurrence": s0["recurrence"],
                "gamma_fix": MODES[mode]["gamma_fix"], "torch": torch.__version__, "device": DEVICE.type}
    if s0.get("g_rec") is not None:
        res_meta["g_rec"] = s0["g_rec"]
    res_id = save_reservoir(model, res_meta)
    enc_seed = int(stable_hash(job["key"], 6), 16)
    batch = SIM_BATCH_FINAL if final else SIM_BATCH_CV
    C1, st1 = simulate(model, X1n, enc, batch, K, rr=rr1, rr_thr=rr_thr, y=D1["y"], enc_seed=enc_seed)
    st2 = None
    if final:
        C2, st2 = simulate(model, X2n, enc, batch, K, rr=rr2, rr_thr=rr_thr, y=D2["y"], enc_seed=enc_seed + 1)
    n_sim = st1["n_beats"] + (st2["n_beats"] if st2 else 0)
    sim_s = st1["encode_seconds"] + st1["simulate_seconds"] + ((st2["encode_seconds"] + st2["simulate_seconds"]) if st2 else 0)
    update_speed("sim_beats_per_s", n_sim / max(sim_s, 1e-6), sub=enc)
    enc_s = st1["encode_seconds"] + (st2["encode_seconds"] if st2 else 0)
    update_speed("encode_beats_per_s", n_sim / max(enc_s, 1e-6), sub=enc)
    liq = {"A_in_used": ain, "A_in_source": src, "A_in_paper": PAPER_AIN[enc], "mu_in": mu_in, "I_target": I_TARGET,
           "normalization": normp, "reservoir_id": res_id, "structure": structure_stats(model),
           "DS1_simulation": st1, "DS2_simulation": st2}
    shared = time.time() - t0
    update_speed("lsm_job_overhead_s", max(0.0, shared - sim_s) + 1.0)   # + saving / bookkeeping between jobs
    log(f"  liquid: A_in={ain:.4f} ({src}) mu_in={mu_in:.4f} firing={st1['liquid_firing_prob_per_step']:.4f} "
        f"drive/target={st1['input_current_over_target']:.2f} silent={st1['silent_neurons']} sim={sim_s:.0f}s")
    if final:
        _maybe_save_states(runs, C2, D2)
    feat_tr = lambda rows: counts_to_rates(C1[rows])
    feat_ev = (lambda rows: counts_to_rates(C2[rows])) if final else feat_tr
    _run_readouts(runs, feat_tr, feat_ev, D1["y"], D2 if final else D1, final, liq,
                  {"shared_seconds": shared, "simulation_seconds": sim_s, "beats_simulated": n_sim}, len(D1["y"]))
    # (the spike counts stay alive through the fit closures until the last fit of this job finishes)

def _front_end(split):
    D1, D2 = DATA["morph"]["DS1"], DATA["morph"]["DS2"]
    if split == "final":
        X1n, X2n, normp = normalize_final(D1["X"], D2["X"], "robust")
        return D1, D2, np.asarray(X1n, np.float32), np.asarray(X2n, np.float32), np.arange(len(D1["y"])), normp
    f = FOLD_BY[parse_split(split)]
    X1n, normp = normalize_cv(D1["X"], f["train_idx"], "robust")
    return D1, D2, X1n, None, f["train_idx"], normp

def rr_inputs(D, rows, mean_ratio=False):
    """The 8 RR features, plus pre-RR / recording mean RR as a 9th when mean_ratio."""
    return np.hstack([D["rr"][rows], D["rr_mean_ratio"][rows, None]]) if mean_ratio else D["rr"][rows]

def execute_feat(job, runs):
    s0 = runs[0]["spec"]; split = job["key"][2]; final = split == "final"
    t0 = time.time()
    D1, D2, X1n, X2n, tr_rows, normp = _front_end(split)
    model_name, enc = s0["model"], s0["encoder"]
    info = {"normalization": normp, "feature_model": model_name}
    if model_name in ("no_liquid", "elm", "elm_rr"):
        if s0["seed"] is not None:
            torch.manual_seed(int(stable_hash(job["key"], 6), 16))
        c1 = encoder_window_counts(X1n, enc); c2 = encoder_window_counts(X2n, enc) if final else None
        rr_thr = None
        if model_name == "elm_rr":       # the 30 RR channels of RR-in-liquid, counted per window, next to the encoder's
            rr_thr = rr_thresholds(D1["rr"][tr_rows])
            T_in = encode(X1n[:1], enc).shape[1]    # encoder output length, as the liquid sees it (after the counts above)
            c1 = np.concatenate([c1, rr_window_counts(D1["rr"], rr_thr, T_in)], axis=2)
            c2 = np.concatenate([c2, rr_window_counts(D2["rr"], rr_thr, T_in)], axis=2) if final else None
        if model_name == "no_liquid":
            rates = lambda c: (c.reshape(len(c), -1).astype(np.float64) / WIN).astype(np.float32)
            F1 = lambda r: rates(c1[r]); F2 = (lambda r: rates(c2[r])) if final else None
        else:
            # elm_rr: same A_in, mu_in, seed and input size as the fixed RR-in-liquid liquid, so the same input weights
            ain = measure_ain(X1n, tr_rows, enc, rr=D1["rr"] if rr_thr is not None else None, rr_thr=rr_thr)
            mu_in = I_TARGET / (ain * LIQ["P_in"])
            Wf = build_liquid(c1.shape[2], mu_in, s0["seed"], True).input_rsv_weights.detach().cpu().numpy().astype(np.float32)
            proj = lambda c: np.einsum("nwc,ck->nwk", c.astype(np.float32), Wf).reshape(len(c), -1)
            tot = np.zeros(NWIN * N_NEUR, np.float64)
            for i in range(0, len(tr_rows), 4096):
                tot += proj(c1[tr_rows[i:i + 4096]]).sum(0)
            mean = (tot / len(tr_rows)).astype(np.float32)
            F1 = lambda r: np.maximum(proj(c1[r]) - mean, 0)
            F2 = (lambda r: np.maximum(proj(c2[r]) - mean, 0)) if final else None
            info.update({"A_in_measured": ain, "mu_in": mu_in, "elm": "window input counts @ fixed-liquid W_in, centered on train mean, ReLU"})
    elif model_name == "raw_beat":
        F1 = lambda r: X1n[r]; F2 = (lambda r: X2n[r]) if final else None
    elif model_name == "rr_only":
        F1 = lambda r: D1["rr"][r]; F2 = (lambda r: D2["rr"][r]) if final else None
    elif model_name == "rr_only_mrr":
        F1 = lambda r: rr_inputs(D1, r, True); F2 = (lambda r: rr_inputs(D2, r, True)) if final else None
    elif model_name == "rr_shape_mrr":
        F1 = lambda r: np.hstack([shape_features(X1n[r]), rr_inputs(D1, r, True)])
        F2 = (lambda r: np.hstack([shape_features(X2n[r]), rr_inputs(D2, r, True)])) if final else None
    else:  # rr_shape and lda
        F1 = lambda r: np.hstack([shape_features(X1n[r]), D1["rr"][r]])
        F2 = (lambda r: np.hstack([shape_features(X2n[r]), D2["rr"][r]])) if final else None
    shared = time.time() - t0
    if model_name == "lda":
        for e in runs:
            t1 = time.time()
            tr, ev, _ = _split_rows(e["spec"]["split"], len(D1["y"]))
            ev = np.arange(len(D2["y"])) if final else ev
            Fev, Dev = (F2, D2) if final else (F1, D1)
            sc = StandardScaler().fit(F1(tr))
            lda = LinearDiscriminantAnalysis(solver="lsqr", shrinkage="auto").fit(sc.transform(F1(tr)), D1["y"][tr])
            yp = lda.predict(sc.transform(Fev(ev))); scores = lda.decision_function(sc.transform(Fev(ev)))
            m = compute_metrics(Dev["y"][ev], yp)
            _finish_run(e, m, ev, Dev["pid"][ev], Dev["y"][ev], yp, scores, {"scaler": sc, "classifier": lda}, None, info,
                        {"shared_seconds": shared, "total_seconds": shared + time.time() - t1})
        return
    _run_readouts(runs, F1, F2 if final else F1, D1["y"],
                  D2 if final else D1, final, info, {"shared_seconds": shared}, len(D1["y"]))

def execute_nn(job, runs):
    for e in runs:
        s = e["spec"]; t0 = time.time()
        D1, D2, X1n, X2n, tr_rows, normp = _front_end(s["split"])
        tr, ev, _ = _split_rows(s["split"], len(D1["y"]))
        final = s["split"] == "final"
        Xev, Dev = (X2n, D2) if final else (X1n, D1)
        ev = np.arange(len(D2["y"])) if final else ev
        yp, logits, curve, state = train_nn(s["model"], X1n[tr], D1["y"][tr], Xev[ev],
                                            rr_tr=rr_inputs(D1, tr, s.get("rr_set") == "mean") if s["use_rr"] else None,
                                            rr_ev=rr_inputs(Dev, ev, s.get("rr_set") == "mean") if s["use_rr"] else None, seed=s["seed"],
                                            yev_for_curve=None if final else Dev["y"][ev],
                                            arch=s.get("arch"), epochs=s.get("epochs"))
        m = compute_metrics(Dev["y"][ev], yp)
        if not final:            # held-out Macro-F1 after every epoch: the tuning search picks its epoch count from these
            m["eval_macro_f1_by_epoch"] = {str(r["epoch"]): r["eval_macro_f1"] for r in curve}
        m["n_params"] = state["n_params"]
        _finish_run(e, m, ev, Dev["pid"][ev], Dev["y"][ev], yp, logits, state, curve, {"normalization": normp},
                    {"total_seconds": time.time() - t0, "epochs": s.get("epochs") or NN_EPOCHS})

def execute_check(job, runs):
    e = runs[0]; t0 = time.time()
    g = {}
    for mu in LIQ["mu_rsv"]:
        s_code = _TorchGamma(1.0, mu).sample((200000,)).mean().item()
        s_fix = _TorchGamma(1.0, 1.0 / mu).sample((200000,)).mean().item()
        g[f"intended_mean_{mu}"] = {"authors_code_Gamma(k,theta)": s_code, "fixed_Gamma(k,1/theta)": s_fix, "1/mu": 1 / mu}
    CHECKS["gamma_parameterisation"] = g
    rows = []
    for filt in ("morph", "median"):
        for norm in ("standard", "robust"):
            X1n, _, _ = normalize_final(DATA[filt]["DS1"]["X"], DATA[filt]["DS2"]["X"], norm)
            for enc in ENCS:
                a = measure_ain(X1n, np.arange(len(X1n)), enc)
                rows.append({"filter": filt, "norm": norm, "encoder": enc, "A_in_measured": a, "A_in_paper": PAPER_AIN[enc],
                             "measured_over_paper": a / PAPER_AIN[enc], "mu_in_published": I_TARGET / (PAPER_AIN[enc] * LIQ["P_in"]),
                             "mu_in_fixed": I_TARGET / (a * LIQ["P_in"])})
    ain_df = pd.DataFrame(rows)
    for d in ("tables", "checks"):       # checks/ is carried to later sessions and merges; tables/ is rebuilt
        ain_df.to_csv(os.path.join(WORK_DIR, d, "ain_measured.csv"), index=False)
    struct = []
    a11 = ain_df.query("filter=='morph' and norm=='robust' and encoder=='multi'").A_in_measured.iloc[0]
    for mode, mcfg in MODES.items():
        a = PAPER_AIN["multi"] if mcfg["ain"] == "paper" else a11
        mdl = build_liquid(200, I_TARGET / (a * LIQ["P_in"]), SEEDS[0], mcfg["gamma_fix"])
        struct.append({"mode": mode, "seed": SEEDS[0], "A_in": a, **structure_stats(mdl)})
    for d in ("tables", "checks"):
        pd.DataFrame(struct).to_csv(os.path.join(WORK_DIR, d, "liquid_structure.csv"), index=False)
    CHECKS["A_in_table"] = rows; CHECKS["liquid_structure"] = struct
    jdump({"checks": CHECKS, "env": ENV, "install": INSTALL_REPORT}, os.path.join(WORK_DIR, "checks", "checks.json"))
    print(ain_df.round(4).to_string(index=False))
    print(pd.DataFrame(struct)[["mode", "n_recurrent_synapses", "EE_mean_abs_weight", "IE_mean_abs_weight", "mean_nonzero_input_weight"]].to_string(index=False))
    CACHE.mark_done(e["rid"], time.time() - t0, None, e["group"])

def _torch_raster(model, S, dtype, use_lif):
    """Full spike raster (B, 650, 343). use_lif=True: the authors' snn.Leaky; False: the update rule written out."""
    Win = model.input_rsv_weights.to(dtype).cpu(); Wr = model.rsv_weights.to(dtype).cpu()
    beta = torch.where(model.is_excit.cpu(), LIQ["betaE"], LIQ["betaI"]).to(dtype)
    x = torch.from_numpy(S).to(dtype)
    B, Tin, C = x.shape
    spk = torch.zeros(B, N_NEUR, dtype=dtype); mem = torch.zeros(B, N_NEUR, dtype=dtype)
    lif = snntorch.Leaky(beta=beta, threshold=1.0, reset_mechanism="subtract", reset_delay=False) if use_lif else None
    out = torch.zeros(B, LIQ["num_sim_steps"], N_NEUR, dtype=torch.bool)
    for t in range(LIQ["num_sim_steps"]):
        I = ((x[:, t] if t < Tin else torch.zeros(B, C, dtype=dtype)) @ Win + spk @ Wr) * LIQ["dt"]
        if use_lif:
            spk, mem = lif(I, mem)
        else:
            rst = (mem > 1.0).to(dtype)
            u = beta.clamp(0, 1) * mem + I - rst * 1.0
            spk = (u > 1.0).to(dtype)
            mem = u - (spk - rst) * 1.0
        out[:, t] = spk > 0.5
    return out.numpy()

def _brian2_raster(model, S1):
    import brian2 as b2
    b2.prefs.codegen.target = "numpy"
    b2.BrianLogger.log_level_error()
    b2.start_scope()
    b2.defaultclock.dt = 1 * b2.ms
    Win = model.input_rsv_weights.double().cpu().numpy(); W = model.rsv_weights.double().cpu().numpy()
    G = b2.NeuronGroup(N_NEUR, "mem : 1\nspk : 1\nIacc : 1\nbeta : 1 (constant)", threshold="spk > 0.5",
                       reset="", namespace={})
    G.beta = np.where(model.is_excit.cpu().numpy(), LIQ["betaE"], LIQ["betaI"])
    # Brian2 step k = model step k-1: input spikes of model step t are sent at t ms and summed into Iacc,
    # which the update at step t+1 consumes; recurrent spikes arrive one model step later, as in the authors' loop.
    G.run_regularly("rst = int(mem > 1.0)\n"
                    "u = beta*mem + Iacc*1.0 - rst*1.0\n"
                    "spk = int(u > 1.0)\n"
                    "mem = u - (spk - rst)*1.0\n"
                    "Iacc = 0", when="start")
    tt, cc = np.nonzero(S1)
    inp = b2.SpikeGeneratorGroup(S1.shape[1], cc, tt * b2.ms)
    pi, pj = np.nonzero(Win)
    Sin = b2.Synapses(inp, G, "w : 1", on_pre="Iacc_post += w"); Sin.connect(i=pi, j=pj); Sin.w = Win[pi, pj]
    objs = [G, inp, Sin]
    ri, rj = np.nonzero(W)
    if len(ri):
        Srec = b2.Synapses(G, G, "w : 1", on_pre="Iacc_post += w"); Srec.connect(i=ri, j=rj); Srec.w = W[ri, rj]
        objs.append(Srec)
    mon = b2.SpikeMonitor(G); objs.append(mon)
    net = b2.Network(*objs)
    net.run((LIQ["num_sim_steps"] + 1) * b2.ms)
    r = np.zeros((LIQ["num_sim_steps"], N_NEUR), bool)
    k = np.round(np.asarray(mon.t / b2.ms)).astype(int) - 1
    ok = (k >= 0) & (k < LIQ["num_sim_steps"])
    r[k[ok], np.asarray(mon.i)[ok]] = True
    return r

def execute_brian2(job, runs):
    e = runs[0]; t0 = time.time()
    D1, D2 = DATA["morph"]["DS1"], DATA["morph"]["DS2"]
    X1n, X2n, _ = normalize_final(D1["X"], D2["X"], "robust")
    ain = measure_ain(X1n, np.arange(len(X1n)), "multi")
    res = {"A_in": ain, "liquids": {}}
    for mode in ("fixed", "published"):
        a = ain if MODES[mode]["ain"] == "measured" else PAPER_AIN["multi"]
        model = build_liquid(200, I_TARGET / (a * LIQ["P_in"]), SEEDS[0], MODES[mode]["gamma_fix"])
        pick = np.concatenate([np.where(D2["y"] == c)[0][:k] for c, k in
                               ((0, BRIAN2_BEATS - 2 * (BRIAN2_BEATS // 3)), (1, BRIAN2_BEATS // 3), (2, BRIAN2_BEATS // 3))])
        S = encode(X2n[pick], "multi")
        lif32 = _torch_raster(model, S, torch.float32, True)
        eq32 = _torch_raster(model, S, torch.float32, False)
        eq64 = _torch_raster(model, S, torch.float64, False)
        rows = []
        for b in range(len(pick)):
            br = _brian2_raster(model, S[b])
            rows.append({"beat_index_DS2": int(pick[b]), "class": CLASS_NAMES[D2["y"][pick[b]]],
                         "spikes_brian2": int(br.sum()), "spikes_torch64_equations": int(eq64[b].sum()),
                         "spikes_authors_lif_float32": int(lif32[b].sum()),
                         "brian2_vs_torch64_mismatches": int((br != eq64[b]).sum()),
                         "brian2_vs_authors_float32_mismatches": int((br != lif32[b]).sum()),
                         "equations_vs_snntorch_float32_mismatches": int((eq32[b] != lif32[b]).sum())})
        df = pd.DataFrame(rows)
        res["liquids"][mode] = {"per_beat": rows, "total_cells": int(len(pick) * LIQ["num_sim_steps"] * N_NEUR),
                                "brian2_vs_torch64_mismatches": int(df.brian2_vs_torch64_mismatches.sum()),
                                "brian2_vs_authors_float32_mismatches": int(df.brian2_vs_authors_float32_mismatches.sum()),
                                "equations_vs_snntorch_float32_mismatches": int(df.equations_vs_snntorch_float32_mismatches.sum())}
        print(mode); print(df.to_string(index=False))
    jdump(res, os.path.join(WORK_DIR, "checks", "brian2_check.json"))
    CACHE.mark_done(e["rid"], time.time() - t0, None, e["group"])

def execute_huang(job, runs):
    """Huang et al. 2014 on one split: the 15 SVM fits run as one task in the fit pool (one core)."""
    e = runs[0]; s = e["spec"]; t0 = time.time()
    D1, D2 = DATA["morph"]["DS1"], DATA["morph"]["DS2"]
    final = s["split"] == "final"
    tr, ev, _ = _split_rows(s["split"], len(D1["y"]))
    Dev = D2 if final else D1
    if final:
        ev = np.arange(len(D2["y"]))
    def fit(cancel=None, shared=None):
        return huang_fit_predict(s, tr, ev, D1, Dev, cancel)
    def record(res, dt):
        y_pred, vote, votes, models = res
        yev = Dev["y"][ev]
        m = compute_metrics(yev, y_pred)
        vm = compute_metrics(yev, vote)          # the SVM ensemble alone (before the SVEB rule)
        m.update({f"vote_{k}": vm[k] for k in ("macro_f1", "recall_N", "precision_N", "recall_VEB", "precision_VEB")})
        m["vote_ave_N_VEB"] = float(np.mean([vm["recall_N"], vm["precision_N"], vm["recall_VEB"], vm["precision_VEB"]]))  # Huang's Ave
        m["n_support_vectors"] = int(sum(c.n_support_.sum() for c in models))
        update_speed("huang_s_per_run", dt, robust=True)
        _finish_run(e, m, ev, Dev["pid"][ev], yev, y_pred, (votes / len(models)).astype(np.float32),
                    {"svms": models, "matrix_seed": s["seed"], "pinv": True}, None,
                    {"feature": "200 raw samples z-scored @ pinv(A) (50) + pre-RR (s)", "sveb_rule": "pre-RR / record mean RR"},
                    {"total_seconds": time.time() - t0 + dt, "fit_seconds": dt})
    if FIT_POOL is not None:
        FIT_POOL.submit_job([{"rid": e["rid"], "gcells": None, "est_s": SPEED["huang_s_per_run"], "ram_gb": 1.5,
                              "fit": fit, "record": lambda res, dt, busy: record(res, dt)}],
                            _JOB_CTX.get("ram_gb", 0.0), SharedPrep())
        _JOB_CTX["submitted"] = True
        return
    t1 = time.time(); res = fit(); record(res, time.time() - t1)

def execute_hybrid(job, runs):
    """Huang's SVEB rule on a finished run's saved per-beat predictions (no training)."""
    for e in runs:
        s = e["spec"]; t0 = time.time()
        z = np.load(os.path.join(WORK_DIR, "runs", s["base"], "predictions.npz"))
        D = DATA["morph"]["DS2" if s["split"] == "final" else "DS1"]
        ev = z["eval_index"].astype(int)
        if not np.array_equal(z["y_true"].astype(int), D["y"][ev]):
            raise ValueError(f"saved predictions of {s['base']} do not match this session's beat order")
        y_pred = hybrid_rule(z["y_pred"].astype(int), D["rr_mean_ratio"][ev], s["rule"], s["threshold"])
        _finish_run(e, compute_metrics(D["y"][ev], y_pred), ev, D["pid"][ev], D["y"][ev], y_pred, None, None, None,
                    {"base_run": s["base"], "base_label": s["base_label"], "rule": s["rule"], "threshold": s["threshold"]},
                    {"total_seconds": time.time() - t0})

def working_settings(spec):
    """Candidates (lambda = 2, each g_rec) plus three references that are never chosen."""
    rows = [{"label": f"lambda {spec['lambda_val']:g}, g_rec {g:g}", "lambda_val": spec["lambda_val"], "g_rec": float(g),
             "mode": "fixed", "recurrence": True, "candidate": True} for g in spec["gains"]]
    rows += [{"label": f"lambda {spec['lambda_val']:g}, recurrence off", "lambda_val": spec["lambda_val"], "g_rec": 1.0,
              "mode": "fixed", "recurrence": False, "candidate": False},
             {"label": "paper as described (lambda 0.8, g_rec 1)", "lambda_val": 0.8, "g_rec": 1.0, "mode": "fixed",
              "recurrence": True, "candidate": False},
             {"label": "paper as published (both bugs)", "lambda_val": 0.8, "g_rec": 1.0, "mode": "published",
              "recurrence": True, "candidate": False}]
    return rows

def choose_g_rec(rows, max_firing):
    """Edge of chaos: among non-saturated candidates, the one whose perturbation neither dies nor grows
    (damage ratio closest to 1 on a log scale); if none spreads at all, the strongest non-saturated one."""
    ok = [r for r in rows if r["candidate"] and r["firing_per_step"] <= max_firing]
    if not ok:
        return min(r["g_rec"] for r in rows if r["candidate"])
    spread = [r for r in ok if r["damage_ratio"] > 0]
    if spread:
        return min(spread, key=lambda r: abs(np.log(r["damage_ratio"])))["g_rec"]
    return max(r["g_rec"] for r in ok)

def execute_wdiag(job, runs):
    """Label-free diagnostic: for each setting, flip one input spike at step t_perturb and count how many neurons'
    spikes differ afterwards (mean over beats). damage ratio = mean difference 41-80 steps later / 1-10 steps later:
    about 0 = ordered (the liquid forgets at once), about 1 = edge of chaos, >> 1 = chaotic."""
    e = runs[0]; s = e["spec"]; t0 = time.time()
    D1 = DATA[s["filter"]]["DS1"]
    X1n, _, _ = normalize_final(D1["X"], D1["X"][:1], s["norm"])
    ain = measure_ain(X1n, np.arange(len(X1n)), s["encoder"])
    rows_b = np.sort(np.random.default_rng(s["seed"]).choice(len(X1n), min(s["n_beats"], len(X1n)), replace=False))
    S = encode(X1n[rows_b], s["encoder"]); tp = s["t_perturb"]
    out = []
    for st in working_settings(s):
        a = ain if MODES[st["mode"]]["ain"] == "measured" else PAPER_AIN[s["encoder"]]
        model = build_liquid(S.shape[2], I_TARGET / (a * LIQ["P_in"]), s["seed"], MODES[st["mode"]]["gamma_fix"],
                             st["lambda_val"], st["recurrence"], st["g_rec"])
        ch = int((model.input_rsv_weights != 0).sum(1).argmax())     # the input channel with the most synapses
        S2 = S.copy(); S2[:, tp, ch] = 1 - S2[:, tp, ch]
        R1, irec, iin = liquid_raster(model, S)
        R2, _, _ = liquid_raster(model, S2)
        d = (R1 != R2).sum(2).mean(0)
        early, late = float(d[tp + 1:tp + 11].mean()), float(d[tp + 41:tp + 81].mean())
        ss = structure_stats(model)
        out.append({**st, "firing_per_step": float(R1[:, :NWIN * WIN].mean()),
                    "last_window_firing_per_step": float(R1[:, (NWIN - 1) * WIN:NWIN * WIN].mean()),
                    "mean_abs_recurrent_current": irec, "mean_input_current": iin,
                    "recurrent_over_input_current": irec / max(iin, 1e-12),
                    "perturbation_early": early, "perturbation_late": late,
                    "damage_ratio": late / early if early > 0 else 0.0,
                    "n_recurrent_synapses": ss["n_recurrent_synapses"], "mean_recurrent_in_degree": ss["mean_recurrent_in_degree"],
                    "perturbation_curve_from_t_perturb": [round(float(v), 4) for v in d[tp:tp + 121]]})
        log(f"  working-liquid diagnostic: {st['label']}: firing {out[-1]['firing_per_step']:.3f}/step, "
            f"recurrent/input current {out[-1]['recurrent_over_input_current']:.3f}, damage ratio {out[-1]['damage_ratio']:.3f}")
    g = choose_g_rec(out, s["max_firing"])
    res = {"run_id": e["rid"], "session": SESSION_TAG, "A_in_measured": ain, "beats_DS1": rows_b.tolist(), "t_perturb": tp,
           "perturbation": "one input spike flipped (channel with the most input synapses)", "settings": out,
           "max_firing": s["max_firing"], "chosen_g_rec": g,
           "rule": "non-saturated candidate with damage ratio closest to 1 (log scale); none spreading -> strongest non-saturated"}
    jdump(res, os.path.join(WORK_DIR, "checks", "working_liquid_diag.json"))
    log(f"  working-liquid diagnostic: chosen g_rec = {g:g} (lambda {s['lambda_val']:g})")
    CACHE.mark_done(e["rid"], time.time() - t0, None, e["group"])

EXECUTORS = {"lsm": execute_lsm, "feat": execute_feat, "nn": execute_nn, "check": execute_check, "brian2": execute_brian2,
             "huang": execute_huang, "hybrid": execute_hybrid, "wdiag": execute_wdiag}

FEAT_DIM = {"no_liquid": lambda enc: NWIN * (200 if enc == "multi" else N_IN[enc]), "elm": lambda enc: NWIN * N_NEUR,
            "raw_beat": lambda enc: 577, "rr_only": lambda enc: len(RR_NAMES), "rr_shape": lambda enc: 73 + len(RR_NAMES),
            "lda": lambda enc: 0, "elm_rr": lambda enc: NWIN * N_NEUR, "rr_only_mrr": lambda enc: len(RR_NAMES) + 1,
            "rr_shape_mrr": lambda enc: 73 + len(RR_NAMES) + 1}

def _sizes(spec):
    n1 = len(DATA["morph"]["DS1"]["y"]); n2 = len(DATA["morph"]["DS2"]["y"])
    final = spec.get("split") == "final"
    return n1, n2, final, (n1 if final else int(0.75 * n1)), n1 + (n2 if final else 0)

def _nfeat(spec):
    if spec["kind"] == "lsm":
        if spec.get("readout") == "svm_late":
            return N_NEUR
        return NWIN * N_NEUR + (len(RR_NAMES) if spec.get("readout") == "svm_rr" else 0)
    return FEAT_DIM[spec["model"]](spec["encoder"])

def estimate_parts(job, runs):
    """(seconds of main-thread / GPU work, [seconds of each classifier fit], fit speed key) for one job."""
    s = runs[0]["spec"]; k = s["kind"]
    n1, n2, final, ntr, beats = _sizes(s)
    if k == "lsm":
        main = beats / SPEED["sim_beats_per_s"][s["encoder"]] + SPEED["lsm_job_overhead_s"]
    elif k == "feat":
        main = (beats / SPEED["encode_beats_per_s"][s["encoder"]] if s["encoder"] else 0) + 10
        if s["model"] == "lda":
            return main + 5 * len(runs), []
    elif k == "nn":
        t = 0.0
        for r in runs:
            sp = r["spec"]
            if sp.get("arch"):
                t += sp["epochs"] * ntr / 1000 * SPEED["nn_unit_s_per_epoch_kbeat"][sp["model"]] * nn_cost(sp["model"], sp["arch"]) ** 0.5
            else:
                t += NN_EPOCHS * ntr / 1000 * SPEED["nn_s_per_epoch_kbeat"][sp["model"]]
        return t + 20 * len(runs), []
    elif k == "check":
        return 120, []
    elif k == "huang":
        if PIPELINE:          # one pool task per run, expressed in the pool's units so the guards and projection see it
            return 15.0 * len(runs), [SPEED["huang_s_per_run"] / SPEED["fit_s_per_Gcell"]["pipe"]] * len(runs)
        return (15.0 + SPEED["huang_s_per_run"]) * len(runs), []
    elif k == "hybrid":
        return 2.0 * len(runs), []
    elif k == "wdiag":
        return 20.0 * (len(WORKING_GAINS) + 3) + 60, []
    else:
        return 30 * BRIAN2_BEATS + 30, []
    return main, [ntr * _nfeat(r["spec"]) / 1e9 for r in runs]      # fits returned in Gcells

def _seq_fit_seconds(gcells):
    if not gcells:
        return 0.0
    threads = max(1, min(int(round(SPEED["fit_threads"])), len(gcells)))
    return math.ceil(len(gcells) / threads) * max(gcells) * SPEED["fit_s_per_Gcell"][str(fit_n_jobs(threads))]

def estimate_seconds(job, runs):
    """Older runner: projected wall time of one job (main-thread work + its fits sharing the cores)."""
    main, gcells = estimate_parts(job, runs)
    return main + _seq_fit_seconds(gcells)

def fit_seconds_pipe(gcells):
    return [g * SPEED["fit_s_per_Gcell"]["pipe"] for g in gcells]

def job_ram_gb(job, runs):
    """RAM a job holds until all its fits finish (spike counts or features, plus its normalised beats)."""
    s = runs[0]["spec"]; n1, n2, final, ntr, beats = _sizes(s)
    if s["kind"] == "lsm":
        return beats * NWIN * N_NEUR / 1e9 + 0.6
    if s["kind"] == "feat":
        return beats * min(_nfeat(s), 12000) * (1 if s["model"] in ("no_liquid", "elm") else 4) / 1e9 + 0.6
    return 1.0

CORE_SEARCH_PATH = os.path.join(WORK_DIR, "checks", "core_search.json")

def _core_search_saved():
    """The core search's choice from an earlier benchmark on this machine (local only), or None."""
    if MACHINE != "local" or FIT_WORKERS != "auto" or not os.path.exists(CORE_SEARCH_PATH):
        return None
    try:
        cs = json.load(open(CORE_SEARCH_PATH))
        if cs.get("cpu_count") == N_CPU and cs.get("gpu") == ENV.get("gpu"):
            return cs
    except Exception:
        pass
    return None

def pipeline_workers():
    """Parallel fits: every core (or FIT_WORKERS, or the local core search's choice), capped by RAM so each worker can
    hold a fit and a job's counts."""
    budget = _avail_ram_gb() - RAM_RESERVE_GB
    by_ram = max(1, int(budget // (fit_ram_gb(NWIN * N_NEUR, len(DATA["morph"]["DS1"]["y"])) + 1.1)))
    saved = _core_search_saved()
    want = int(saved["best_workers"]) if saved else (N_CPU if FIT_WORKERS == "auto" else int(FIT_WORKERS))
    return max(1, min(want, by_ram)), budget, by_ram

def simulate_pipeline(items, W):
    """items: [(main_seconds, [fit seconds])] in run order. As in the runner: the main thread may start a job with fits
    only while fewer than W fits wait for a worker; while it would wait, it runs pending jobs without fits (CNN/GRU)
    instead. Returns total seconds."""
    free = [0.0] * W; heapq.heapify(free)
    starts, tp, end = [], 0.0, 0.0
    fillers = collections.deque(m for m, f in items if not f)
    for main, fits in items:
        if not fits:
            continue                                   # jobs without fits are taken from `fillers`
        while True:
            waiting = sorted(x for x in starts if x > tp)
            if len(waiting) < W:
                break
            if fillers:
                tp += fillers.popleft()                # GPU busy with a CNN/GRU job meanwhile
            else:
                tp = waiting[len(waiting) - W]
        starts = [x for x in starts if x > tp]
        tp += main
        for d in fits:
            w = heapq.heappop(free); st = max(w, tp); heapq.heappush(free, st + d)
            starts.append(st); end = max(end, st + d)
    tp += sum(fillers)                                 # whatever is left after the last fits were handed out
    return max(tp, end)

def project_time(jobs):
    """Remaining time for every pending job (plus the second stages of the re-tunings if not yet planned), by group."""
    rows, items = [], []
    def add(group, nruns, main, gcells):
        fits = fit_seconds_pipe(gcells) if PIPELINE else []
        secs = main + (sum(fits) / PIPE_W if PIPELINE else _seq_fit_seconds(gcells))
        rows.append({"group": group, "runs": nruns, "jobs": 1, "seconds": secs})
        items.append((main, fits))
    for j in jobs:
        pr = [r for r in j["runs"] if not CACHE.is_done(r["rid"])]
        if pr:
            main, gcells = estimate_parts(j, pr)
            add(pr[0]["group"], len(pr), main, gcells)
    if group_on("retune") and retune_choice()[0] is None:      # stage 2 appears once stage 1 is done
        for sp in ["final"] + SPLITS_CV:
            spec = lsm_spec("fixed", SEEDS[0], "morph", "robust", "multi", "svm", sp)
            fake = [{"spec": spec, "rid": "", "group": "retune"}]
            for _ in SEEDS:
                main, gcells = estimate_parts({"key": None}, fake)
                add("retune", 1, main, gcells)
    if group_on("nn_tune"):
        for m in ("cnn", "gru"):
            for use_rr in (False, True):
                if nn_tune_choice(m, use_rr)[0] is None:
                    a = DEFAULT_ARCH[m]
                    for seed in SEEDS:
                        for sp in ["final"] + SPLITS_CV:
                            fake = [{"spec": nn_spec(m, use_rr, seed, sp, arch=a, epochs=NN_TUNE_CHECKPOINTS[len(NN_TUNE_CHECKPOINTS) // 2], trial="best"),
                                     "rid": "", "group": "nn_tune"}]
                            add("nn_tune", 1, *estimate_parts({"key": None}, fake))
    df = pd.DataFrame(rows, columns=["group", "runs", "jobs", "seconds"])
    if df.empty:
        return df
    df = df.groupby("group", sort=False).sum().reindex([g for g in GROUP_ORDER if g in set(df.group)]).reset_index()
    if PIPELINE:            # the schedule overlaps GPU and CPU work: scale the group shares to the simulated total
        total = simulate_pipeline(items, PIPE_W)
        df["seconds"] = df.seconds * total / max(df.seconds.sum(), 1e-9)
    df["hours"] = df.seconds / 3600
    return df

def print_projection(jobs, title):
    df = project_time(jobs)
    if df.empty:
        print(title + ": nothing left to run."); return df
    total = df.hours.sum()
    per_session = SESSION_BUDGET_HOURS - ANALYSIS_RESERVE_MIN / 60 - 0.2       # minus setup and analysis
    keys = {"sim_beats_per_s", "encode_beats_per_s", "nn_s_per_epoch_kbeat", "nn_unit_s_per_epoch_kbeat"}
    fit_key = "fit_s_per_Gcell:pipe" if PIPELINE else None
    guessed = sorted({f"{k}:{e}" for k in keys for e in SPEED[k]} - SPEED_MEASURED)
    if PIPELINE and fit_key not in SPEED_MEASURED:
        guessed.append(fit_key)
    if not PIPELINE:
        guessed += sorted({f"fit_s_per_Gcell:{e}" for e in ("1", "2", "3")} | {"fit_threads"} - SPEED_MEASURED)
    runner = (f"pipeline: {PIPE_W} parallel fits (CPU cores {N_CPU}, RAM allows {PIPE_BY_RAM}), "
              f"{SPEED['fit_s_per_Gcell']['pipe']:.0f} s per Gcell per fit under load" if PIPELINE else "older one-job-at-a-time runner")
    sess = (f"about {math.ceil(total / per_session)} session(s) of {SESSION_BUDGET_HOURS} h" if MACHINE == "kaggle"
            else f"one continuous run (safety stop at {SESSION_BUDGET_HOURS} h)")
    lines = [f"\n===== {title} =====", df.assign(hours=df.hours.round(2)).drop(columns="seconds").to_string(index=False),
             f"TOTAL remaining: {total:.1f} h  ->  {sess}" + (f", about {total / 30:.1f} week(s) of a ~30 h/week GPU quota" if MACHINE == "kaggle" else ""),
             f"runner: {runner}",
             f"machine: {MACHINE}; device: {ENV.get('gpu', DEVICE.type)}; CPU: {ENV.get('cpu_model')}",
             f"measured speeds: {sorted(SPEED_MEASURED)}",
             ("still default guesses (less reliable): " + ", ".join(guessed)) if guessed else "all speeds measured"]
    print("\n".join(lines))
    df.to_csv(os.path.join(WORK_DIR, "tables", "time_projection.csv"), index=False)
    atomic_write_text(os.path.join(WORK_DIR, "tables", "time_projection.txt"), "\n".join(lines) + "\n")
    return df

PIPE_W, PIPE_BUDGET_GB, PIPE_BY_RAM = pipeline_workers()
if PIPELINE:
    print(f"pipeline: {PIPE_W} parallel fits (CPU cores: {N_CPU}, FIT_WORKERS={FIT_WORKERS!r}, RAM allows {PIPE_BY_RAM} "
          f"with {PIPE_BUDGET_GB:.1f} GB available after the {RAM_RESERVE_GB} GB reserve)"
          + (f"; {_core_search_saved()['best_workers']} chosen by the core search of session {_core_search_saved().get('session')}"
             if _core_search_saved() else ""))


pipeline: 38 parallel fits (CPU cores: 38, FIT_WORKERS='auto', RAM allows 38 with 103.4 GB available after the 4.0 GB reserve)


## 10. Run everything that is pending

In [11]:
# The runner: executes every pending job in priority order until done, out of time, or out of disk.
# PIPELINE=True: this thread does the GPU-side work one job at a time (simulation, NN training, checks); each finished
# job's classifier fits go to FIT_POOL (one fit per core). While the pool already has a full queue, the GPU is given
# a pending neural-network job instead of waiting. Results do not depend on the order: every run has its own seeds.
def _pending(job, skip):
    return [r for r in job["runs"] if not CACHE.is_done(r["rid"]) and r["rid"] not in skip
            and (RETRY_FAILED or r["rid"] not in CACHE.failed)]

def refresh_plan():
    plan = build_plan()
    with LOCK:
        CACHE.pending = [(e["rid"], e["group"]) for e in plan if not CACHE.is_done(e["rid"])]
        CACHE.write()
    return plan, make_jobs(plan)

def benchmark_selection(jobs):
    """Representative full-size jobs: one final-test and one single-fold CV liquid job per encoder, one CNN and one GRU
    job (untuned and one tuning trial each), the cheap checks; with the pipeline, enough further jobs in plan order to
    keep every fit worker busy for a while (so the speed under full load is measured). All are ordinary runs."""
    want = [("lsm", True, e) for e in ENCS] + [("lsm", False, e) for e in ENCS] + [("nn", True, "cnn"), ("nn", True, "gru")]
    keys = {j["key"] for j in jobs if j["key"][0] in ("check", "brian2")}
    for kind, final, x in want:
        for j in jobs:
            s = j["runs"][0]["spec"]
            if s["kind"] != kind or not _pending(j, set()) or (s["split"] == "final") != final or s.get("arch"):
                continue
            if kind == "lsm" and (s["encoder"] != x or (not final and j["key"][2] == "cv-shared")):
                continue
            if kind == "nn" and s["model"] != x:
                continue
            keys.add(j["key"]); break
    for final in (True, False):        # one Huang job of each size: measures the SVM ensemble's speed
        j = next((j for j in jobs if j["runs"][0]["spec"]["kind"] == "huang" and (j["runs"][0]["spec"]["split"] == "final") == final
                  and _pending(j, set())), None)
        if j is not None:
            keys.add(j["key"])
    for m in ("cnn", "gru"):          # one tuning trial each: measures the speed of tuned architectures
        j = next((j for j in jobs if j["runs"][0]["spec"].get("arch") and j["runs"][0]["spec"]["model"] == m
                  and j["runs"][0]["spec"].get("trial") not in (None, 0, "best") and _pending(j, set())), None)
        if j is not None:
            keys.add(j["key"])
    if PIPELINE:
        n_fits = sum(len(estimate_parts(j, _pending(j, set()))[1]) for j in jobs if j["key"] in keys and _pending(j, set()))
        for j in jobs:
            if n_fits >= 3 * PIPE_W:
                break
            if j["key"] in keys or j["key"][0] not in ("lsm", "feat") or not _pending(j, set()):
                continue
            keys.add(j["key"]); n_fits += len(estimate_parts(j, _pending(j, set()))[1])
    else:
        single = next((j for j in jobs if j["key"][0] == "lsm" and len(_pending(j, set())) == 1 and j["key"][2] != "final"), None)
        if single is not None:
            keys.add(single["key"])        # a one-fit job: measures the fit speed when it gets 3 cores
    return keys

RAM_LOG = os.path.join(WORK_DIR, "logs", f"resources_{SESSION_TAG}.csv")
def log_resources(event):
    rss, peak = _proc_ram_gb()
    row = {"elapsed_h": round(elapsed_h(), 4), "event": event, "rss_gb": round(rss, 2), "peak_rss_gb": round(peak, 2),
           "available_gb": round(_avail_ram_gb(), 2), "reserved_gb": round(RAM.used, 2) if PIPELINE else None,
           "fits_waiting": FIT_POOL.queued if PIPELINE else None, "fits_running": FIT_POOL.running if PIPELINE else None,
           "jobs_holding_ram": len(FIT_POOL.jobs) if PIPELINE else None,
           "gpu_mem_gb": round(torch.cuda.max_memory_allocated() / 1e9, 2) if DEVICE.type == "cuda" else None}
    new = not os.path.exists(RAM_LOG)
    with LOCK, open(RAM_LOG, "a") as f:
        if new:
            f.write(",".join(row) + "\n")
        f.write(",".join("" if v is None else str(v) for v in row.values()) + "\n")
    return row

def _guard(job, runs):
    """None if the job may start, else the reason to stop."""
    main, gcells = estimate_parts(job, runs)
    if PIPELINE:
        fits = fit_seconds_pipe(gcells)
        need = main + (FIT_POOL.backlog_s + sum(fits)) / FIT_POOL.n + (max(fits) if fits else 0)
    else:
        need = estimate_seconds(job, runs)
    if need > seconds_left():
        return f"time budget: next job needs ~{need/60:.1f} min (incl. queued fits), {max(seconds_left(), 0)/60:.1f} min left"
    if work_dir_gb() > DISK_LIMIT_GB:
        return f"disk: {WORK_DIR} is {work_dir_gb(force=True):.1f} GB (limit {DISK_LIMIT_GB})"
    return None

def run_job(job, runs, n):
    s0 = runs[0]["spec"]
    main, gcells = estimate_parts(job, runs)
    log(f"job {n}: {runs[0]['group']} | {s0['kind']} | {len(runs)} run(s) | e.g. {runs[0]['rid'][:80]} | "
        f"main {main/60:.1f} min, {len(gcells)} fit(s)")
    _JOB_CTX.clear()
    _JOB_CTX.update(ram_gb=job_ram_gb(job, runs) if PIPELINE else 0.0, submitted=False)
    if PIPELINE:
        RAM.reserve(_JOB_CTX["ram_gb"], headroom=2 * fit_ram_gb(NWIN * N_NEUR, len(DATA["morph"]["DS1"]["y"])))
    try:
        EXECUTORS[s0["kind"]](job, runs)
    except Exception:
        tb = traceback.format_exc()
        for r in runs:
            if not CACHE.is_done(r["rid"]) and not _JOB_CTX.get("submitted"):
                _record_failure(r["rid"], tb)
        log("  JOB FAILED:", tb.strip().splitlines()[-1])
    finally:
        if PIPELINE and not _JOB_CTX.get("submitted"):
            RAM.release(_JOB_CTX["ram_gb"])
    for r in runs:
        tried_this_session.add(r["rid"])       # never retry within the same session
    save_speeds()
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()

def run_self_test():
    """SELF_TEST = True: check the machine and every stage of the code on a few beats, run no experiment."""
    res, t0 = {}, time.time()
    def item(name, ok, detail):
        res[name] = {"ok": bool(ok), "detail": detail}
        tag = "PASS" if ok else ("NOTE" if name.startswith("torch 2.4.0") else "FAIL")
        print(f"  [{tag}] {name:<28} {detail}")
    print("\n===== SELF-TEST (runs at the start of every session) =====")
    item("packages", True, ", ".join(f"{k} {ENV.get(k)}" for k in ("python", "torch", "numpy", "sklearn", "snntorch", "wfdb", "brian2")))
    item("torch 2.4.0 (authors')", TORCH_PINNED, torch.__version__ + ("" if TORCH_PINNED else " (fine; only matters for the exact-replication attempt)"))
    item("brian2", not str(ENV.get("brian2", "")).startswith("unavailable"), str(ENV.get("brian2")))
    if DEVICE.type == "cuda":
        x = torch.randn(4096, N_NEUR, device=DEVICE); w = torch.randn(N_NEUR, N_NEUR, device=DEVICE)
        torch.cuda.synchronize(); t = time.time()
        for _ in range(LIQ["num_sim_steps"]):
            y = x @ w
        torch.cuda.synchronize()
        item("GPU", True, f"{ENV.get('gpu')}, {ENV.get('gpu_mem_gb')} GB, CUDA {ENV.get('cuda')}; 650 liquid-sized steps in {time.time()-t:.2f} s")
        try:
            c = nn.Conv1d(1, 8, 5).to(DEVICE); c(torch.randn(16, 1, 577, device=DEVICE)).sum().backward()
            item("cuDNN", True, "convolution forward + backward ok")
        except Exception as ex:
            item("cuDNN", False, repr(ex))
    else:
        item("GPU", False, "PyTorch sees no GPU: everything would run on the CPU (very slow)")
    item("authors' code", ENV.get("authors_commit") == AUTHORS_COMMIT, f"commit {ENV.get('authors_commit')}")
    cs = CHECKS.get("checksums", {})
    item("MIT-BIH checksums", not cs.get("mismatches"), f"{cs.get('files_checked')} files checked, {len(cs.get('mismatches', []))} mismatches, "
         f"missing (unused) {len(cs.get('missing', []))}")
    item("DS1 = authors' order", CHECKS.get("ds1_matches_authors_order"), f"class counts {CHECKS.get('class_counts')}")
    item("authors' folds", CHECKS.get("folds_regenerate_exactly"), "fold indices regenerate exactly")
    try:
        D1, D2 = DATA["morph"]["DS1"], DATA["morph"]["DS2"]
        X1n, X2n, _ = normalize_final(D1["X"][:64], D2["X"][:64], "robust")
        for mode in ("published", "fixed"):
            a = PAPER_AIN["multi"] if MODES[mode]["ain"] == "paper" else measure_ain(X1n, np.arange(64), "multi")
            mdl = build_liquid(200, I_TARGET / (a * LIQ["P_in"]), SEEDS[0], MODES[mode]["gamma_fix"])
            C, st = simulate(mdl, X1n, "multi", 64, 200, y=D1["y"][:64], enc_seed=1)
            F = counts_to_rates(C)
            yy = np.arange(64) % 3
            fit_readout(lambda r: F[r], 64, yy, lambda r: F[r], 64, "hinge", 1)
            item(f"liquid + readout ({mode})", True, f"64 beats: firing {st['liquid_firing_prob_per_step']:.4f}/step, "
                 f"drive/target {st['input_current_over_target']:.2f}")
    except Exception as ex:
        item("liquid + readout", False, f"{ex.__class__.__name__}: {ex}")
    try:   # the GPU simulator behind every result vs the float64 equations that Brian2 matched (same beats as the Brian2 check)
        D1, D2 = DATA["morph"]["DS1"], DATA["morph"]["DS2"]
        X1n, X2n, _ = normalize_final(D1["X"], D2["X"], "robust")
        a_meas = measure_ain(X1n, np.arange(len(X1n)), "multi")
        pick = np.concatenate([np.where(D2["y"] == c)[0][:k] for c, k in
                               ((0, BRIAN2_BEATS - 2 * (BRIAN2_BEATS // 3)), (1, BRIAN2_BEATS // 3), (2, BRIAN2_BEATS // 3))])
        S = encode(X2n[pick], "multi")
        gv = {"session": SESSION_TAG, "device": str(DEVICE), "gpu": ENV.get("gpu"), "beats_DS2": pick.tolist(), "liquids": {}}
        for mode in ("fixed", "published"):
            a = a_meas if MODES[mode]["ain"] == "measured" else PAPER_AIN["multi"]
            mdl = build_liquid(200, I_TARGET / (a * LIQ["P_in"]), SEEDS[0], MODES[mode]["gamma_fix"])
            ref = _torch_raster(mdl, S, torch.float64, False)            # (beats, 650, 343): what Brian2 matched exactly
            ref_c = ref[:, :NWIN * WIN].reshape(len(pick), NWIN, WIN, N_NEUR).sum(2).reshape(len(pick), -1)
            sim_c, _ = simulate(mdl, X2n[pick], "multi", len(pick), 200, enc_seed=0)   # window counts, as in every run
            sim_c = sim_c.astype(np.int64)
            gv["liquids"][mode] = {"window_count_cells": int(ref_c.size), "mismatches": int((sim_c != ref_c).sum()),
                                   "spikes_reference": int(ref_c.sum()), "spikes_simulate": int(sim_c.sum())}
        jdump(gv, os.path.join(WORK_DIR, "checks", "gpu_vs_reference.json"))
        item("GPU simulator vs reference", sum(v["mismatches"] for v in gv["liquids"].values()) == 0,
             "; ".join(f"{m}: {v['mismatches']} of {v['window_count_cells']:,} window counts differ" for m, v in gv["liquids"].items())
             + f" (simulate() on {DEVICE} vs the float64 equations Brian2 matched, {len(pick)} DS2 beats)")
    except Exception as ex:
        item("GPU simulator vs reference", False, f"{ex.__class__.__name__}: {ex}")
    try:   # extensions: Huang ensemble on a few beats, the hybrid rule, the RR window counts used by ELM + RR
        D1 = DATA["morph"]["DS1"]
        rows = np.sort(np.concatenate([np.where(D1["y"] == c)[0][:k] for c, k in ((0, 120), (1, 40), (2, 40))]))
        hs = huang_spec(SEEDS[0], HUANG_GAMMAS[0], "final"); hs["n_matrices"] = 2
        yp, vote, votes, _ = huang_fit_predict(hs, rows[::2], rows[1::2], D1, D1)
        ok_rule = (hybrid_rule([0, 1, 2, 0], [0.5, 0.9, 0.5, 0.9], "replace", 0.8).tolist() == [1, 0, 2, 0]
                   and hybrid_rule([0, 1, 2, 0], [0.5, 0.9, 0.5, 0.9], "add", 0.8).tolist() == [1, 1, 2, 0])
        thr = rr_thresholds(D1["rr"]); T_in = D1["X"].shape[1]
        ch = rr_channels(D1["rr"][:8], thr, T_in)
        pad = np.zeros((8, NWIN * WIN, ch.shape[2]), np.int64); pad[:, :min(T_in, NWIN * WIN)] = ch[:, :NWIN * WIN]
        ok_rrw = np.array_equal(rr_window_counts(D1["rr"][:8], thr, T_in), pad.reshape(8, NWIN, WIN, -1).sum(2))
        rr9 = rr_inputs(D1, np.arange(len(D1["y"])), True)
        ok_rrw = ok_rrw and rr_channels(rr9[:8], rr_thresholds(rr9), T_in).shape[2] == 4 * RR_LEVELS
        item("extensions (Huang, hybrid, ELM+RR)", ok_rule and ok_rrw and len(yp) == len(rows[1::2]),
             f"Huang 2-matrix ensemble on {len(rows)} beats: Macro-F1 {compute_metrics(D1['y'][rows[1::2]], yp)['macro_f1']:.3f}; "
             f"hybrid rule {'ok' if ok_rule else 'WRONG'}; RR window counts {'ok' if ok_rrw else 'WRONG'}; "
             f"Huang beats {D1['huang_beats'].shape}, mean-RR ratio median {np.median(D1['rr_mean_ratio']):.3f}")
    except Exception as ex:
        item("extensions (Huang, hybrid, ELM+RR)", False, f"{ex.__class__.__name__}: {ex}")
    try:   # working liquid: g_rec multiplies only the recurrent weights; the full raster matches simulate()
        D1 = DATA["morph"]["DS1"]
        Xw, _, _ = normalize_final(D1["X"][:64], D1["X"][:1], "robust")
        m1 = build_liquid(200, 0.3, SEEDS[0], True, WORKING_LAMBDA, True)
        m3 = build_liquid(200, 0.3, SEEDS[0], True, WORKING_LAMBDA, True, 3.0)
        ok_g = bool(torch.allclose(m3.rsv_weights, 3.0 * m1.rsv_weights) and torch.equal(m3.input_rsv_weights, m1.input_rsv_weights))
        Rw, _, _ = liquid_raster(m3, encode(Xw[:4], "multi"))
        cw, _ = simulate(m3, Xw[:4], "multi", 4, 200, enc_seed=0)
        ok_r = np.array_equal(Rw[:, :NWIN * WIN].reshape(4, NWIN, WIN, N_NEUR).sum(2).reshape(4, -1), cw.astype(np.int64))
        item("working liquid (g_rec, raster)", ok_g and ok_r,
             f"g_rec scales recurrent weights only: {ok_g}; raster == simulate(): {ok_r}; "
             f"lambda {WORKING_LAMBDA:g}: {structure_stats(m1)['n_recurrent_synapses']} recurrent synapses")
    except Exception as ex:
        item("working liquid (g_rec, raster)", False, f"{ex.__class__.__name__}: {ex}")
    free = shutil.disk_usage(WORK_DIR).free / 1e9
    item("disk", free > 15, f"{free:.0f} GB free at {WORK_DIR} (all outputs together grow to ~12 GB)")
    item("RAM / cores", True, f"{ENV.get('ram_gb')} GB RAM, {N_CPU} cores -> {PIPE_W} parallel fits" if PIPELINE else f"{N_CPU} cores")
    item("log file", os.path.exists(LOG_PATH), f"{LOG_PATH} (tail -f {os.path.join(WORK_DIR, 'logs', 'latest.log')})")
    n_fail = sum(not v["ok"] for k, v in res.items() if k not in ("torch 2.4.0 (authors')",))
    print(f"===== SELF-TEST: {'all checks passed' if n_fail == 0 else str(n_fail) + ' check(s) FAILED'} ({time.time()-t0:.0f} s) =====\n")
    jdump({"passed": n_fail == 0, "checks": res, "env": ENV, "session": SESSION_TAG}, os.path.join(WORK_DIR, "checks", "self_test.json"))
    return n_fail

# ---------------- core search (local machine, benchmark only) ----------------
CORE_SEARCH_EPOCHS, CORE_SEARCH_ROWS = 3, 16000

def _core_counts():
    cap = max(1, min(N_CPU, PIPE_BY_RAM))
    return sorted({k for k in (1, 2, 3, 4, 6, 8, 10, 12, 14, 16, 20, 24, 28, 32, 40, 48, 56, 64) if k <= cap} | {cap})

def _score_workers(table, sim_uncontended, fit_anchor=None):
    """Projected total hours for each worker count: fit speed and simulation slowdown as measured by the search
    (scaled to the benchmark's own measurements when fit_anchor = (K, measured s/Gcell at K) is given)."""
    global PIPE_W
    keep = (PIPE_W, dict(SPEED["sim_beats_per_s"]), SPEED["fit_s_per_Gcell"]["pipe"])
    ref = {r["workers"]: r for r in table}
    out = []
    try:
        for r in table:
            PIPE_W = r["workers"]
            SPEED["sim_beats_per_s"] = {e: v * r["sim_speed_factor"] for e, v in sim_uncontended.items()}
            SPEED["fit_s_per_Gcell"]["pipe"] = (r["fit_s_per_Gcell"] if fit_anchor is None else
                                                fit_anchor[1] * r["fit_s_per_Gcell"] / ref[fit_anchor[0]]["fit_s_per_Gcell"])
            df = project_time(JOBS)
            out.append({**r, "projected_hours": float(df.hours.sum()) if len(df) else 0.0})
    finally:
        PIPE_W, SPEED["sim_beats_per_s"], SPEED["fit_s_per_Gcell"]["pipe"] = keep[0], keep[1], keep[2]
    best = min(out, key=lambda r: r["projected_hours"])
    # fewer workers if within 1 % (less RAM, same speed)
    best = min((r for r in out if r["projected_hours"] <= best["projected_hours"] * 1.01), key=lambda r: r["workers"])
    return out, best

def core_search():
    """Runs K classifier fits at once for several K while the GPU keeps simulating, as in the real pipeline, and
    measures (a) how long a fit takes at each K and (b) how much the simulation slows down. The projected total time of
    the whole plan then decides K. The worker count never changes a result, only the speed."""
    t_all = time.time()
    D1, D2 = DATA["morph"]["DS1"], DATA["morph"]["DS2"]
    X1n, _, _ = normalize_final(D1["X"], D2["X"], "robust")
    rows = np.sort(np.random.default_rng(0).choice(len(D1["y"]), min(CORE_SEARCH_ROWS, len(D1["y"])), replace=False))
    a = measure_ain(X1n, np.arange(len(X1n)), "multi")
    mdl = build_liquid(200, I_TARGET / (a * LIQ["P_in"]), SEEDS[0], True)
    C, _ = simulate(mdl, X1n[rows], "multi", SIM_BATCH_CV, 200, y=D1["y"][rows], enc_seed=7)
    F = lambda r: counts_to_rates(C[r]); y = D1["y"][rows]; n = len(rows)
    gcells = n * C.shape[1] / 1e9
    prep = prepare_train(F, n, True)
    Xsim = X1n[rows[:min(SIM_BATCH_CV, n)]]
    def sim_until(stop_fn, min_s=0.0):
        beats, t0 = 0, time.time()
        while not stop_fn() or time.time() - t0 < min_s:
            simulate(mdl, Xsim, "multi", SIM_BATCH_CV, 200, enc_seed=7); beats += len(Xsim)
        return beats / (time.time() - t0)
    sim_alone = sim_until(lambda: True, min_s=10.0)
    log(f"core search: {n} beats x {C.shape[1]} features, {CORE_SEARCH_EPOCHS} epochs per fit; simulation alone {sim_alone:.0f} beats/s; "
        f"trying {_core_counts()} parallel fits (RAM allows {PIPE_BY_RAM}, CPU cores {N_CPU})")
    table = []
    for K in _core_counts():
        durs, lock = [], threading.Lock()
        def one():
            t = time.time()
            fit_readout(F, n, y, F, 1, "hinge", 1, prepared=prep, n_jobs=1, epochs=CORE_SEARCH_EPOCHS)
            with lock:
                durs.append(time.time() - t)
        th = [threading.Thread(target=one) for _ in range(K)]
        t0 = time.time()
        for t in th:
            t.start()
        sim_bps = sim_until(lambda: not any(t.is_alive() for t in th))
        for t in th:
            t.join()
        wall = time.time() - t0
        per_fit = float(np.mean(durs))
        r = {"workers": K, "seconds_per_fit": round(per_fit, 2), "wall_seconds": round(wall, 2),
             "fit_s_per_Gcell": per_fit / gcells * READOUT_EPOCHS / CORE_SEARCH_EPOCHS,
             "fits_per_hour": 0.0,
             "sim_beats_per_s": round(sim_bps), "sim_speed_factor": sim_bps / sim_alone}
        r["fits_per_hour"] = round(K * 3600 / (r["fit_s_per_Gcell"] * len(D1["y"]) * C.shape[1] / 1e9), 1)   # full-size final-test fits
        table.append(r)
        log(f"  {K:3d} parallel fits: {per_fit:6.1f} s per fit, {r['fit_s_per_Gcell']:5.0f} s/Gcell, "
            f"~{r['fits_per_hour']:5.0f} full-size fits/hour; simulation at {r['sim_speed_factor']:.0%} of its speed alone")
        gc.collect()
    del C, prep
    sim_unc = {e: (v if f"sim_beats_per_s:{e}" in SPEED_MEASURED else sim_alone) for e, v in SPEED["sim_beats_per_s"].items()}
    scored, best = _score_workers(table, sim_unc)
    for r in scored:
        log(f"  {r['workers']:3d} workers -> projected total {r['projected_hours']:6.1f} h" + ("   <- chosen" if r is best else ""))
    cs = {"session": SESSION_TAG, "cpu_count": N_CPU, "gpu": ENV.get("gpu"), "cpu_model": ENV.get("cpu_model"), "ram_gb": ENV.get("ram_gb"),
          "best_workers": best["workers"], "table": scored, "sim_uncontended_beats_per_s": sim_unc, "rescored_after_benchmark": False,
          "seconds": round(time.time() - t_all)}
    jdump(cs, CORE_SEARCH_PATH)
    pd.DataFrame(scored).to_csv(os.path.join(WORK_DIR, "tables", "core_search.csv"), index=False)
    log(f"core search done in {(time.time() - t_all) / 60:.1f} min: {best['workers']} parallel fits (saved for later sessions in {CORE_SEARCH_PATH})")
    return cs

def core_search_rescore(cs):
    """After the benchmark: redo the choice with the speeds the benchmark measured at the chosen worker count."""
    K = cs["best_workers"]
    if "fit_s_per_Gcell:pipe" not in SPEED_MEASURED:
        return
    r_k = next(r for r in cs["table"] if r["workers"] == K)
    sim_unc = {e: v / max(r_k["sim_speed_factor"], 1e-6) for e, v in SPEED["sim_beats_per_s"].items()}
    scored, best = _score_workers(cs["table"], sim_unc, fit_anchor=(K, SPEED["fit_s_per_Gcell"]["pipe"]))
    log("core search, re-scored with the benchmark's measured speeds:")
    for r in scored:
        log(f"  {r['workers']:3d} workers -> projected total {r['projected_hours']:6.1f} h" + ("   <- best" if r is best else ""))
    cs.update({"table": scored, "best_workers": best["workers"], "rescored_after_benchmark": True, "sim_uncontended_beats_per_s": sim_unc})
    jdump(cs, CORE_SEARCH_PATH)
    pd.DataFrame(scored).to_csv(os.path.join(WORK_DIR, "tables", "core_search.csv"), index=False)
    if best["workers"] != K:
        log(f"  -> later sessions will use {best['workers']} parallel fits (this benchmark used {K})")

PLAN, JOBS = refresh_plan()
tried_this_session, STOP_REASON, n_jobs_run = set(), None, 0
SELF_TEST_FAILS = 0 if MERGE_MODE else run_self_test()   # every session (not merge mode); changes no result
CORE_SEARCH_RESULT = None
if (CORE_SEARCH and MACHINE == "local" and BENCHMARK and PIPELINE and FIT_WORKERS == "auto" and not SELF_TEST and not MERGE_MODE):
    CORE_SEARCH_RESULT = core_search()
    PIPE_W = CORE_SEARCH_RESULT["best_workers"]
RAM = RamBudget(PIPE_BUDGET_GB) if PIPELINE else None
FIT_POOL = FitPool(PIPE_W, RAM) if PIPELINE else None
BENCH_KEYS = benchmark_selection(JOBS) if BENCHMARK else None
if MERGE_MODE:
    BENCH_KEYS = set()                        # merge mode: run nothing, only combine and analyse
    log(f"MERGE mode: combined {len(MERGE_DIRS)} output folders into {WORK_DIR}; nothing is run")
elif SELF_TEST:
    BENCH_KEYS = set()                        # SELF_TEST = True: stop after the checks
    if SMOKE_TEST or BENCHMARK:
        log("NOTE: SELF_TEST = True stops after the checks, so the smoke test / benchmark did not run. "
            "The checks run at the start of every session anyway: set SELF_TEST = False to run them.")
elif SELF_TEST_FAILS:
    log(f"WARNING: {SELF_TEST_FAILS} self-test check(s) failed (see above); continuing")
if BENCHMARK:
    log(f"BENCHMARK mode: {len(BENCH_KEYS)} jobs selected (checks + representative full-size jobs)")
log(f"session start: {len(PLAN)} planned runs, {len(CACHE.done)} already done, {len(CACHE.pending)} pending; "
    f"budget {SESSION_BUDGET_HOURS} h, disk {work_dir_gb(force=True):.2f} GB; "
    + (f"pipeline with {PIPE_W} fit workers, RAM budget {PIPE_BUDGET_GB:.1f} GB" if PIPELINE else "one job at a time"))
log_resources("start")
_done_at_refresh, _last_status = len(CACHE.done), time.time()

def _next_job(nn_only=False):
    for j in JOBS:
        if BENCH_KEYS is not None and j["key"] not in BENCH_KEYS:
            continue
        if nn_only and j["key"][0] != "nn":
            continue
        pr = _pending(j, tried_this_session)
        if pr:
            return j, pr
    return None

def _maybe_refresh(force=False):
    global PLAN, JOBS, _done_at_refresh
    if force or len(CACHE.done) != _done_at_refresh:
        PLAN, JOBS = refresh_plan()            # second stages of the re-tunings appear once their first stage is done
        _done_at_refresh = len(CACHE.done)

while True:
    if PIPELINE and time.time() - _last_status > 600:
        r = log_resources("status")
        log(f"  status: fits waiting {r['fits_waiting']}, running {r['fits_running']}, done {FIT_POOL.n_done}, "
            f"failed {FIT_POOL.n_failed}; RAM reserved {r['reserved_gb']} GB, available {r['available_gb']} GB, "
            f"process peak {r['peak_rss_gb']} GB")
        _last_status = time.time()
    if PIPELINE and FIT_POOL.queued >= FIT_POOL.n:
        todo = _next_job(nn_only=True)        # the fit queue is full: give the GPU a neural-network job meanwhile
        if todo is None:
            FIT_POOL.wait_any(5.0)
            continue
    else:
        todo = _next_job()
    if todo is None:
        if PIPELINE and FIT_POOL.outstanding() > 0:
            FIT_POOL.wait_any(10.0)
            _maybe_refresh()
            continue
        _maybe_refresh(force=True)
        if _next_job() is not None:
            continue
        STOP_REASON = ("merge mode, no experiment run" if MERGE_MODE else "self-test only, no experiment run" if SELF_TEST else
                       "benchmark jobs finished" if BENCHMARK else "all planned runs are done")
        break
    job, runs = todo
    reason = _guard(job, runs)
    if reason:
        STOP_REASON = reason
        break
    n_jobs_run += 1
    run_job(job, runs, n_jobs_run)
    if any(set(r["groups"]) & {"retune", "nn_tune"} for r in runs) or n_jobs_run % 25 == 0:
        _maybe_refresh()
    else:
        with LOCK:
            CACHE.pending = [(rid, g) for rid, g in CACHE.pending if not CACHE.is_done(rid)]
            CACHE.write()
    if n_jobs_run % 10 == 0:
        log_resources(f"job {n_jobs_run}")
    gc.collect()

if PIPELINE:
    log(f"no more jobs will start ({STOP_REASON}); waiting for {FIT_POOL.outstanding()} classifier fit(s) to finish")
    while FIT_POOL.outstanding() > 0:
        if seconds_left() < 0 and not FIT_POOL.cancel.is_set():
            log("  time is up: cancelling the remaining fits (they stay pending for the next session)")
            FIT_POOL.cancel.set()
        FIT_POOL.wait_any(10.0)
        if time.time() - _last_status > 600:
            r = log_resources("draining")
            log(f"  draining: {r['fits_waiting']} waiting, {r['fits_running']} running")
            _last_status = time.time()
    FIT_POOL.ex.shutdown(wait=True)
    log_resources("end")
    save_speeds()
    log(f"fit pool: {FIT_POOL.n_done} fits done, {FIT_POOL.n_failed} failed, {FIT_POOL.n_cancelled} cancelled")

PLAN, JOBS = refresh_plan()
if CORE_SEARCH_RESULT is not None:
    core_search_rescore(CORE_SEARCH_RESULT)
    PIPE_W = CORE_SEARCH_RESULT["best_workers"]          # the projection below uses the final choice
log(f"runner stopped: {STOP_REASON}. jobs this session: {n_jobs_run}; done {len(CACHE.done)}, "
    f"failed {len(CACHE.failed)}, pending {len(CACHE.pending)}; speeds {json.dumps(SPEED, default=_json_default)}")
TIME_PROJECTION = print_projection(JOBS, "BENCHMARK RESULT: projected time for everything that is left" if BENCHMARK
                                   else "Projected time for everything that is left")



===== SELF-TEST (runs at the start of every session) =====
  [PASS] packages                     python 3.13.13, torch 2.14.1+cu126, numpy 2.1.3, sklearn 1.7.2, snntorch 0.9.4, wfdb 4.3.1, brian2 2.10.1
  [NOTE] torch 2.4.0 (authors')       2.14.1+cu126 (fine; only matters for the exact-replication attempt)
  [PASS] brian2                       2.10.1
  [PASS] GPU                          Tesla P40, 24.0 GB, CUDA 12.6; 650 liquid-sized steps in 0.14 s


  [PASS] cuDNN                        convolution forward + backward ok
  [PASS] authors' code                commit 02e45c4f3232886aed16a40d77de1f851ef4fcdd
  [PASS] MIT-BIH checksums            704 files checked, 0 mismatches, missing (unused) 0
  [PASS] DS1 = authors' order         class counts {'DS1': [45827, 943, 3787], 'DS2': [44216, 1837, 3219]}
  [PASS] authors' folds               fold indices regenerate exactly


  [PASS] liquid + readout (published) 64 beats: firing 0.4849/step, drive/target 23.86


  [PASS] liquid + readout (fixed)     64 beats: firing 0.0268/step, drive/target 1.00


  [PASS] GPU simulator vs reference   fixed: 0 of 109,760 window counts differ; published: 0 of 109,760 window counts differ (simulate() on cuda vs the float64 equations Brian2 matched, 10 DS2 beats)
  [PASS] extensions (Huang, hybrid, ELM+RR) Huang 2-matrix ensemble on 200 beats: Macro-F1 0.793; hybrid rule ok; RR window counts ok; Huang beats (50557, 200), mean-RR ratio median 1.000


  [PASS] working liquid (g_rec, raster) g_rec scales recurrent weights only: True; raster == simulate(): True; lambda 2: 2666 recurrent synapses
  [PASS] disk                         763 GB free at /root/lsm_ecg/output-benchmark (all outputs together grow to ~12 GB)
  [PASS] RAM / cores                  113.3 GB RAM, 38 cores -> 38 parallel fits
  [PASS] log file                     /root/lsm_ecg/output-benchmark/logs/session_20261121_064329.log (tail -f /root/lsm_ecg/output-benchmark/logs/latest.log)
===== SELF-TEST: all checks passed (7 s) =====



... (50 outputs left out)

[  0.08 h]    32 workers -> projected total    0.1 h


[  0.08 h]    38 workers -> projected total    0.1 h


[  0.08 h] runner stopped: benchmark jobs finished. jobs this session: 0; done 7801, failed 0, pending 1; speeds {"sim_beats_per_s": {"multi": 2969.781529551221, "rate": 2126.288861039693, "latency": 2531.8218931128904}, "encode_beats_per_s": {"multi": 28139.577550813003, "rate": 5402.268022001717, "latency": 8870.343116991891}, "fit_s_per_Gcell": {"1": 400.0, "2": 220.0, "3": 160.0, "pipe": 215.7718844183648}, "fit_threads": 4.0, "nn_s_per_epoch_kbeat": {"cnn": 0.04, "gru": 0.025}, "nn_unit_s_per_epoch_kbeat": {"cnn": 0.04217435344724354, "gru": 0.022916162182772697}, "lsm_job_overhead_s": 2.2189762984027896, "huang_s_per_run": 115.47928738594055}



===== BENCHMARK RESULT: projected time for everything that is left =====
       group  runs  jobs  hours
working_diag     1     1   0.07
TOTAL remaining: 0.1 h  ->  one continuous run (safety stop at 240.0 h)
runner: pipeline: 1 parallel fits (CPU cores 38, RAM allows 38), 216 s per Gcell per fit under load
machine: local; device: Tesla P40; CPU: Intel(R) Xeon(R) CPU E5-2696 v4 @ 2.20GHz
measured speeds: ['encode_beats_per_s:latency', 'encode_beats_per_s:multi', 'encode_beats_per_s:rate', 'fit_s_per_Gcell:pipe', 'huang_s_per_run', 'lsm_job_overhead_s', 'nn_unit_s_per_epoch_kbeat:cnn', 'nn_unit_s_per_epoch_kbeat:gru', 'sim_beats_per_s:latency', 'sim_beats_per_s:multi', 'sim_beats_per_s:rate']
still default guesses (less reliable): nn_s_per_epoch_kbeat:cnn, nn_s_per_epoch_kbeat:gru


## 11. Analysis: tables, statistics, figures, summary.md

In [12]:
# Analysis: master table, paper-style tables and statistics, comparisons, figures, summary.md.
# Every section tolerates partial results (it reports what exists so far).
TBL, FIG = os.path.join(WORK_DIR, "tables"), os.path.join(WORK_DIR, "figures")
INK, INK2, SURF, GRID = "#0b0b0b", "#52514e", "#fcfcfb", "#e4e3df"
SERIES = ["#2a78d6", "#eb6834", "#1baf7a"]            # validated categorical slots 1-3
plt.rcParams.update({"figure.facecolor": SURF, "axes.facecolor": SURF, "axes.edgecolor": GRID, "axes.labelcolor": INK2,
                     "xtick.color": INK2, "ytick.color": INK2, "text.color": INK, "axes.grid": True, "grid.color": GRID,
                     "grid.linewidth": 0.6, "axes.spines.top": False, "axes.spines.right": False, "font.size": 9,
                     "legend.frameon": False})

def load_results():
    rows = []
    for rid in sorted(os.listdir(os.path.join(WORK_DIR, "runs"))):
        d = os.path.join(WORK_DIR, "runs", rid)
        try:
            cfg = json.load(open(os.path.join(d, "config.json"))); met = json.load(open(os.path.join(d, "metrics.json")))
        except Exception:
            continue
        liq = json.load(open(os.path.join(d, "liquid_stats.json"))) if os.path.exists(os.path.join(d, "liquid_stats.json")) else {}
        s = cfg["spec"]
        r = {"run_id": rid, "group": cfg["group"], "groups": ",".join(cfg.get("groups", [cfg["group"]])),
             "config_id": cfg.get("config_id"), **{f"spec_{k}": v for k, v in s.items()},
             **{k: v for k, v in met.items() if k not in ("confusion_matrix", "eval_macro_f1_by_epoch")},
             "confusion_matrix": json.dumps(met["confusion_matrix"])}
        if isinstance(s.get("arch"), dict):
            r["spec_arch"] = json.dumps(s["arch"], sort_keys=True)
        sim = liq.get("DS1_simulation") or {}
        r.update({"A_in_used": liq.get("A_in_used"), "mu_in": liq.get("mu_in"),
                  "liquid_firing_prob": sim.get("liquid_firing_prob_per_step"),
                  "drive_over_target": sim.get("input_current_over_target"), "silent_neurons": sim.get("silent_neurons"),
                  "n_recurrent_synapses": (liq.get("structure") or {}).get("n_recurrent_synapses")})
        rows.append(r)
    return pd.DataFrame(rows)

R = load_results()
R.to_csv(os.path.join(WORK_DIR, "results_master.csv"), index=False)
print(f"results_master.csv: {len(R)} runs" + ("  (nothing has run yet, so there is nothing to analyse)" if R.empty else ""))
SUMMARY = []          # markdown lines for summary.md
KEY = ["accuracy", "macro_f1", "recall_N", "recall_SVEB", "recall_VEB", "precision_N", "precision_SVEB",
       "precision_VEB", "f1_N", "f1_SVEB", "f1_VEB"]

def _is_cv(df):
    return df["spec_split"].astype(str).str.startswith("cv-")

def lsm_base(df):
    if df.empty or "spec_kind" not in df:
        return df.iloc[0:0]
    base = df[(df.spec_kind == "lsm") & df.config_id.notna()]
    if "spec_ain_override" in base:                        # k-stale runs saved with a config id before the fix
        base = base[base.spec_ain_override.isna()]
    return base

def mean_sd(df, by, cols=("macro_f1", "accuracy", "recall_SVEB", "recall_VEB")):
    g = df.groupby(by)[list(cols)]
    out = g.mean().add_suffix("_mean").join(g.std(ddof=1).add_suffix("_sd")).join(g.size().rename("n"))
    return out.reset_index()

def md_table(df, floatfmt=4, max_rows=40):
    if df is None or df.empty:
        return "_no results yet_\n"
    d = df.head(max_rows).copy()
    for c in ("config_id", "seed", "spec_seed", "n", "n_seeds", "n_blocks", "n_configs", "n_splits", "best_by_rank", "vs_config",
              "spec_gain", "gain", "spec_num_levels", "num_levels", "folds", "identical", "runs", "jobs"):
        if c in d.columns and d[c].dtype.kind == "f" and d[c].dropna().apply(float.is_integer).all():
            d[c] = d[c].astype("Int64")
    for c in d.columns:
        if d[c].dtype.kind == "f":
            d[c] = d[c].map(lambda v: "" if pd.isna(v) else f"{v:.{floatfmt}f}")
    head = "| " + " | ".join(map(str, d.columns)) + " |\n|" + "---|" * len(d.columns) + "\n"
    return head + "".join("| " + " | ".join(map(str, r)) + " |\n" for r in d.values)

def safe(section):
    def deco(fn):
        if R.empty:                       # nothing has run yet: nothing to analyse
            return fn
        try:
            fn()
        except Exception as ex:
            SUMMARY.append(f"\n> Section **{section}** could not be produced yet: `{ex.__class__.__name__}: {str(ex)[:200]}`\n")
            print(f"[analysis] {section}: {ex.__class__.__name__}: {ex}")
        return fn
    return deco

def savefig(fig, name):
    fig.tight_layout(); fig.savefig(os.path.join(FIG, name), dpi=150); plt.close(fig)

# ---------------- 1. main grid: final (Table 8 analogue) and CV (Tables 6, 10) ----------------
@safe("main grid")
def _main():
    B = lsm_base(R)
    fin = B[B.spec_split == "final"]
    if len(fin):
        t = mean_sd(fin, ["spec_mode", "config_id"]).sort_values(["spec_mode", "macro_f1_mean"], ascending=[True, False])
        t.to_csv(os.path.join(TBL, "main_final_by_config.csv"), index=False)
        fin[["spec_mode", "spec_seed", "config_id"] + KEY].sort_values(["spec_mode", "config_id", "spec_seed"]).to_csv(
            os.path.join(TBL, "main_final_all_seeds.csv"), index=False)
        SUMMARY.append("\n## Config 11 on DS2 (paper: acc 93.51, Macro-F1 63.92)\n")
        SUMMARY.append(md_table(t[t.config_id == 11]))
    cv = B[_is_cv(B)]
    if len(cv):
        per = cv.groupby(["spec_mode", "spec_seed", "config_id"])
        t = per[KEY].mean().add_suffix("_mean").join(per[KEY].std(ddof=1).add_suffix("_sd")).join(per.size().rename("n_folds")).reset_index()
        t.to_csv(os.path.join(TBL, "main_cv_summary.csv"), index=False)

# ---------------- 2. paper statistics on the CV grid ----------------
def friedman_block(M, ids):
    n, k = M.shape
    chi2, p = sstats.friedmanchisquare(*[M[:, j] for j in range(k)])
    W = chi2 / (n * (k - 1)); FF = (n - 1) * chi2 / (n * (k - 1) - chi2)
    pFF = sstats.f.sf(FF, k - 1, (k - 1) * (n - 1))
    ranks = np.vstack([sstats.rankdata(-r) for r in M]).mean(0)
    best = ids[int(np.argmin(ranks))]
    rows = []
    for j, cid in enumerate(ids):
        if cid == best:
            continue
        w = sstats.wilcoxon(M[:, ids.index(best)], M[:, j])
        rows.append({"vs_config": cid, "W": w.statistic, "p": w.pvalue})
    wdf = pd.DataFrame(rows).sort_values("p").reset_index(drop=True)
    m = len(wdf)
    wdf["p_holm"] = np.minimum(1, np.maximum.accumulate((m - np.arange(m)) * wdf.p.values)); wdf["significant"] = wdf.p_holm < 0.05
    return {"chi2": chi2, "p": p, "iman_davenport_F": FF, "p_F": pFF, "kendall_W": W, "best_by_rank": best}, \
           pd.DataFrame({"config_id": ids, "mean_rank": ranks}).sort_values("mean_rank"), wdf

@safe("paper statistics")
def _stats():
    cv = lsm_base(R); cv = cv[_is_cv(cv)]
    out = []
    for (mode, seed), g in cv.groupby(["spec_mode", "spec_seed"]):
        piv = g.pivot_table(index="spec_split", columns="config_id", values="macro_f1")
        if piv.shape[1] < 3 or piv.isna().any().any() or piv.shape[0] < 3:
            continue
        ids = [int(c) for c in piv.columns]
        res, ranks, wdf = friedman_block(piv.values, ids)
        out.append({"mode": mode, "seed": seed, "n_blocks": piv.shape[0], "n_configs": len(ids), **res})
        ranks.to_csv(os.path.join(TBL, f"friedman_ranks_{mode}_s{seed}.csv"), index=False)
        wdf.to_csv(os.path.join(TBL, f"wilcoxon_holm_{mode}_s{seed}.csv"), index=False)
    if out:
        df = pd.DataFrame(out); df.to_csv(os.path.join(TBL, "friedman_main_cv.csv"), index=False)
        SUMMARY.append("\n## Friedman / Iman-Davenport / Kendall's W on the CV grid (paper: chi2 169.63, F 17.54, W 0.615)\n")
        SUMMARY.append(md_table(df[["mode", "seed", "n_blocks", "n_configs", "chi2", "iman_davenport_F", "kendall_W", "best_by_rank"]]))

# ---------------- 3. exact-replication attempt vs the authors' released results ----------------
@safe("exact-replication comparison")
def _exact():
    cv = lsm_base(R); cv = cv[_is_cv(cv) & (cv.spec_mode == "published") & (cv.spec_seed == 2026)]
    rows = []
    for cid, c in CONFIGS.items():
        p = os.path.join(REPO_DIR, "results/cv_24_configurations", c["authors_csv"])
        if not os.path.exists(p):
            continue
        A = pd.read_csv(p)
        for _, a in A.iterrows():
            sp = f"cv-r{int(a.rep)}f{int(a.fold)}"
            o = cv[(cv.config_id == cid) & (cv.spec_split == sp)]
            if o.empty:
                continue
            o = o.iloc[0]
            acm = [[int(x) for x in row.split()] for row in re.findall(r"\[([\d\s]+)\]", a.confusion_matrix)]
            rows.append({"config_id": cid, "encoder": c["encoder"], "split": sp, "authors_macro_f1": a.macro_f1,
                         "ours_macro_f1": o.macro_f1, "diff": o.macro_f1 - a.macro_f1,
                         "identical_confusion_matrix": acm == json.loads(o.confusion_matrix)})
    if rows:
        df = pd.DataFrame(rows); df.to_csv(os.path.join(TBL, "exact_attempt_cv.csv"), index=False)
        s = df.groupby("encoder").agg(folds=("split", "size"), identical=("identical_confusion_matrix", "sum"),
                                      mean_abs_diff=("diff", lambda d: d.abs().mean())).reset_index()
        SUMMARY.append("\n## Exact-replication attempt (published, seed 2026) vs the authors' per-fold CSVs\n")
        SUMMARY.append(f"torch 2.4.0 active: {TORCH_PINNED}; GPU: {ENV.get('gpu', 'none')} (authors: RTX 4060 Laptop, CUDA 11.8).\n\n")
        SUMMARY.append(md_table(s))
        fig, ax = plt.subplots(figsize=(4.2, 4.2))
        for i, enc in enumerate(ENCS):
            d = df[df.encoder == enc]
            ax.scatter(d.authors_macro_f1, d.ours_macro_f1, s=18, color=SERIES[i], label=enc, edgecolor=SURF, linewidth=0.8)
        lo, hi = df[["authors_macro_f1", "ours_macro_f1"]].min().min() - 0.02, df[["authors_macro_f1", "ours_macro_f1"]].max().max() + 0.02
        ax.plot([lo, hi], [lo, hi], color=INK2, lw=1, ls="--")
        ax.set_xlabel("authors' Macro-F1 (per fold)"); ax.set_ylabel("ours, published code, seed 2026")
        ax.set_title("Per-fold Macro-F1: authors vs this run", color=INK, loc="left"); ax.legend()
        savefig(fig, "exact_attempt_per_fold.png")

# ---------------- 4. encoders: published vs fixed ----------------
@safe("encoders")
def _encoders():
    B = lsm_base(R); B = B[(B.spec_filter == "morph") & (B.spec_norm == "robust") & (B.spec_readout == "svm")]
    if B.empty:
        return
    t = []
    for (mode, enc, part), g in B.assign(part=np.where(_is_cv(B), "cv", "final")).groupby(["spec_mode", "spec_encoder", "part"]):
        seed_means = g.groupby("spec_seed").macro_f1.mean()
        t.append({"mode": mode, "encoder": enc, "part": part, "macro_f1_mean": seed_means.mean(),
                  "macro_f1_sd_over_seeds": seed_means.std(ddof=1) if len(seed_means) > 1 else np.nan, "n_seeds": len(seed_means)})
    t = pd.DataFrame(t); t.to_csv(os.path.join(TBL, "encoders_morph_robust_svm.csv"), index=False)
    SUMMARY.append("\n## Encoders (morph + robust + SVM), published vs fixed\n"); SUMMARY.append(md_table(t))
    d = t[t.part == "final"]
    if d.empty:
        return
    fig, ax = plt.subplots(figsize=(5.2, 3.2))
    modes = [m for m in ("published", "fixed", "fix_gamma", "fix_ain") if m in d["mode"].values]
    w = 0.8 / len(modes)
    for i, mode in enumerate(modes):
        dd = d[d["mode"] == mode].set_index("encoder").reindex(ENCS)
        x = np.arange(3) + (i - (len(modes) - 1) / 2) * w
        ax.bar(x, dd.macro_f1_mean, w * 0.9, yerr=dd.macro_f1_sd_over_seeds.fillna(0), color=(SERIES + ["#eda100"])[i],
               label=mode, error_kw={"ecolor": INK2, "lw": 0.8})
    ax.set_xticks(np.arange(3)); ax.set_xticklabels(ENCS); ax.set_ylabel("Macro-F1 on DS2 (mean ± sd over seeds)")
    ax.set_title("Encoder comparison, published vs fixed liquid", color=INK, loc="left"); ax.legend(ncol=len(modes))
    savefig(fig, "encoders_published_vs_fixed.png")

# ---------------- 5. input-gain sweep ----------------
@safe("gain sweep")
def _gain():
    G = R[(R.spec_kind == "lsm") & (R.spec_mode == "fixed") & (R.spec_filter == "morph") & (R.spec_norm == "robust")
          & (R.spec_readout == "svm") & (R.spec_lambda_val == 0.8) & (R.spec_recurrence == True) & (R.spec_rr_in_liquid == False)
          & (R.spec_num_levels.isin([200]) | R.spec_num_levels.isna())]
    if G.empty:
        return
    G = G.assign(part=np.where(_is_cv(G), "cv", "final"))
    t = G.groupby(["spec_encoder", "spec_gain", "part"]).agg(macro_f1=("macro_f1", "mean"), macro_f1_sd=("macro_f1", "std"),
                                                             firing=("liquid_firing_prob", "mean"), drive=("drive_over_target", "mean"),
                                                             n=("macro_f1", "size")).reset_index()
    t.to_csv(os.path.join(TBL, "gain_sweep.csv"), index=False)
    SUMMARY.append("\n## Input-gain sweep (fixed liquid; gain 1 = as the paper describes)\n"); SUMMARY.append(md_table(t))
    fig, axs = plt.subplots(1, 2, figsize=(9, 3.4))
    for i, enc in enumerate(ENCS):
        d = t[(t.spec_encoder == enc) & (t.part == "final")].sort_values("spec_gain")
        if d.empty:
            continue
        axs[0].errorbar(d.spec_gain, d.macro_f1, yerr=d.macro_f1_sd.fillna(0), color=SERIES[i], marker="o", ms=4, lw=2, label=enc, capsize=2)
        axs[1].plot(d.firing, d.macro_f1, color=SERIES[i], marker="o", ms=4, lw=2, label=enc)
    for ax in axs:
        ax.set_xscale("log"); ax.set_ylabel("Macro-F1 on DS2"); ax.legend()
    axs[0].set_xlabel("input gain (× the fixed liquid's weights)"); axs[1].set_xlabel("liquid firing probability per step")
    axs[0].set_title("Accuracy vs input strength", color=INK, loc="left"); axs[1].set_title("Accuracy vs spike activity", color=INK, loc="left")
    savefig(fig, "gain_sweep.png")

# ---------------- 6. ablations, RR, baselines ----------------
def _variant(r):
    k = r.spec_kind
    if k == "lsm":
        v = f"liquid {r.spec_mode}"
        if r.spec_recurrence is False or r.spec_recurrence == False: v += ", recurrence off"
        if r.spec_lambda_val != 0.8: v += f", lambda {r.spec_lambda_val}"
        if r.spec_rr_in_liquid: v += ", RR in liquid"
        if isinstance(getattr(r, "spec_rr_set", None), str): v += " (+ mean-RR ratio)"
        if r.spec_readout == "svm_rr": v += ", RR in readout"
        gr = getattr(r, "spec_g_rec", None)
        if gr is not None and not pd.isna(gr): v += f", g_rec {float(gr):g}"
        if r.spec_readout == "svm_late": v += ", last-window readout"
        if r.spec_gain != 1: v += f", gain {r.spec_gain}"
        if r.spec_num_levels not in (200, None) and not pd.isna(r.spec_num_levels): v += f", K {int(r.spec_num_levels)}"
        if r.spec_ain_override is not None and not pd.isna(r.spec_ain_override): v += ", A_in 0.05892"
        return v
    if k == "feat":
        return {"no_liquid": "no liquid (encoder counts)", "elm": "random features (ELM)", "raw_beat": "raw beat, linear",
                "rr_only": "RR only, linear", "rr_shape": "RR + beat shape, linear", "lda": "RR + beat shape, LDA",
                "elm_rr": "random features (ELM) + RR channels", "rr_only_mrr": "RR only + mean-RR ratio, linear",
                "rr_shape_mrr": "RR + beat shape + mean-RR ratio, linear"}[r.spec_model]
    if k == "nn":
        t = getattr(r, "spec_trial", None)
        tag = "" if t is None or (isinstance(t, float) and pd.isna(t)) else " tuned" if t == "best" else f" trial {t}"
        rs = getattr(r, "spec_rr_set", None)
        return f"{r.spec_model.upper()}{tag}{' + RR' if r.spec_use_rr else ''}{' + mean-RR ratio' if isinstance(rs, str) else ''}"
    if k == "huang":
        return f"Huang 2014, gamma {float(r.spec_gamma):.3g}"
    if k == "hybrid":
        return f"hybrid ({r.spec_rule}): {r.spec_base_label}"
    return k

@safe("ablations / RR / baselines")
def _compare():
    D = R[(R.spec_filter == "morph") & (R.spec_norm == "robust")].copy()
    D = D[(D.spec_kind != "lsm") | ((D.spec_readout.isin(["svm", "svm_rr"])) & (D.spec_num_levels.isin([200]) | D.spec_num_levels.isna()))]
    if D.empty:
        return
    D["variant"] = D.apply(_variant, axis=1)
    D = D[~D.variant.str.contains(" trial ")]          # tuning trials are reported in their own table
    D["enc"] = D.spec_encoder.fillna("-")
    D["part"] = np.where(_is_cv(D), "cv", "final")
    t = D.groupby(["variant", "enc", "part"]).agg(macro_f1=("macro_f1", "mean"), macro_f1_sd=("macro_f1", "std"),
                                                  recall_SVEB=("recall_SVEB", "mean"), f1_SVEB=("f1_SVEB", "mean"),
                                                  recall_VEB=("recall_VEB", "mean"), accuracy=("accuracy", "mean"),
                                                  firing=("liquid_firing_prob", "mean"), n=("macro_f1", "size")).reset_index()
    t = t[~t.variant.str.contains("gain|K |A_in", regex=True)]
    t.to_csv(os.path.join(TBL, "ablations_rr_baselines.csv"), index=False)
    SUMMARY.append("\n## Liquid ablations, RR timing, baselines (morph + robust front end)\n")
    SUMMARY.append(md_table(t[t.part == "final"].sort_values("macro_f1", ascending=False), max_rows=60))
    d = t[(t.part == "final") & (t.enc.isin(["multi", "-"]))].sort_values("macro_f1")
    if d.empty:
        return
    fig, ax = plt.subplots(figsize=(6.4, 0.32 * len(d) + 1.2))
    y = np.arange(len(d))
    ax.barh(y - 0.18, d.macro_f1, 0.34, color=SERIES[0], label="Macro-F1", xerr=d.macro_f1_sd.fillna(0), error_kw={"ecolor": INK2, "lw": 0.8})
    ax.barh(y + 0.18, d.recall_SVEB, 0.34, color=SERIES[1], label="SVEB recall")
    ax.set_yticks(y); ax.set_yticklabels(d.variant); ax.set_xlabel("DS2 test set (mean over seeds)")
    ax.set_title("What does the work? (Multi-Threshold where applicable)", color=INK, loc="left", fontsize=9); ax.legend(loc="lower right")
    savefig(fig, "ablations_rr_baselines.png")

# ---------------- 7. corrected resampled t-tests (Nadeau & Bengio) ----------------
def nb_test(a, b):
    """a, b: Series indexed by CV split (already averaged over seeds). Corrected repeated k-fold t-test."""
    j = a.index.intersection(b.index)
    if len(j) < 3:
        return None
    d = (a[j] - b[j]).values
    n_tr, n_te = 0.75, 0.25
    var = np.var(d, ddof=1) * (1 / len(d) + n_te / n_tr)
    t = d.mean() / math.sqrt(var) if var > 0 else np.inf
    return {"n_splits": len(d), "mean_diff": d.mean(), "t": t, "p": 2 * sstats.t.sf(abs(t), len(d) - 1)}

@safe("corrected t-tests")
def _ttests():
    C = R[_is_cv(R)].copy()
    if C.empty:
        return
    C["variant"] = C.apply(_variant, axis=1); C["enc"] = C.spec_encoder.fillna("-")
    def series(variant, enc="multi", cfg=None):
        d = C[(C.variant == variant) & (C.enc == enc)]
        if cfg is not None:
            d = d[d.config_id == cfg]
        else:
            d = d[(d.spec_filter == "morph") & (d.spec_norm == "robust")]
            if "spec_readout" in d:
                d = d[d.spec_readout.isin(["svm", "svm_rr"]) | d.spec_readout.isna()]
        return d.groupby("spec_split").macro_f1.mean()
    pairs = [("published vs fixed (config 11)", series("liquid published", cfg=11), series("liquid fixed", cfg=11)),
             ("fixed: multi vs rate", series("liquid fixed", "multi", 11), series("liquid fixed", "rate", 7)),
             ("fixed: multi vs latency", series("liquid fixed", "multi", 11), series("liquid fixed", "latency", 9)),
             ("fixed liquid vs no liquid", series("liquid fixed"), series("no liquid (encoder counts)")),
             ("fixed liquid vs random features", series("liquid fixed"), series("random features (ELM)")),
             ("fixed liquid vs recurrence off", series("liquid fixed"), series("liquid fixed, recurrence off")),
             ("fixed + RR readout vs fixed", series("liquid fixed, RR in readout"), series("liquid fixed")),
             ("fixed + RR in liquid vs fixed", series("liquid fixed, RR in liquid"), series("liquid fixed")),
             ("CNN + RR vs fixed liquid", series("CNN + RR", "-"), series("liquid fixed")),
             ("tuned CNN vs fixed liquid", series("CNN tuned", "-"), series("liquid fixed")),
             ("tuned GRU vs fixed liquid", series("GRU tuned", "-"), series("liquid fixed")),
             ("tuned CNN + RR vs fixed liquid + RR readout", series("CNN tuned + RR", "-"), series("liquid fixed, RR in readout")),
             ("tuned GRU + RR vs fixed liquid + RR readout", series("GRU tuned + RR", "-"), series("liquid fixed, RR in readout")),
             ("tuned GRU + RR vs published liquid + RR readout", series("GRU tuned + RR", "-"), series("liquid published, RR in readout"))]
    rows = []
    for name, a, b in pairs:
        res = nb_test(a, b)
        if res:
            rows.append({"comparison": name, **res})
    if rows:
        df = pd.DataFrame(rows); df.to_csv(os.path.join(TBL, "corrected_ttests.csv"), index=False)
        SUMMARY.append("\n## Corrected resampled t-tests on DS1 CV (Nadeau & Bengio; seeds averaged per split)\n")
        SUMMARY.append(md_table(df))

# ---------------- 8. retune and k-stale diagnostics ----------------
@safe("retune")
def _retune():
    best, df = retune_choice()
    if df is not None:
        df.to_csv(os.path.join(TBL, "retune_stage1.csv"), index=False)
        SUMMARY.append("\n## Re-tuning the fixed liquid (stage 1, DS1 CV rep 1)\n"); SUMMARY.append(md_table(df.head(10)))

@safe("NN tuning")
def _nn_tune():
    rows, picks = [], []
    for m in ("cnn", "gru"):
        for use_rr in (False, True):
            best, df = nn_tune_choice(m, use_rr)
            if df is None:
                continue
            rows.append(df)
            default = df[(df.trial == 0) & (df.epochs == df[df.trial == 0].sort_values("cv_macro_f1").epochs.iloc[-1])].iloc[0]
            picks.append({"model": m.upper() + (" + RR" if use_rr else ""), "best_trial": best["trial"], "epochs": best["epochs"],
                          "n_params": int(df[df.trial == best["trial"]].n_params.iloc[0]), "cv_macro_f1": best["cv_macro_f1"],
                          "untuned_arch_best_cv_macro_f1": float(default.cv_macro_f1), "arch": json.dumps(best["arch"], sort_keys=True)})
    if rows:
        pd.concat(rows).to_csv(os.path.join(TBL, "nn_tuning_stage1.csv"), index=False)
        P = pd.DataFrame(picks); P.to_csv(os.path.join(TBL, "nn_tuning_choice.csv"), index=False)
        SUMMARY.append("\n## Tuned CNN / GRU baselines (stage 1: DS1 CV rep 1 only; DS2 untouched)\n")
        SUMMARY.append(md_table(P.drop(columns="arch")))
        SUMMARY.append("\nChosen architectures:\n\n" + "".join(f"- {r['model']}: `{r['arch']}`\n" for r in picks))
        SUMMARY.append("\nNote: stage-2 CV numbers include the rep-1 folds used for the choice (slightly optimistic, as for the "
                       "liquid re-tuning); the DS2 numbers are clean.\n")

@safe("k-stale diagnostic")
def _kstale():
    B = R[(R.spec_kind == "lsm") & (R.spec_mode == "published") & (R.spec_seed == 2026) & _is_cv(R)]
    if B.empty:
        return
    B = B.assign(cid=B.apply(lambda r: next((cid for cid, c in CONFIGS.items() if (c["filter"], c["norm"], c["encoder"], c["readout"])
                                             == (r.spec_filter, r.spec_norm, r.spec_encoder, r.spec_readout)), None), axis=1))
    B = B.assign(kind=np.where(B.spec_ain_override.notna(), "A_in 0.05892", "paper A_in"))
    rows = []
    for cid, c in CONFIGS.items():
        if c["encoder"] == "multi":
            continue
        p = os.path.join(REPO_DIR, "results/cv_24_configurations", c["authors_csv"])
        a = pd.read_csv(p).macro_f1.mean() if os.path.exists(p) else np.nan
        r = {"config_id": cid, "encoder": c["encoder"], "authors_cv_macro_f1": a}
        for kind, g in B[B.cid == cid].groupby("kind"):
            r[f"ours_{kind}"] = g.macro_f1.mean()
        rows.append(r)
    df = pd.DataFrame(rows); df.to_csv(os.path.join(TBL, "kstale_diagnostic.csv"), index=False)
    SUMMARY.append("\n## Which A_in did the authors' Rate/Latency runs use? (CV mean Macro-F1, published code, seed 2026)\n")
    SUMMARY.append(md_table(df))

# ---------------- 8b. extensions: Huang baseline, hybrid rule, mean-RR feature, ELM + RR ----------------
@safe("extensions")
def _extensions():
    keep = R.spec_kind.isin(["huang", "hybrid"])
    if "spec_model" in R:
        keep |= R.spec_model.isin(["elm_rr", "rr_only_mrr", "rr_shape_mrr"])
    if "spec_rr_set" in R:
        keep |= R.spec_rr_set.notna()
    if "spec_g_rec" in R:
        keep |= R.spec_g_rec.notna()
    pdiag = os.path.join(WORK_DIR, "checks", "working_liquid_diag.json")
    if os.path.exists(pdiag):
        wd = json.load(open(pdiag))
        dt = pd.DataFrame([{k: v for k, v in r.items() if k != "perturbation_curve_from_t_perturb"} for r in wd["settings"]])
        dt["chosen"] = dt.candidate & (dt.g_rec == wd["chosen_g_rec"])
        dt.to_csv(os.path.join(TBL, "working_liquid_diag.csv"), index=False)
        SUMMARY.append(f"\n## Working liquid: perturbation diagnostic (chosen g_rec = {wd['chosen_g_rec']:g})\n")
        SUMMARY.append(md_table(dt[["label", "firing_per_step", "last_window_firing_per_step", "recurrent_over_input_current",
                                    "perturbation_early", "perturbation_late", "damage_ratio", "n_recurrent_synapses", "chosen"]]))
    E = R[keep].copy()
    if E.empty:
        return
    E["variant"] = E.apply(_variant, axis=1)
    E["enc"] = E.spec_encoder.fillna("-") if "spec_encoder" in E else "-"
    rows = []
    for (v, enc), g in E.groupby(["variant", "enc"]):
        cv, fin = g[_is_cv(g)], g[g.spec_split == "final"]
        ps = cv.groupby("spec_split").macro_f1.mean()            # seeds averaged per split
        r = {"variant": v, "encoder": enc, "cv_macro_f1": ps.mean(), "cv_splits": len(ps),
             "cv_reps23": ps[[s for s in ps.index if not s.startswith("cv-r1")]].mean(),
             "ds2_macro_f1": fin.macro_f1.mean(), "ds2_sd": fin.macro_f1.std(ddof=1), "ds2_runs": len(fin),
             "ds2_recall_SVEB": fin.recall_SVEB.mean(), "ds2_precision_SVEB": fin.precision_SVEB.mean(),
             "ds2_recall_VEB": fin.recall_VEB.mean(), "ds2_precision_VEB": fin.precision_VEB.mean()}
        if "vote_ave_N_VEB" in g and g.spec_kind.iloc[0] == "huang":   # Huang's own selection criterion, rep-1 folds only
            r["rep1_vote_ave_N_VEB"] = cv[cv.spec_split.str.startswith("cv-r1")].vote_ave_N_VEB.mean()
        rows.append(r)
    df = pd.DataFrame(rows)
    if "rep1_vote_ave_N_VEB" in df and df.rep1_vote_ave_N_VEB.notna().any():
        df["huang_primary"] = df.rep1_vote_ave_N_VEB == df.rep1_vote_ave_N_VEB.max()    # chosen on DS1 rep 1, never DS2
    df.to_csv(os.path.join(TBL, "extensions_summary.csv"), index=False)
    SUMMARY.append("\n## Extensions: Huang baseline, hybrid SVEB rule, mean-RR feature, ELM + RR (descriptive; verdicts come later)\n")
    SUMMARY.append(md_table(df.sort_values("ds2_macro_f1", ascending=False), max_rows=80))

# ---------------- 9. config 11 confusion matrices and saved state vectors ----------------
@safe("config 11 figures")
def _cfg11():
    fin = lsm_base(R); fin = fin[(fin.spec_split == "final") & (fin.config_id == 11) & (fin.spec_seed == SEEDS[0])]
    modes = [m for m in ("published", "fixed") if m in fin.spec_mode.values]
    if not modes:
        return
    fig, axs = plt.subplots(1, len(modes), figsize=(3.6 * len(modes), 3.3))
    axs = np.atleast_1d(axs)
    for ax, mode in zip(axs, modes):
        cm = np.array(json.loads(fin[fin.spec_mode == mode].iloc[0].confusion_matrix))
        frac = cm / cm.sum(1, keepdims=True)
        ax.imshow(frac, cmap="Blues", vmin=0, vmax=1)
        for i in range(3):
            for j in range(3):
                ax.text(j, i, f"{cm[i, j]}\n{frac[i, j]:.1%}", ha="center", va="center", fontsize=8, color=SURF if frac[i, j] > 0.6 else INK)
        ax.set_xticks(range(3)); ax.set_yticks(range(3)); ax.set_xticklabels(CLASS_NAMES); ax.set_yticklabels(CLASS_NAMES)
        ax.set_xlabel("predicted"); ax.set_ylabel("true"); ax.grid(False); ax.set_title(f"config 11, {mode}, seed {SEEDS[0]}", color=INK, loc="left")
    savefig(fig, "config11_confusion_matrices.png")

@safe("state-vector PCA")
def _pca():
    files = sorted(glob.glob(os.path.join(WORK_DIR, "states", "*.npz")))
    pick = [f for f in files if f"_s{SEEDS[0]}_" in os.path.basename(f)]
    if not pick:
        return
    fig, axs = plt.subplots(1, len(pick), figsize=(4 * len(pick), 3.6)); axs = np.atleast_1d(axs)
    rows = []
    for ax, f in zip(axs, pick):
        z = np.load(f); X = z["counts"].astype(np.float32) / WIN; y = z["y"]
        rng = np.random.default_rng(0)
        idx = np.concatenate([rng.choice(np.where(y == c)[0], min(1500, (y == c).sum()), replace=False) for c in range(3) if (y == c).any()])
        Xs = (X[idx] - X[idx].mean(0)) / (X[idx].std(0) + 1e-6)
        U, S, Vt = np.linalg.svd(Xs, full_matrices=False)
        P = Xs @ Vt[:2].T
        for c in range(3):
            m = y[idx] == c
            ax.scatter(P[m, 0], P[m, 1], s=4, color=SERIES[c], label=CLASS_NAMES[c], alpha=0.6, linewidths=0)
        mode = "published" if "published" in os.path.basename(f) else "fixed"
        ax.set_title(f"{mode}: liquid states, PCA", color=INK, loc="left"); ax.legend(markerscale=3)
        nz = (X > 0).mean()
        rows.append({"file": os.path.basename(f), "mode": mode, "nonzero_fraction": float(nz),
                     "pc1_pc2_variance_share": float((S[:2] ** 2).sum() / (S ** 2).sum())})
    savefig(fig, "state_vectors_pca.png")
    pd.DataFrame(rows).to_csv(os.path.join(TBL, "state_vectors_summary.csv"), index=False)

# ---------------- summary.md ----------------
done_by_group = pd.Series([v[4] if len(v) > 4 else "?" for v in CACHE.done.values()]).value_counts()
pend_by_group = pd.Series([g for _, g in CACHE.pending]).value_counts()
status = pd.DataFrame({"done": done_by_group, "pending": pend_by_group}).reindex(GROUP_ORDER).fillna(0).astype(int)
head = [f"# LSM-ECG replication: results so far\n",
        f"_Updated {datetime.datetime.utcnow().isoformat(timespec='seconds')} UTC, session {SESSION_TAG}. "
        f"Runner stopped: {STOP_REASON}._\n",
        "\n## Environment\n", md_table(pd.DataFrame([{k: ENV.get(k) for k in ('torch', 'torch_pinned_2_4_0', 'gpu', 'cuda', 'python', 'snntorch', 'sklearn', 'smoke_test')}])),
        "\n## Checks\n",
        f"- DS1 beat order identical to the authors' file: **{CHECKS.get('ds1_matches_authors_order')}**; class counts {CHECKS.get('class_counts')}\n",
        f"- Authors' fold indices regenerate exactly: **{CHECKS.get('folds_regenerate_exactly')}**\n",
        f"- Checksums: {CHECKS.get('checksums', {}).get('files_checked')} files checked, "
        f"{len(CHECKS.get('checksums', {}).get('mismatches', []))} mismatches; missing from the input: "
        f"{CHECKS.get('checksums', {}).get('missing') or 'none'} (none of them used by the pipeline)\n"]
if os.path.exists(os.path.join(WORK_DIR, "checks", "brian2_check.json")):
    b2r = json.load(open(os.path.join(WORK_DIR, "checks", "brian2_check.json")))
    for mode, v in b2r["liquids"].items():
        head.append(f"- Brian2 vs PyTorch ({mode} liquid, {len(v['per_beat'])} beats, {v['total_cells']:,} neuron-steps): "
                    f"{v['brian2_vs_torch64_mismatches']} mismatches vs float64 equations, {v['brian2_vs_authors_float32_mismatches']} vs the authors' float32 snnTorch neuron; "
                    f"equations vs snnTorch (float32): {v['equations_vs_snntorch_float32_mismatches']}\n")
head += ["\n## Progress by group\n", md_table(status.reset_index().rename(columns={"index": "group"}))]
for _f in ("ain_measured.csv", "liquid_structure.csv"):     # written once by the checks job; restore after a resume / merge
    if os.path.exists(os.path.join(WORK_DIR, "checks", _f)) and not os.path.exists(os.path.join(TBL, _f)):
        shutil.copy2(os.path.join(WORK_DIR, "checks", _f), os.path.join(TBL, _f))
if MERGE_MODE:
    head += ["\n## Merged output\n", "".join(f"- from `{md}`\n" for md in MERGE_DIRS)]
_stp = os.path.join(WORK_DIR, "checks", "self_test.json")
if os.path.exists(_stp):
    _st = json.load(open(_stp))
    head += [f"\n## Self-test (session {_st.get('session', '?')}): {'all checks passed' if _st['passed'] else 'some checks FAILED'}\n"]
    head += [f"- {'PASS' if v['ok'] else ('NOTE' if k.startswith('torch 2.4.0') else 'FAIL')} {k}: {v['detail']}\n" for k, v in _st["checks"].items()]
_res = sorted(glob.glob(os.path.join(WORK_DIR, "logs", "resources_*.csv")))
if _res:
    try:
        _rl = pd.read_csv(_res[-1])
        head += ["\n## Resources this session\n",
                 f"- runner: {'pipeline, ' + str(PIPE_W) + ' parallel fits' if PIPELINE else 'one job at a time'}; CPU cores {N_CPU}; "
                 f"RAM {ENV.get('ram_gb')} GB; GPU {ENV.get('gpu', 'none')}\n",
                 f"- process peak RAM {_rl.peak_rss_gb.max():.1f} GB; lowest available RAM {_rl.available_gb.min():.1f} GB; "
                 f"peak GPU memory {_rl.gpu_mem_gb.max() if 'gpu_mem_gb' in _rl and _rl.gpu_mem_gb.notna().any() else '-'} GB "
                 f"(details: logs/{os.path.basename(_res[-1])})\n"]
    except Exception as ex:
        print("resources summary:", ex)
_tp = os.path.join(TBL, "time_projection.txt")
if os.path.exists(_tp):
    head += ["\n## Projected time remaining (from measured speeds)\n", "```\n" + open(_tp).read() + "```\n"]
atomic_write_text(os.path.join(WORK_DIR, "summary.md"), "".join(head + SUMMARY))
print("summary.md, tables/ and figures/ written:", sorted(os.listdir(TBL)), sorted(os.listdir(FIG)))


results_master.csv: 7799 runs


summary.md, tables/ and figures/ written: ['ablations_rr_baselines.csv', 'ain_measured.csv', 'core_search.csv', 'corrected_ttests.csv', 'encoders_morph_robust_svm.csv', 'exact_attempt_cv.csv', 'extensions_summary.csv', 'friedman_main_cv.csv', 'friedman_ranks_fix_ain_s2026.0.csv', 'friedman_ranks_fix_ain_s420.0.csv', 'friedman_ranks_fix_ain_s69.0.csv', 'friedman_ranks_fix_gamma_s2026.0.csv', 'friedman_ranks_fix_gamma_s420.0.csv', 'friedman_ranks_fix_gamma_s69.0.csv', 'friedman_ranks_fixed_s2026.0.csv', 'friedman_ranks_fixed_s420.0.csv', 'friedman_ranks_fixed_s69.0.csv', 'friedman_ranks_published_s2026.0.csv', 'friedman_ranks_published_s420.0.csv', 'friedman_ranks_published_s69.0.csv', 'gain_sweep.csv', 'kstale_diagnostic.csv', 'liquid_structure.csv', 'main_cv_summary.csv', 'main_final_all_seeds.csv', 'main_final_by_config.csv', 'nn_tuning_choice.csv', 'nn_tuning_stage1.csv', 'retune_stage1.csv', 'state_vectors_summary.csv', 'time_projection.csv', 'time_projection.txt', 'wilcoxon_holm_fi

## 12. End of session

In [13]:
# End of session: final cache write, disk use, what is left, how to continue.
CACHE.write()
gb = work_dir_gb(force=True)
left = pd.Series([g for _, g in CACHE.pending]).value_counts().reindex(GROUP_ORDER).dropna().astype(int)
print(f"session time: {elapsed_h():.2f} h | output size: {gb:.2f} GB | done {len(CACHE.done)} | failed {len(CACHE.failed)} | pending {len(CACHE.pending)}")
print(f"stopped because: {STOP_REASON}")
if MERGE_MODE:
    print(f"Merged {len(MERGE_DIRS)} output folders into {WORK_DIR}: {len(CACHE.done)} finished runs. "
          + (f"{len(CACHE.pending)} runs of the whole plan are not in either folder yet (pending by group below)." if len(left) else
             "Nothing in the plan is missing."))
    if len(left):
        print("pending by group:\n" + left.to_string())
elif SELF_TEST:
    print("Self-test only (see the SELF-TEST block above). Set SELF_TEST = False to run the smoke test, benchmark or real run.")
elif len(left):
    print("pending by group:\n" + left.to_string())
    if MACHINE == "kaggle":
        print("\nTo continue: Save Version -> attach this version's output as an input (Add Input -> Your Work / Notebook Output),\n"
              "put its folder path in RESUME_DIRS in the first cell, then Save & Run All again.")
    else:
        print(f"\nTo continue: run the notebook again with the same WORK_DIR. It reads {CACHE_PATH} and skips everything finished.")
else:
    print("Everything in the plan is done.")
if CACHE.failed:
    print(f"\n{len(CACHE.failed)} failed runs (tracebacks in logs/failures/); they are retried next session if RETRY_FAILED=True.")


session time: 0.16 h | output size: 6.88 GB | done 7801 | failed 0 | pending 1
stopped because: benchmark jobs finished
pending by group:
working_diag    1

To continue: run the notebook again with the same WORK_DIR. It reads /root/lsm_ecg/output-benchmark/cache.txt and skips everything finished.
